# YOLOv11: Full Project for Kaggle

This notebook recreates the entire YOLOv11 project structure and allows for training and inference.

### Setup Instructions:
1. Run all cells to create the file structure.
2. **Training**: All commands starting with `!` are meant to be run directly in code cells (not in a terminal).
3. Ensure you have enabled **GPU T4 x2** in the Kaggle Accelerator settings.

In [ ]:
!pip install pycocotools -q

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
!mkdir -p yolov11/losses yolov11/utils yolov11/data

In [ ]:
%%writefile yolov11/__init__.py
"""
YOLOv11 Implementation
Supports Detection, Segmentation, and Pose Estimation

Key improvements over YOLOv8:
- C3k2 blocks for improved efficiency
- C2PSA spatial attention mechanism
- Reduced parameters with higher accuracy
"""

from .model import YOLOv11, create_model
from .backbone import CSPDarknet
from .neck import PANet
from .head import DetectionHead, SegmentationHead, PoseHead

__version__ = "11.0.0"
__all__ = [
    "YOLOv11",
    "create_model",
    "CSPDarknet", 
    "PANet",
    "DetectionHead",
    "SegmentationHead", 
    "PoseHead"
]


In [ ]:
%%writefile yolov11/blocks.py
"""
YOLOv11 Building Blocks
Core components: C3k2 (efficient CSP), C2PSA (spatial attention), and standard blocks
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, List


def autopad(kernel_size: int, padding: Optional[int] = None, dilation: int = 1) -> int:
    """Calculate padding to maintain spatial dimensions."""
    if padding is None:
        padding = (kernel_size - 1) // 2 * dilation
    return padding


class Conv(nn.Module):
    """
    Standard convolution block: Conv2d + BatchNorm2d + SiLU
    
    The basic building block used throughout the architecture.
    """
    
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 1,
        stride: int = 1,
        padding: Optional[int] = None,
        groups: int = 1,
        dilation: int = 1,
        activation: bool = True
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size,
            stride,
            autopad(kernel_size, padding, dilation),
            groups=groups,
            dilation=dilation,
            bias=False
        )
        self.bn = nn.BatchNorm2d(out_channels, eps=1e-3, momentum=0.03)
        self.act = nn.SiLU(inplace=True) if activation else nn.Identity()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.bn(self.conv(x)))
    
    def forward_fuse(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass without batch norm (for fused inference)."""
        return self.act(self.conv(x))


class DWConv(Conv):
    """Depthwise convolution with groups=in_channels."""
    
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 1,
        stride: int = 1,
        dilation: int = 1,
        activation: bool = True
    ):
        super().__init__(
            in_channels,
            out_channels,
            kernel_size,
            stride,
            groups=in_channels,
            dilation=dilation,
            activation=activation
        )


class DWConvTranspose2d(nn.ConvTranspose2d):
    """Depthwise transposed convolution."""
    
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 2,
        stride: int = 2,
        padding: int = 0,
        output_padding: int = 0
    ):
        super().__init__(
            in_channels,
            out_channels,
            kernel_size,
            stride,
            padding,
            output_padding,
            groups=in_channels,
            bias=False
        )


class Bottleneck(nn.Module):
    """
    Standard bottleneck block with optional residual connection.
    Structure: 1x1 conv -> 3x3 conv [+ residual]
    """
    
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        shortcut: bool = True,
        groups: int = 1,
        expansion: float = 0.5
    ):
        super().__init__()
        hidden_channels = int(out_channels * expansion)
        self.cv1 = Conv(in_channels, hidden_channels, 1, 1)
        self.cv2 = Conv(hidden_channels, out_channels, 3, 1, groups=groups)
        self.add = shortcut and in_channels == out_channels
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.cv2(self.cv1(x)) if self.add else self.cv2(self.cv1(x))


class C2f(nn.Module):
    """
    CSP Bottleneck with 2 convolutions.
    Improves gradient flow while maintaining computational efficiency.
    
    Structure: Input -> Conv1x1 -> Split -> [n x Bottleneck] -> Concat -> Conv1x1 -> Output
    """
    
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        n: int = 1,
        shortcut: bool = False,
        groups: int = 1,
        expansion: float = 0.5
    ):
        super().__init__()
        self.c = int(out_channels * expansion)
        self.cv1 = Conv(in_channels, 2 * self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, out_channels, 1, 1)
        self.m = nn.ModuleList(
            Bottleneck(self.c, self.c, shortcut, groups, expansion=1.0)
            for _ in range(n)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = list(self.cv1(x).chunk(2, 1))
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))
    
    def forward_split(self, x: torch.Tensor) -> torch.Tensor:
        """Alternative forward using split instead of chunk."""
        y = list(self.cv1(x).split((self.c, self.c), 1))
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))


class SPPF(nn.Module):
    """
    Spatial Pyramid Pooling - Fast
    
    Uses sequential max pooling for efficiency while maintaining receptive field.
    Structure: Input -> Conv1x1 -> [MaxPool5x5]x3 (sequential) -> Concat -> Conv1x1 -> Output
    """
    
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 5
    ):
        super().__init__()
        hidden_channels = in_channels // 2
        self.cv1 = Conv(in_channels, hidden_channels, 1, 1)
        self.cv2 = Conv(hidden_channels * 4, out_channels, 1, 1)
        self.m = nn.MaxPool2d(
            kernel_size=kernel_size,
            stride=1,
            padding=kernel_size // 2
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.cv1(x)
        y1 = self.m(x)
        y2 = self.m(y1)
        y3 = self.m(y2)
        return self.cv2(torch.cat([x, y1, y2, y3], 1))


class Concat(nn.Module):
    """Concatenation layer along specified dimension."""
    
    def __init__(self, dimension: int = 1):
        super().__init__()
        self.d = dimension
    
    def forward(self, x: List[torch.Tensor]) -> torch.Tensor:
        return torch.cat(x, self.d)


class Upsample(nn.Module):
    """Upsampling layer using interpolation."""
    
    def __init__(self, scale_factor: int = 2, mode: str = 'nearest'):
        super().__init__()
        self.scale_factor = scale_factor
        self.mode = mode
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.interpolate(x, scale_factor=self.scale_factor, mode=self.mode)


class Proto(nn.Module):
    """
    Mask prototype module for instance segmentation.
    Generates prototype masks combined with mask coefficients for instance-specific masks.
    """
    
    def __init__(
        self,
        in_channels: int,
        proto_channels: int = 256,
        num_protos: int = 32
    ):
        super().__init__()
        self.cv1 = Conv(in_channels, proto_channels, 3)
        self.upsample = nn.ConvTranspose2d(
            proto_channels, proto_channels, 2, 2, 0, bias=True
        )
        self.cv2 = Conv(proto_channels, proto_channels, 3)
        self.cv3 = Conv(proto_channels, num_protos, 1)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.cv3(self.cv2(self.upsample(self.cv1(x))))


class DFL(nn.Module):
    """
    Distribution Focal Loss layer for box regression (vectorized).
    
    Converts discrete probability distribution over bins [0, 1, ..., reg_max-1]
    to continuous values, capturing localization uncertainty.
    """
    
    def __init__(self, reg_max: int = 16):
        super().__init__()
        self.reg_max = reg_max
        # Register weights as buffer for efficient computation
        self.register_buffer('weights', torch.arange(reg_max, dtype=torch.float).view(1, 1, reg_max, 1))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Vectorized DFL forward pass.
        
        Args:
            x: Input tensor (B, 4*reg_max, H, W) or (B, N, 4*reg_max)
        
        Returns:
            Integrated values (B, 4, H, W) or (B, N, 4)
        """
        b, c, *hw = x.shape
        
        # Reshape to (B, 4, reg_max, H*W)
        x = x.view(b, 4, self.reg_max, -1)
        
        # Softmax over reg_max dimension
        x = F.softmax(x, dim=2)
        
        # Vectorized weighted sum: (B, 4, reg_max, N) * (1, 1, reg_max, 1) -> sum -> (B, 4, N)
        x = (x * self.weights).sum(dim=2)
        
        # Reshape back to original spatial dimensions
        if hw:
            x = x.view(b, 4, *hw)
        
        return x


# YOLOv11 Specific Blocks

class C3k(nn.Module):
    """
    CSP Bottleneck with 3 convolutions and customizable kernel size.
    
    Args:
        c1: Input channels
        c2: Output channels
        n: Number of bottleneck blocks
        shortcut: Use residual connection
        g: Groups for convolution
        e: Expansion ratio
        k: Kernel size for bottleneck convs
    """
    
    def __init__(
        self,
        c1: int,
        c2: int,
        n: int = 1,
        shortcut: bool = True,
        g: int = 1,
        e: float = 0.5,
        k: int = 3
    ):
        super().__init__()
        c_ = int(c2 * e)
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(c1, c_, 1, 1)
        self.cv3 = Conv(2 * c_, c2, 1)
        self.m = nn.Sequential(
            *[Bottleneck(c_, c_, shortcut, g, expansion=1.0) for _ in range(n)]
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.cv3(torch.cat((self.m(self.cv1(x)), self.cv2(x)), 1))


class C3k2(nn.Module):
    """
    YOLOv11 CSP Bottleneck with 2 convolutions (efficient version).
    
    Replaces C2f with more efficient implementation using smaller kernels.
    Key differences from C2f:
    - Uses C3k sub-blocks for improved feature extraction
    - Better gradient flow
    - Reduced computational cost
    """
    
    def __init__(
        self,
        c1: int,
        c2: int,
        n: int = 1,
        c3k: bool = False,
        e: float = 0.5,
        g: int = 1,
        shortcut: bool = True
    ):
        super().__init__()
        self.c = int(c2 * e)
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1, 1)
        
        if c3k:
            self.m = nn.ModuleList(
                C3k(self.c, self.c, 2, shortcut, g) for _ in range(n)
            )
        else:
            self.m = nn.ModuleList(
                Bottleneck(self.c, self.c, shortcut, g, expansion=1.0)
                for _ in range(n)
            )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = list(self.cv1(x).chunk(2, 1))
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))


class Attention(nn.Module):
    """
    Spatial attention module for YOLOv11.
    Uses PyTorch 2.0+ scaled_dot_product_attention for efficiency (Flash Attention).
    """
    
    def __init__(self, dim: int, num_heads: int = 8, attn_ratio: float = 0.5):
        super().__init__()
        self.num_heads = max(1, min(num_heads, dim // 8))
        self.head_dim = dim // self.num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = Conv(dim, dim * 3, 1)
        self.proj = Conv(dim, dim, 1)
        
        # Check if scaled_dot_product_attention is available (PyTorch 2.0+)
        self._use_sdpa = hasattr(F, 'scaled_dot_product_attention')
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        N = H * W
        
        qkv = self.qkv(x)
        qkv = qkv.view(B, 3, self.num_heads, self.head_dim, N)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]  # Each: (B, heads, head_dim, N)
        
        if self._use_sdpa:
            # Use PyTorch 2.0+ Flash Attention (2-3x faster)
            # Transpose to (B, heads, N, head_dim) for SDPA
            q = q.transpose(-2, -1)  # (B, heads, N, head_dim)
            k = k.transpose(-2, -1)
            v = v.transpose(-2, -1)
            
            out = F.scaled_dot_product_attention(q, k, v, scale=self.scale)
            out = out.transpose(-2, -1)  # (B, heads, head_dim, N)
        else:
            # Fallback to manual attention for older PyTorch
            attn = (q.transpose(-2, -1) @ k) * self.scale
            attn = F.softmax(attn, dim=-1)
            out = v @ attn.transpose(-2, -1)
        
        out = out.reshape(B, C, H, W)
        return self.proj(out)


class C2PSA(nn.Module):
    """
    YOLOv11 Cross Stage Partial with Spatial Attention (C2PSA).
    
    Enhances spatial attention within feature maps, enabling the model
    to focus on critical image regions.
    
    Structure: Input -> Conv1x1 -> Split -> [Attention blocks] -> Concat -> Conv1x1 -> Output
    """
    
    def __init__(
        self,
        c1: int,
        c2: int,
        n: int = 1,
        e: float = 0.5
    ):
        super().__init__()
        self.c = int(c2 * e)
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv(2 * self.c, c2, 1, 1)
        
        self.m = nn.Sequential(
            *[
                nn.Sequential(
                    Attention(self.c, num_heads=self.c // 64 if self.c >= 64 else 1),
                    nn.Sequential(
                        Conv(self.c, self.c * 2, 1),
                        Conv(self.c * 2, self.c, 1)
                    )
                )
                for _ in range(n)
            ]
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        a, b = self.cv1(x).chunk(2, 1)
        b = self.m(b)
        return self.cv2(torch.cat([a, b], 1))


In [ ]:
%%writefile yolov11/backbone.py
"""
YOLOv11 Backbone - CSPDarknet with C3k2 and C2PSA
Extracts multi-scale features from input images
"""

import torch
import torch.nn as nn
from typing import Tuple, Dict, List

from .blocks import Conv, C3k2, C2PSA, SPPF


# Model scaling configurations: (depth_multiplier, width_multiplier)
MODEL_SCALES: Dict[str, Tuple[float, float]] = {
    'n': (0.33, 0.25),  # nano
    's': (0.33, 0.50),  # small
    'm': (0.67, 0.75),  # medium
    'l': (1.00, 1.00),  # large
    'x': (1.33, 1.25),  # xlarge
}

# Base channel sizes at each stage
BASE_CHANNELS = [64, 128, 256, 512, 1024]

# Number of C3k2 blocks at each stage
BASE_DEPTHS = [3, 6, 6, 3]


def make_divisible(x: float, divisor: int = 8) -> int:
    """Make number divisible by divisor."""
    return max(divisor, int(x + divisor / 2) // divisor * divisor)


class CSPDarknet(nn.Module):
    """
    YOLOv11 Backbone (CSPDarknet with C3k2 + C2PSA)
    
    Architecture:
        P1: Conv 3x3 s=2 -> 320x320
        P2: Conv 3x3 s=2 + C3k2 -> 160x160
        P3: Conv 3x3 s=2 + C3k2 -> 80x80   [output]
        P4: Conv 3x3 s=2 + C3k2 -> 40x40   [output]
        P5: Conv 3x3 s=2 + C3k2 + SPPF + C2PSA -> 20x20 [output]
    
    Args:
        model_size: One of 'n', 's', 'm', 'l', 'x'
        in_channels: Input image channels (default: 3 for RGB)
    """
    
    def __init__(self, model_size: str = 's', in_channels: int = 3):
        super().__init__()
        
        if model_size not in MODEL_SCALES:
            raise ValueError(f"Model size must be one of {list(MODEL_SCALES.keys())}")
        
        depth_mult, width_mult = MODEL_SCALES[model_size]
        
        channels = [make_divisible(c * width_mult) for c in BASE_CHANNELS]
        depths = [max(round(d * depth_mult), 1) for d in BASE_DEPTHS]
        
        self.out_channels = channels[2:]  # P3, P4, P5 channels
        
        # Stem
        self.stem = Conv(in_channels, channels[0], 3, 2)
        
        # Stage 1 (P2)
        self.stage1 = nn.Sequential(
            Conv(channels[0], channels[1], 3, 2),
            C3k2(channels[1], channels[1], n=depths[0], shortcut=True)
        )
        
        # Stage 2 (P3) - First output
        self.stage2 = nn.Sequential(
            Conv(channels[1], channels[2], 3, 2),
            C3k2(channels[2], channels[2], n=depths[1], shortcut=True)
        )
        
        # Stage 3 (P4) - Second output
        self.stage3 = nn.Sequential(
            Conv(channels[2], channels[3], 3, 2),
            C3k2(channels[3], channels[3], n=depths[2], shortcut=True)
        )
        
        # Stage 4 (P5) - Third output with SPPF + C2PSA
        self.stage4 = nn.Sequential(
            Conv(channels[3], channels[4], 3, 2),
            C3k2(channels[4], channels[4], n=depths[3], shortcut=True),
            SPPF(channels[4], channels[4], kernel_size=5),
            C2PSA(channels[4], channels[4], n=1)
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize model weights."""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Forward pass.
        
        Args:
            x: Input tensor (B, 3, H, W)
        
        Returns:
            Tuple of feature maps (P3, P4, P5):
                P3: (B, C3, H/8, W/8)   - small objects
                P4: (B, C4, H/16, W/16) - medium objects  
                P5: (B, C5, H/32, W/32) - large objects
        """
        x = self.stem(x)
        x = self.stage1(x)
        p3 = self.stage2(x)
        p4 = self.stage3(p3)
        p5 = self.stage4(p4)
        
        return p3, p4, p5
    
    def get_out_channels(self) -> List[int]:
        """Get output channel sizes for P3, P4, P5."""
        return self.out_channels


if __name__ == "__main__":
    for size in ['n', 's', 'm', 'l', 'x']:
        model = CSPDarknet(model_size=size)
        x = torch.randn(1, 3, 640, 640)
        p3, p4, p5 = model(x)
        
        params = sum(p.numel() for p in model.parameters()) / 1e6
        print(f"YOLOv11{size} backbone:")
        print(f"  Parameters: {params:.2f}M")
        print(f"  P3: {p3.shape}, P4: {p4.shape}, P5: {p5.shape}")
        print(f"  Channels: {model.get_out_channels()}")
        print()


In [ ]:
%%writefile yolov11/neck.py
"""
YOLOv11 Neck - PANet (Path Aggregation Network)
Combines FPN (top-down) and PAN (bottom-up) pathways with C3k2 blocks
"""

import torch
import torch.nn as nn
from typing import Tuple, List

from .blocks import Conv, C3k2, Upsample


class PANet(nn.Module):
    """
    YOLOv11 Neck combining FPN and PAN with C3k2 blocks
    
    Architecture:
        FPN (Top-down):
            P5 -> Upsample + Concat(P4) -> C3k2 -> N4
            N4 -> Upsample + Concat(P3) -> C3k2 -> N3
        
        PAN (Bottom-up):
            N3 -> Conv s=2 + Concat(N4) -> C3k2 -> N4'
            N4' -> Conv s=2 + Concat(P5) -> C3k2 -> N5
    
    Args:
        in_channels: List of input channel sizes [P3, P4, P5]
        depth_mult: Depth multiplier for C3k2 blocks
    """
    
    def __init__(
        self,
        in_channels: List[int],
        depth_mult: float = 0.33
    ):
        super().__init__()
        
        c3, c4, c5 = in_channels
        n = max(round(3 * depth_mult), 1)
        
        self.out_channels = [c3, c4, c5]
        
        # FPN (Top-down)
        self.upsample1 = Upsample(scale_factor=2, mode='nearest')
        self.lateral_conv1 = Conv(c5, c4, 1, 1)
        self.fpn_c2f1 = C3k2(c4 + c4, c4, n=n, shortcut=False)
        
        self.upsample2 = Upsample(scale_factor=2, mode='nearest')
        self.lateral_conv2 = Conv(c4, c3, 1, 1)
        self.fpn_c2f2 = C3k2(c3 + c3, c3, n=n, shortcut=False)
        
        # PAN (Bottom-up)
        self.downsample1 = Conv(c3, c3, 3, 2)
        self.pan_c2f1 = C3k2(c3 + c4, c4, n=n, shortcut=False)
        
        self.downsample2 = Conv(c4, c4, 3, 2)
        self.pan_c2f2 = C3k2(c4 + c5, c5, n=n, shortcut=False)
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize model weights."""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(
        self,
        features: Tuple[torch.Tensor, torch.Tensor, torch.Tensor]
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Forward pass.
        
        Args:
            features: Tuple of (P3, P4, P5) from backbone
        
        Returns:
            Tuple of fused features (N3, N4, N5):
                N3: For small objects
                N4: For medium objects
                N5: For large objects
        """
        p3, p4, p5 = features
        
        # FPN (Top-down)
        fpn_p5 = self.lateral_conv1(p5)
        fpn_p5_up = self.upsample1(fpn_p5)
        fpn_n4 = self.fpn_c2f1(torch.cat([fpn_p5_up, p4], dim=1))
        
        fpn_n4_reduced = self.lateral_conv2(fpn_n4)
        fpn_n4_up = self.upsample2(fpn_n4_reduced)
        fpn_n3 = self.fpn_c2f2(torch.cat([fpn_n4_up, p3], dim=1))
        
        # PAN (Bottom-up)
        pan_n3_down = self.downsample1(fpn_n3)
        pan_n4 = self.pan_c2f1(torch.cat([pan_n3_down, fpn_n4], dim=1))
        
        pan_n4_down = self.downsample2(pan_n4)
        pan_n5 = self.pan_c2f2(torch.cat([pan_n4_down, p5], dim=1))
        
        return fpn_n3, pan_n4, pan_n5
    
    def get_out_channels(self) -> List[int]:
        """Get output channel sizes."""
        return self.out_channels


if __name__ == "__main__":
    in_channels = [128, 256, 512]
    
    neck = PANet(in_channels, depth_mult=0.33)
    
    p3 = torch.randn(1, 128, 80, 80)
    p4 = torch.randn(1, 256, 40, 40)
    p5 = torch.randn(1, 512, 20, 20)
    
    n3, n4, n5 = neck((p3, p4, p5))
    
    params = sum(p.numel() for p in neck.parameters()) / 1e6
    print(f"PANet:")
    print(f"  Parameters: {params:.2f}M")
    print(f"  N3: {n3.shape}, N4: {n4.shape}, N5: {n5.shape}")


In [ ]:
%%writefile yolov11/head.py
"""
YOLOv11 Detection Heads
Supports Detection, Segmentation, and Pose Estimation
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Tuple
import math

from .blocks import Conv, DFL, Proto


# COCO keypoint definitions (17 keypoints)
COCO_KEYPOINTS = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle',
]

# Skeleton connections for visualization
COCO_SKELETON = [
    (0, 1), (0, 2), (1, 3), (2, 4),  # Face
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),  # Arms
    (5, 11), (6, 12), (11, 12),  # Torso
    (11, 13), (13, 15), (12, 14), (14, 16)  # Legs
]


class DetectionHead(nn.Module):
    """
    Anchor-free detection head with decoupled classification and regression.
    
    Uses Distribution Focal Loss (DFL) for box regression.
    
    Args:
        num_classes: Number of detection classes
        in_channels: List of input channels from neck [N3, N4, N5]
        reg_max: Maximum discrete regression value for DFL
    """
    
    def __init__(
        self,
        num_classes: int = 80,
        in_channels: List[int] = [128, 256, 512],
        reg_max: int = 16
    ):
        super().__init__()
        
        self.num_classes = num_classes
        self.reg_max = reg_max
        self.num_outputs_per_anchor = num_classes + 4 * reg_max
        
        self.dfl = DFL(reg_max)
        
        self.cls_convs = nn.ModuleList()
        self.reg_convs = nn.ModuleList()
        self.cls_preds = nn.ModuleList()
        self.reg_preds = nn.ModuleList()
        
        for ch in in_channels:
            self.cls_convs.append(nn.Sequential(
                Conv(ch, ch, 3, 1),
                Conv(ch, ch, 3, 1)
            ))
            
            self.reg_convs.append(nn.Sequential(
                Conv(ch, ch, 3, 1),
                Conv(ch, ch, 3, 1)
            ))
            
            self.cls_preds.append(nn.Conv2d(ch, num_classes, 1))
            self.reg_preds.append(nn.Conv2d(ch, 4 * reg_max, 1))
        
        self._initialize_biases()
    
    def _initialize_biases(self):
        """Initialize prediction biases for training stability."""
        for cls_pred in self.cls_preds:
            b = cls_pred.bias.view(-1, )
            b.data.fill_(-math.log((1 - 0.01) / 0.01))  # Prior probability 0.01
            cls_pred.bias = nn.Parameter(b, requires_grad=True)
    
    def forward(
        self,
        features: Tuple[torch.Tensor, ...]
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass.
        
        Args:
            features: Tuple of feature maps (N3, N4, N5) from neck
        
        Returns:
            Tuple of (cls_outputs, reg_outputs) per scale
        """
        cls_outputs = []
        reg_outputs = []
        
        for i, feat in enumerate(features):
            cls_feat = self.cls_convs[i](feat)
            cls_out = self.cls_preds[i](cls_feat)
            
            reg_feat = self.reg_convs[i](feat)
            reg_out = self.reg_preds[i](reg_feat)
            
            cls_outputs.append(cls_out)
            reg_outputs.append(reg_out)
        
        return cls_outputs, reg_outputs
    
    def decode_boxes(
        self,
        reg_outputs: List[torch.Tensor],
        anchors: List[torch.Tensor],
        strides: List[int]
    ) -> torch.Tensor:
        """
        Decode box predictions using DFL.
        
        Args:
            reg_outputs: List of regression outputs per scale
            anchors: List of anchor points per scale
            strides: List of strides for each scale
        
        Returns:
            Decoded boxes in xyxy format
        """
        boxes = []
        
        for reg_out, anchor, stride in zip(reg_outputs, anchors, strides):
            b, _, h, w = reg_out.shape
            
            reg_dist = self.dfl(reg_out)
            reg_dist = reg_dist.view(b, 4, -1).permute(0, 2, 1)
            anchor = anchor.view(-1, 2)
            
            lt = reg_dist[..., :2]
            rb = reg_dist[..., 2:]
            
            x1y1 = anchor - lt * stride
            x2y2 = anchor + rb * stride
            
            boxes.append(torch.cat([x1y1, x2y2], dim=-1))
        
        return torch.cat(boxes, dim=1)


class SegmentationHead(nn.Module):
    """
    Instance segmentation head extending detection with mask coefficients.
    
    Uses prototype masks linearly combined with per-instance coefficients.
    
    Args:
        num_classes: Number of detection classes
        in_channels: List of input channels from neck
        num_protos: Number of prototype masks
        proto_channels: Channels for prototype generation
        reg_max: Maximum discrete regression value for DFL
    """
    
    def __init__(
        self,
        num_classes: int = 80,
        in_channels: List[int] = [128, 256, 512],
        num_protos: int = 32,
        proto_channels: int = 256,
        reg_max: int = 16
    ):
        super().__init__()
        
        self.num_classes = num_classes
        self.num_protos = num_protos
        self.reg_max = reg_max
        
        self.dfl = DFL(reg_max)
        self.proto = Proto(in_channels[0], proto_channels, num_protos)
        
        self.cls_convs = nn.ModuleList()
        self.reg_convs = nn.ModuleList()
        self.cls_preds = nn.ModuleList()
        self.reg_preds = nn.ModuleList()
        self.mask_preds = nn.ModuleList()
        
        for ch in in_channels:
            self.cls_convs.append(nn.Sequential(
                Conv(ch, ch, 3, 1),
                Conv(ch, ch, 3, 1)
            ))
            
            self.reg_convs.append(nn.Sequential(
                Conv(ch, ch, 3, 1),
                Conv(ch, ch, 3, 1)
            ))
            
            self.cls_preds.append(nn.Conv2d(ch, num_classes, 1))
            self.reg_preds.append(nn.Conv2d(ch, 4 * reg_max, 1))
            self.mask_preds.append(nn.Conv2d(ch, num_protos, 1))
        
        self._initialize_biases()
    
    def _initialize_biases(self):
        """Initialize prediction biases."""
        for cls_pred in self.cls_preds:
            b = cls_pred.bias.view(-1, )
            b.data.fill_(-math.log((1 - 0.01) / 0.01))
            cls_pred.bias = nn.Parameter(b, requires_grad=True)
    
    def forward(
        self,
        features: Tuple[torch.Tensor, ...]
    ) -> Tuple[List[torch.Tensor], List[torch.Tensor], List[torch.Tensor], torch.Tensor]:
        """
        Forward pass.
        
        Args:
            features: Tuple of feature maps (N3, N4, N5) from neck
        
        Returns:
            Tuple of (cls_outputs, reg_outputs, mask_coeffs, protos)
        """
        cls_outputs = []
        reg_outputs = []
        mask_outputs = []
        
        protos = self.proto(features[0])
        
        for i, feat in enumerate(features):
            cls_feat = self.cls_convs[i](feat)
            cls_out = self.cls_preds[i](cls_feat)
            
            reg_feat = self.reg_convs[i](feat)
            reg_out = self.reg_preds[i](reg_feat)
            mask_out = self.mask_preds[i](reg_feat)
            
            cls_outputs.append(cls_out)
            reg_outputs.append(reg_out)
            mask_outputs.append(mask_out)
        
        return cls_outputs, reg_outputs, mask_outputs, protos
    
    def assemble_masks(
        self,
        mask_coeffs: torch.Tensor,
        protos: torch.Tensor,
        boxes: torch.Tensor,
        img_size: Tuple[int, int]
    ) -> torch.Tensor:
        """
        Assemble instance masks from prototypes and coefficients.
        
        Args:
            mask_coeffs: Mask coefficients (N, num_protos)
            protos: Prototype masks (B, num_protos, H, W)
            boxes: Detection boxes in xyxy format (N, 4)
            img_size: Original image size (H, W)
        
        Returns:
            Instance masks (N, H, W)
        """
        ph, pw = protos.shape[2:]
        protos_flat = protos[0].view(self.num_protos, -1)
        
        masks = torch.mm(mask_coeffs, protos_flat)
        masks = masks.sigmoid().view(-1, ph, pw)
        
        masks = self._crop_masks(masks, boxes, img_size)
        
        return masks
    
    def _crop_masks(
        self,
        masks: torch.Tensor,
        boxes: torch.Tensor,
        img_size: Tuple[int, int]
    ) -> torch.Tensor:
        """Crop and resize masks to image size."""
        h, w = img_size
        n = masks.shape[0]
        
        masks = F.interpolate(
            masks.unsqueeze(1),
            size=(h, w),
            mode='bilinear',
            align_corners=False
        ).squeeze(1)
        
        output_masks = torch.zeros((n, h, w), device=masks.device)
        
        for i in range(n):
            x1, y1, x2, y2 = boxes[i].int().tolist()
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            
            output_masks[i, y1:y2, x1:x2] = masks[i, y1:y2, x1:x2]
        
        return output_masks


class PoseHead(nn.Module):
    """
    Pose estimation head extending detection with keypoint prediction.
    
    Predicts 17 COCO keypoints per detection.
    
    Args:
        num_classes: Number of detection classes (typically 1 for person)
        in_channels: List of input channels from neck
        num_keypoints: Number of keypoints to predict (17 for COCO)
        reg_max: Maximum discrete regression value for DFL
    """
    
    def __init__(
        self,
        num_classes: int = 1,
        in_channels: List[int] = [128, 256, 512],
        num_keypoints: int = 17,
        reg_max: int = 16
    ):
        super().__init__()
        
        self.num_classes = num_classes
        self.num_keypoints = num_keypoints
        self.reg_max = reg_max
        
        self.dfl = DFL(reg_max)
        
        self.cls_convs = nn.ModuleList()
        self.reg_convs = nn.ModuleList()
        self.kpt_convs = nn.ModuleList()
        
        self.cls_preds = nn.ModuleList()
        self.reg_preds = nn.ModuleList()
        self.kpt_preds = nn.ModuleList()
        
        for ch in in_channels:
            self.cls_convs.append(nn.Sequential(
                Conv(ch, ch, 3, 1),
                Conv(ch, ch, 3, 1)
            ))
            
            self.reg_convs.append(nn.Sequential(
                Conv(ch, ch, 3, 1),
                Conv(ch, ch, 3, 1)
            ))
            
            self.kpt_convs.append(nn.Sequential(
                Conv(ch, ch, 3, 1),
                Conv(ch, ch, 3, 1)
            ))
            
            self.cls_preds.append(nn.Conv2d(ch, num_classes, 1))
            self.reg_preds.append(nn.Conv2d(ch, 4 * reg_max, 1))
            self.kpt_preds.append(nn.Conv2d(ch, num_keypoints * 3, 1))
        
        self._initialize_biases()
    
    def _initialize_biases(self):
        """Initialize prediction biases."""
        for cls_pred in self.cls_preds:
            b = cls_pred.bias.view(-1, )
            b.data.fill_(-math.log((1 - 0.01) / 0.01))
            cls_pred.bias = nn.Parameter(b, requires_grad=True)
    
    def forward(
        self,
        features: Tuple[torch.Tensor, ...]
    ) -> Tuple[List[torch.Tensor], List[torch.Tensor], List[torch.Tensor]]:
        """
        Forward pass.
        
        Args:
            features: Tuple of feature maps (N3, N4, N5) from neck
        
        Returns:
            Tuple of (cls_outputs, reg_outputs, kpt_outputs)
        """
        cls_outputs = []
        reg_outputs = []
        kpt_outputs = []
        
        for i, feat in enumerate(features):
            cls_feat = self.cls_convs[i](feat)
            cls_out = self.cls_preds[i](cls_feat)
            
            reg_feat = self.reg_convs[i](feat)
            reg_out = self.reg_preds[i](reg_feat)
            
            kpt_feat = self.kpt_convs[i](feat)
            kpt_out = self.kpt_preds[i](kpt_feat)
            
            cls_outputs.append(cls_out)
            reg_outputs.append(reg_out)
            kpt_outputs.append(kpt_out)
        
        return cls_outputs, reg_outputs, kpt_outputs
    
    def decode_keypoints(
        self,
        kpt_outputs: List[torch.Tensor],
        anchors: List[torch.Tensor],
        strides: List[int]
    ) -> torch.Tensor:
        """
        Decode keypoint predictions.
        
        Args:
            kpt_outputs: Keypoint predictions per scale
            anchors: Anchor points per scale
            strides: Strides for each scale
        
        Returns:
            Decoded keypoints (B, N, num_keypoints, 3) with (x, y, visibility)
        """
        keypoints = []
        
        for kpt_out, anchor, stride in zip(kpt_outputs, anchors, strides):
            b, c, h, w = kpt_out.shape
            
            kpt_out = kpt_out.view(b, self.num_keypoints, 3, -1)
            kpt_out = kpt_out.permute(0, 3, 1, 2)
            
            anchor = anchor.view(-1, 2)
            
            xy = kpt_out[..., :2] * stride + anchor.unsqueeze(0).unsqueeze(2)
            visibility = kpt_out[..., 2:].sigmoid()
            
            keypoints.append(torch.cat([xy, visibility], dim=-1))
        
        return torch.cat(keypoints, dim=1)


if __name__ == "__main__":
    in_channels = [128, 256, 512]
    
    print("Testing DetectionHead...")
    det_head = DetectionHead(80, in_channels)
    feats = [torch.randn(1, c, 80//(2**i), 80//(2**i)) for i, c in enumerate(in_channels)]
    cls_out, reg_out = det_head(feats)
    print(f"  Cls shapes: {[c.shape for c in cls_out]}")
    print(f"  Reg shapes: {[r.shape for r in reg_out]}")
    
    print("\nTesting SegmentationHead...")
    seg_head = SegmentationHead(80, in_channels)
    cls_out, reg_out, mask_out, protos = seg_head(feats)
    print(f"  Cls shapes: {[c.shape for c in cls_out]}")
    print(f"  Mask shapes: {[m.shape for m in mask_out]}")
    print(f"  Proto shape: {protos.shape}")
    
    print("\nTesting PoseHead...")
    pose_head = PoseHead(1, in_channels, 17)
    cls_out, reg_out, kpt_out = pose_head(feats)
    print(f"  Cls shapes: {[c.shape for c in cls_out]}")
    print(f"  Kpt shapes: {[k.shape for k in kpt_out]}")


In [ ]:
%%writefile yolov11/model.py
"""
YOLOv11 Main Model
Combines backbone, neck, and task-specific heads
"""

import torch
import torch.nn as nn
from typing import Dict, List, Tuple, Optional, Union, Any

from .backbone import CSPDarknet, MODEL_SCALES
from .neck import PANet
from .head import DetectionHead, SegmentationHead, PoseHead


class YOLOv11(nn.Module):
    """
    YOLOv11 Model - Unified architecture for Detection, Segmentation, and Pose Estimation
    
    Key improvements over YOLOv8:
        - C3k2 blocks replacing C2f for better efficiency
        - C2PSA spatial attention in backbone
        - Fewer parameters with higher accuracy
    
    Architecture:
        Input -> Backbone (CSPDarknet+C2PSA) -> Neck (PANet+C3k2) -> Head (task-specific)
    
    Args:
        num_classes: Number of detection classes
        task: Task type - 'detect', 'segment', or 'pose'
        model_size: Model size - 'n', 's', 'm', 'l', 'x'
        num_keypoints: Number of keypoints for pose task (default: 17 COCO)
        reg_max: Maximum regression value for DFL
    
    Example:
        >>> model = YOLOv11(num_classes=80, task='detect', model_size='s')
        >>> x = torch.randn(1, 3, 640, 640)
        >>> outputs = model(x)
    """
    
    SUPPORTED_TASKS = ['detect', 'segment', 'pose']
    
    def __init__(
        self,
        num_classes: int = 80,
        task: str = 'detect',
        model_size: str = 's',
        num_keypoints: int = 17,
        reg_max: int = 16,
        input_size: int = 640
    ):
        super().__init__()
        
        if task not in self.SUPPORTED_TASKS:
            raise ValueError(f"Task must be one of {self.SUPPORTED_TASKS}, got {task}")
        
        if model_size not in MODEL_SCALES:
            raise ValueError(f"Model size must be one of {list(MODEL_SCALES.keys())}, got {model_size}")
        
        self.task = task
        self.model_size = model_size
        self.num_classes = num_classes
        self.num_keypoints = num_keypoints
        self.reg_max = reg_max
        self.input_size = input_size
        
        # Get depth multiplier for neck
        depth_mult, _ = MODEL_SCALES[model_size]
        
        # Build backbone
        self.backbone = CSPDarknet(model_size=model_size)
        backbone_channels = self.backbone.get_out_channels()
        
        # Build neck
        self.neck = PANet(backbone_channels, depth_mult=depth_mult)
        neck_channels = self.neck.get_out_channels()
        
        # Build task-specific head
        if task == 'detect':
            self.head = DetectionHead(
                num_classes=num_classes,
                in_channels=neck_channels,
                reg_max=reg_max
            )
        elif task == 'segment':
            self.head = SegmentationHead(
                num_classes=num_classes,
                in_channels=neck_channels,
                reg_max=reg_max
            )
        elif task == 'pose':
            self.head = PoseHead(
                num_classes=1,  # Pose typically only detects 'person'
                in_channels=neck_channels,
                num_keypoints=num_keypoints,
                reg_max=reg_max
            )
        
        # Pre-compute strides and anchor grids
        self.strides = [8, 16, 32]  # P3, P4, P5 strides
        self._init_anchors()
    
    def _init_anchors(self):
        """Initialize anchor point grids for each scale."""
        self.anchor_grids = []
        
        for stride in self.strides:
            grid_size = self.input_size // stride
            
            # Create grid of anchor points
            ys, xs = torch.meshgrid(
                torch.arange(grid_size),
                torch.arange(grid_size),
                indexing='ij'
            )
            
            # Anchor points at center of each grid cell
            grid = torch.stack([xs, ys], dim=-1).float() + 0.5
            grid = grid * stride  # Scale to input coordinates
            
            self.register_buffer(f'anchor_grid_{stride}', grid)
            self.anchor_grids.append(grid)
    
    def forward(
        self,
        x: torch.Tensor
    ) -> Union[Dict[str, Any], List[torch.Tensor]]:
        """
        Forward pass through the model.
        
        Args:
            x: Input tensor of shape (B, 3, H, W)
        
        Returns:
            Dictionary with task-specific outputs:
                - detect: {'cls': [...], 'reg': [...]}
                - segment: {'cls': [...], 'reg': [...], 'mask': [...], 'proto': Tensor}
                - pose: {'cls': [...], 'reg': [...], 'kpt': [...]}
        """
        # Backbone
        features = self.backbone(x)
        
        # Neck
        features = self.neck(features)
        
        # Head
        outputs = self.head(features)
        
        # Format outputs based on task
        is_export = torch.onnx.is_in_onnx_export() or getattr(torch.jit, 'is_tracing', lambda: False)()
        
        if self.task == 'detect':
            cls_outputs, reg_outputs = outputs
            if is_export:
                # Return list for ONNX compatibility (no integers)
                return cls_outputs + reg_outputs
            return {
                'cls': cls_outputs,
                'reg': reg_outputs,
                'strides': self.strides
            }
        
        elif self.task == 'segment':
            cls_outputs, reg_outputs, mask_outputs, protos = outputs
            if is_export:
                return cls_outputs + reg_outputs + mask_outputs + [protos]
            return {
                'cls': cls_outputs,
                'reg': reg_outputs,
                'mask': mask_outputs,
                'proto': protos,
                'strides': self.strides
            }
        
        elif self.task == 'pose':
            cls_outputs, reg_outputs, kpt_outputs = outputs
            if is_export:
                return cls_outputs + reg_outputs + kpt_outputs
            return {
                'cls': cls_outputs,
                'reg': reg_outputs,
                'kpt': kpt_outputs,
                'strides': self.strides
            }
        
        return {}  # Default empty dict to satisfy type checker
    
    def get_anchors(self, device: torch.device) -> List[torch.Tensor]:
        """Get anchor grids on specified device."""
        return [getattr(self, f'anchor_grid_{s}').to(device) for s in self.strides]
    
    def fuse(self):
        """Fuse Conv2d + BatchNorm2d layers for inference optimization."""
        if not hasattr(self, 'fused'):
            self.fused = False
            
        if self.fused:
            print("Model already fused.")
            return self

        for m in self.modules():
            if hasattr(m, 'forward_fuse'):
                # Fuse conv and bn
                if hasattr(m, 'conv') and hasattr(m, 'bn') and isinstance(m.bn, nn.BatchNorm2d):
                    m.conv = self._fuse_conv_bn(m.conv, m.bn)
                    m.bn = nn.Identity()
                    m.forward = m.forward_fuse
        
        self.fused = True
        return self
    
    @staticmethod
    def _fuse_conv_bn(conv: nn.Conv2d, bn: nn.BatchNorm2d) -> nn.Conv2d:
        """Fuse convolution and batch normalization layers."""
        # Get parameters
        w_conv = conv.weight
        if conv.bias is not None:
            b_conv = conv.bias
        else:
            b_conv = torch.zeros(conv.out_channels, device=w_conv.device)
        
        w_bn = bn.weight
        b_bn = bn.bias
        running_mean = bn.running_mean
        running_var = bn.running_var
        eps = bn.eps
        
        # Calculate fused weights and bias
        std = torch.sqrt(running_var + eps)
        w_fused = w_conv * (w_bn / std).view(-1, 1, 1, 1)
        b_fused = (b_conv - running_mean) * w_bn / std + b_bn
        
        # Create fused conv
        fused_conv = nn.Conv2d(
            conv.in_channels,
            conv.out_channels,
            conv.kernel_size,
            conv.stride,
            conv.padding,
            conv.dilation,
            conv.groups,
            True  # Include bias
        )
        
        fused_conv.weight.data = w_fused
        fused_conv.bias.data = b_fused
        
        return fused_conv
    
    def info(self, verbose: bool = True) -> Dict[str, any]:
        """Print model information."""
        # Count parameters
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        
        # Count layers
        num_layers = len(list(self.modules()))
        
        info = {
            'task': self.task,
            'model_size': self.model_size,
            'num_classes': self.num_classes,
            'input_size': self.input_size,
            'total_params': total_params,
            'trainable_params': trainable_params,
            'num_layers': num_layers
        }
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"YOLOv11{self.model_size}-{self.task}")
            print(f"{'='*60}")
            print(f"Task:             {self.task}")
            print(f"Model Size:       {self.model_size}")
            print(f"Num Classes:      {self.num_classes}")
            print(f"Input Size:       {self.input_size}x{self.input_size}")
            print(f"Total Params:     {total_params:,} ({total_params/1e6:.2f}M)")
            print(f"Trainable Params: {trainable_params:,}")
            print(f"Num Layers:       {num_layers}")
            print(f"{'='*60}\n")
        
        return info


def create_model(
    num_classes: int = 80,
    task: str = 'detect',
    model_size: str = 's',
    pretrained: bool = False
) -> YOLOv11:
    """
    Factory function to create YOLOv11 model.
    
    Args:
        num_classes: Number of classes to detect
        task: One of 'detect', 'segment', 'pose'
        model_size: One of 'n', 's', 'm', 'l', 'x'
        pretrained: Whether to load pretrained weights (not implemented)
    
    Returns:
        YOLOv11 model instance
    """
    model = YOLOv11(
        num_classes=num_classes,
        task=task,
        model_size=model_size
    )
    
    if pretrained:
        # TODO: Load pretrained weights
        print("Warning: Pretrained weights not yet available")
    
    return model


if __name__ == "__main__":
    # Test all model variants
    print("Testing YOLOv11 models...\n")
    
    for task in ['detect', 'segment', 'pose']:
        for size in ['n', 's', 'm']:
            model = YOLOv11(num_classes=80, task=task, model_size=size)
            model.info()
            
            # Test forward pass
            x = torch.randn(1, 3, 640, 640)
            with torch.no_grad():
                out = model(x)
            
            print(f"  Outputs: {list(out.keys())}")
            for k, v in out.items():
                if isinstance(v, list):
                    print(f"    {k}: {[t.shape for t in v]}")
                elif isinstance(v, torch.Tensor):
                    print(f"    {k}: {v.shape}")
            print()


In [ ]:
%%writefile yolov11/losses/__init__.py
"""
YOLOv11 Loss Functions Package
"""

from .box_loss import CIoULoss, DFLoss
from .cls_loss import ClassificationLoss
from .seg_loss import SegmentationLoss
from .pose_loss import PoseLoss, OKSLoss
from .combined_loss import YOLOv11Loss

__all__ = [
    "CIoULoss",
    "DFLoss",
    "ClassificationLoss",
    "SegmentationLoss",
    "PoseLoss",
    "OKSLoss",
    "YOLOv11Loss"
]


In [ ]:
%%writefile yolov11/losses/box_loss.py
"""
Box Regression Losses for YOLOv11
Implements CIoU Loss and Distribution Focal Loss (DFL)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple


def bbox_iou(
    box1: torch.Tensor,
    box2: torch.Tensor,
    xywh: bool = False,
    GIoU: bool = False,
    DIoU: bool = False,
    CIoU: bool = True,
    eps: float = 1e-7
) -> torch.Tensor:
    """
    Calculate Intersection over Union (IoU) between boxes.
    
    Supports multiple IoU variants:
    - Standard IoU
    - GIoU (Generalized IoU)
    - DIoU (Distance IoU)
    - CIoU (Complete IoU)
    
    Args:
        box1: First set of boxes (N, 4)
        box2: Second set of boxes (N, 4) or (M, 4)
        xywh: If True, boxes are in (x, y, w, h) format, else (x1, y1, x2, y2)
        GIoU: Calculate Generalized IoU
        DIoU: Calculate Distance IoU
        CIoU: Calculate Complete IoU (default)
        eps: Small value to avoid division by zero
    
    Returns:
        IoU values (N,) or (N, M)
    """
    # Convert to xyxy format if needed
    if xywh:
        # (x_center, y_center, width, height) -> (x1, y1, x2, y2)
        b1_x1 = box1[..., 0] - box1[..., 2] / 2
        b1_y1 = box1[..., 1] - box1[..., 3] / 2
        b1_x2 = box1[..., 0] + box1[..., 2] / 2
        b1_y2 = box1[..., 1] + box1[..., 3] / 2
        
        b2_x1 = box2[..., 0] - box2[..., 2] / 2
        b2_y1 = box2[..., 1] - box2[..., 3] / 2
        b2_x2 = box2[..., 0] + box2[..., 2] / 2
        b2_y2 = box2[..., 1] + box2[..., 3] / 2
    else:
        b1_x1, b1_y1, b1_x2, b1_y2 = box1[..., 0], box1[..., 1], box1[..., 2], box1[..., 3]
        b2_x1, b2_y1, b2_x2, b2_y2 = box2[..., 0], box2[..., 1], box2[..., 2], box2[..., 3]
    
    # Intersection area
    inter_x1 = torch.max(b1_x1, b2_x1)
    inter_y1 = torch.max(b1_y1, b2_y1)
    inter_x2 = torch.min(b1_x2, b2_x2)
    inter_y2 = torch.min(b1_y2, b2_y2)
    
    inter_area = (inter_x2 - inter_x1).clamp(0) * (inter_y2 - inter_y1).clamp(0)
    
    # Union area
    b1_area = (b1_x2 - b1_x1) * (b1_y2 - b1_y1)
    b2_area = (b2_x2 - b2_x1) * (b2_y2 - b2_y1)
    union_area = b1_area + b2_area - inter_area + eps
    
    # IoU
    iou = inter_area / union_area
    
    if GIoU or DIoU or CIoU:
        # Enclosing box
        enclose_x1 = torch.min(b1_x1, b2_x1)
        enclose_y1 = torch.min(b1_y1, b2_y1)
        enclose_x2 = torch.max(b1_x2, b2_x2)
        enclose_y2 = torch.max(b1_y2, b2_y2)
        
        if GIoU:
            # Generalized IoU
            enclose_area = (enclose_x2 - enclose_x1) * (enclose_y2 - enclose_y1) + eps
            return iou - (enclose_area - union_area) / enclose_area
        
        if DIoU or CIoU:
            # Distance IoU / Complete IoU
            # Diagonal distance of enclosing box
            c_diag = (enclose_x2 - enclose_x1) ** 2 + (enclose_y2 - enclose_y1) ** 2 + eps
            
            # Center distance
            b1_cx = (b1_x1 + b1_x2) / 2
            b1_cy = (b1_y1 + b1_y2) / 2
            b2_cx = (b2_x1 + b2_x2) / 2
            b2_cy = (b2_y1 + b2_y2) / 2
            center_dist = (b1_cx - b2_cx) ** 2 + (b1_cy - b2_cy) ** 2
            
            if DIoU:
                return iou - center_dist / c_diag
            
            if CIoU:
                # Aspect ratio consistency
                b1_w = b1_x2 - b1_x1
                b1_h = b1_y2 - b1_y1
                b2_w = b2_x2 - b2_x1
                b2_h = b2_y2 - b2_y1
                
                v = (4 / (torch.pi ** 2)) * torch.pow(
                    torch.atan(b2_w / (b2_h + eps)) - torch.atan(b1_w / (b1_h + eps)), 
                    2
                )
                
                with torch.no_grad():
                    alpha = v / (1 - iou + v + eps)
                
                return iou - (center_dist / c_diag + v * alpha)
    
    return iou


class CIoULoss(nn.Module):
    """
    Complete IoU Loss for box regression.
    
    CIoU considers:
    1. Overlap area (IoU)
    2. Center point distance
    3. Aspect ratio consistency
    
    Loss = 1 - CIoU
    """
    
    def __init__(self, reduction: str = 'mean'):
        super().__init__()
        self.reduction = reduction
    
    def forward(
        self, 
        pred_boxes: torch.Tensor, 
        target_boxes: torch.Tensor,
        weights: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Calculate CIoU loss.
        
        Args:
            pred_boxes: Predicted boxes (N, 4) in xyxy format
            target_boxes: Target boxes (N, 4) in xyxy format
            weights: Optional sample weights (N,)
        
        Returns:
            CIoU loss value
        """
        ciou = bbox_iou(pred_boxes, target_boxes, CIoU=True)
        loss = 1.0 - ciou
        
        if weights is not None:
            loss = loss * weights
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


class DFLoss(nn.Module):
    """
    Distribution Focal Loss for box regression.
    
    DFL represents box coordinates as probability distributions
    over discrete bins [0, 1, 2, ..., reg_max-1] instead of
    directly regressing float values.
    
    This helps capture uncertainty in box localization.
    
    Args:
        reg_max: Maximum regression range (default: 16)
    """
    
    def __init__(self, reg_max: int = 16):
        super().__init__()
        self.reg_max = reg_max
    
    def forward(
        self,
        pred_dist: torch.Tensor,
        target: torch.Tensor
    ) -> torch.Tensor:
        """
        Calculate Distribution Focal Loss.
        
        Args:
            pred_dist: Predicted distribution logits (N, reg_max)
            target: Target regression values (N,) in range [0, reg_max-1]
        
        Returns:
            DFL loss value
        """
        # Get left and right bins
        target = target.clamp(0, self.reg_max - 1 - 0.01)
        tl = target.long()  # left bin index
        tr = tl + 1  # right bin index
        
        # Weight for left bin (closer to target = higher weight)
        wl = tr.float() - target
        wr = 1 - wl
        
        # Cross entropy loss for left and right bins
        loss = (
            F.cross_entropy(pred_dist, tl, reduction='none') * wl +
            F.cross_entropy(pred_dist, tr, reduction='none') * wr
        )
        
        return loss.mean()


class BboxLoss(nn.Module):
    """
    Combined Box Loss for YOLOv11.
    
    Combines:
    - CIoU Loss for decoded boxes
    - DFL Loss for distribution regression
    
    Args:
        reg_max: Maximum regression range for DFL
    """
    
    def __init__(self, reg_max: int = 16):
        super().__init__()
        self.reg_max = reg_max
        self.ciou_loss = CIoULoss(reduction='none')
        self.dfl_loss = DFLoss(reg_max)
    
    def forward(
        self,
        pred_dist: torch.Tensor,
        pred_boxes: torch.Tensor,
        anchor_points: torch.Tensor,
        target_boxes: torch.Tensor,
        target_scores: torch.Tensor,
        target_scores_sum: torch.Tensor,
        fg_mask: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Calculate combined box loss.
        
        Args:
            pred_dist: Predicted distributions (B, N, 4*reg_max)
            pred_boxes: Decoded predicted boxes (B, N, 4)
            anchor_points: Anchor points (N, 2)
            target_boxes: Target boxes (B, N, 4)
            target_scores: Target scores/weights (B, N)
            target_scores_sum: Sum of target scores
            fg_mask: Foreground mask (B, N)
        
        Returns:
            Tuple of (ciou_loss, dfl_loss)
        """
        weight = target_scores.sum(-1)[fg_mask].unsqueeze(-1)
        
        # CIoU loss
        ciou = bbox_iou(pred_boxes[fg_mask], target_boxes[fg_mask], CIoU=True)
        loss_iou = ((1.0 - ciou) * weight).sum() / target_scores_sum
        
        # DFL loss
        target_ltrb = self._bbox2dist(anchor_points, target_boxes, self.reg_max)
        loss_dfl = self._df_loss(pred_dist[fg_mask].view(-1, self.reg_max), target_ltrb[fg_mask])
        loss_dfl = (loss_dfl * weight).sum() / target_scores_sum
        
        return loss_iou, loss_dfl
    
    def _bbox2dist(
        self,
        anchor_points: torch.Tensor,
        bbox: torch.Tensor,
        reg_max: int
    ) -> torch.Tensor:
        """Convert bbox to distance format (left, top, right, bottom)."""
        x1y1, x2y2 = bbox.chunk(2, -1)
        lt = anchor_points - x1y1
        rb = x2y2 - anchor_points
        return torch.cat([lt, rb], -1).clamp(0, reg_max - 0.01)
    
    def _df_loss(
        self,
        pred_dist: torch.Tensor,
        target: torch.Tensor
    ) -> torch.Tensor:
        """Calculate DFL loss."""
        target = target.view(-1)
        tl = target.long()
        tr = tl + 1
        wl = tr.float() - target
        wr = 1 - wl
        
        return (
            F.cross_entropy(pred_dist, tl, reduction='none') * wl +
            F.cross_entropy(pred_dist, tr.clamp(max=self.reg_max - 1), reduction='none') * wr
        ).view(-1, 4).mean(-1)


if __name__ == "__main__":
    # Test losses
    print("Testing Box Losses...")
    
    # CIoU Loss
    pred = torch.tensor([[10, 10, 50, 50], [20, 20, 80, 80]], dtype=torch.float32)
    target = torch.tensor([[15, 15, 55, 55], [25, 25, 75, 75]], dtype=torch.float32)
    
    ciou_loss = CIoULoss()
    loss = ciou_loss(pred, target)
    print(f"CIoU Loss: {loss.item():.4f}")
    
    # IoU variants
    print(f"IoU: {bbox_iou(pred, target, CIoU=False).mean().item():.4f}")
    print(f"GIoU: {bbox_iou(pred, target, GIoU=True).mean().item():.4f}")
    print(f"DIoU: {bbox_iou(pred, target, DIoU=True).mean().item():.4f}")
    print(f"CIoU: {bbox_iou(pred, target, CIoU=True).mean().item():.4f}")
    
    # DFL Loss
    dfl_loss = DFLoss(reg_max=16)
    pred_dist = torch.randn(10, 16)
    target_val = torch.rand(10) * 15
    loss = dfl_loss(pred_dist, target_val)
    print(f"DFL Loss: {loss.item():.4f}")


In [ ]:
%%writefile yolov11/losses/cls_loss.py
"""
Classification Loss for YOLOv11
Binary Cross Entropy with Task-Aligned Assigner
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional


class ClassificationLoss(nn.Module):
    """
    Classification loss using Binary Cross Entropy.
    
    Supports both soft labels (for task-aligned assignment) 
    and hard labels.
    
    Args:
        use_sigmoid: Use sigmoid activation (BCE) or softmax (CE)
        reduction: Loss reduction method
    """
    
    def __init__(
        self,
        use_sigmoid: bool = True,
        reduction: str = 'mean',
        label_smoothing: float = 0.0
    ):
        super().__init__()
        self.use_sigmoid = use_sigmoid
        self.reduction = reduction
        self.label_smoothing = label_smoothing
        
        if use_sigmoid:
            self.loss_fn = nn.BCEWithLogitsLoss(reduction='none')
        else:
            self.loss_fn = nn.CrossEntropyLoss(
                reduction='none',
                label_smoothing=label_smoothing
            )
    
    def forward(
        self,
        pred: torch.Tensor,
        target: torch.Tensor,
        weight: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Calculate classification loss.
        
        Args:
            pred: Predicted logits (N, num_classes) or (B, N, num_classes)
            target: Target labels (N,) or soft labels (N, num_classes)
            weight: Optional sample weights
        
        Returns:
            Classification loss value
        """
        if self.use_sigmoid:
            # Binary Cross Entropy
            loss = self.loss_fn(pred, target)
            
            if weight is not None:
                loss = loss * weight.unsqueeze(-1)
            
            if self.reduction == 'mean':
                return loss.mean()
            elif self.reduction == 'sum':
                return loss.sum()
            return loss
        else:
            # Cross Entropy
            loss = self.loss_fn(pred, target)
            
            if weight is not None:
                loss = loss * weight
            
            if self.reduction == 'mean':
                return loss.mean()
            elif self.reduction == 'sum':
                return loss.sum()
            return loss


class FocalLoss(nn.Module):
    """
    Focal Loss for handling class imbalance.
    
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    
    Args:
        alpha: Weighting factor for positive samples
        gamma: Focusing parameter (higher = more focus on hard examples)
        reduction: Loss reduction method
    """
    
    def __init__(
        self,
        alpha: float = 0.25,
        gamma: float = 2.0,
        reduction: str = 'mean'
    ):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(
        self,
        pred: torch.Tensor,
        target: torch.Tensor
    ) -> torch.Tensor:
        """
        Calculate Focal Loss.
        
        Args:
            pred: Predicted logits (N, num_classes)
            target: Target labels (N, num_classes) one-hot or soft
        
        Returns:
            Focal loss value
        """
        # Apply sigmoid to get probabilities
        p = torch.sigmoid(pred)
        
        # Calculate cross entropy
        ce_loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        
        # Calculate p_t
        p_t = p * target + (1 - p) * (1 - target)
        
        # Calculate focal weight
        focal_weight = (1 - p_t) ** self.gamma
        
        # Apply alpha weighting
        alpha_t = self.alpha * target + (1 - self.alpha) * (1 - target)
        
        # Final focal loss
        loss = alpha_t * focal_weight * ce_loss
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


class VarifocalLoss(nn.Module):
    """
    Varifocal Loss - improved focal loss with IoU-aware targets.
    
    For positive samples: loss = -q * (q * log(p) + (1-q) * log(1-p))
    For negative samples: loss = -alpha * p^gamma * log(1-p)
    
    where q is the soft target (IoU score).
    
    Args:
        alpha: Weighting factor for negative samples
        gamma: Focusing parameter for negative samples
        reduction: Loss reduction method
    """
    
    def __init__(
        self,
        alpha: float = 0.75,
        gamma: float = 2.0,
        reduction: str = 'mean'
    ):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(
        self,
        pred: torch.Tensor,
        target: torch.Tensor
    ) -> torch.Tensor:
        """
        Calculate Varifocal Loss.
        
        Args:
            pred: Predicted logits (N, num_classes)
            target: Soft target labels (N, num_classes) with IoU scores
        
        Returns:
            Varifocal loss value
        """
        p = torch.sigmoid(pred)
        
        # Binary cross entropy
        ce_loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        
        # Positive mask
        pos_mask = target > 0
        
        # Varifocal weight
        weight = torch.zeros_like(target)
        weight[pos_mask] = target[pos_mask]  # q for positives
        weight[~pos_mask] = self.alpha * p[~pos_mask].pow(self.gamma)  # alpha * p^gamma for negatives
        
        loss = weight * ce_loss
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


if __name__ == "__main__":
    # Test classification losses
    print("Testing Classification Losses...")
    
    pred = torch.randn(10, 80)
    target = torch.zeros(10, 80)
    target[range(10), torch.randint(0, 80, (10,))] = 1
    
    # BCE Loss
    bce_loss = ClassificationLoss(use_sigmoid=True)
    loss = bce_loss(pred, target)
    print(f"BCE Loss: {loss.item():.4f}")
    
    # Focal Loss
    focal_loss = FocalLoss()
    loss = focal_loss(pred, target)
    print(f"Focal Loss: {loss.item():.4f}")
    
    # Varifocal Loss with soft targets
    soft_target = target * torch.rand(10, 1)  # Simulate IoU scores
    vfl = VarifocalLoss()
    loss = vfl(pred, soft_target)
    print(f"Varifocal Loss: {loss.item():.4f}")


In [ ]:
%%writefile yolov11/losses/seg_loss.py
"""
Segmentation Loss for YOLOv11
Instance mask loss using BCE and Dice
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple


class DiceLoss(nn.Module):
    """
    Dice Loss for segmentation.
    
    Dice = 2 * |A ∩ B| / (|A| + |B|)
    Loss = 1 - Dice
    
    Good for handling class imbalance in segmentation.
    """
    
    def __init__(
        self,
        smooth: float = 1.0,
        reduction: str = 'mean'
    ):
        super().__init__()
        self.smooth = smooth
        self.reduction = reduction
    
    def forward(
        self,
        pred: torch.Tensor,
        target: torch.Tensor
    ) -> torch.Tensor:
        """
        Calculate Dice loss.
        
        Args:
            pred: Predicted mask probabilities (N, H, W) or (N, 1, H, W)
            target: Target binary masks (N, H, W) or (N, 1, H, W)
        
        Returns:
            Dice loss value
        """
        pred = pred.flatten(1)
        target = target.flatten(1)
        
        intersection = (pred * target).sum(1)
        union = pred.sum(1) + target.sum(1)
        
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        loss = 1.0 - dice
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


class SegmentationLoss(nn.Module):
    """
    Instance Segmentation Loss for YOLOv11.
    
    Combines:
    - Binary Cross Entropy for pixel-wise classification
    - Dice Loss for region overlap
    
    Args:
        bce_weight: Weight for BCE loss
        dice_weight: Weight for Dice loss
        reduction: Loss reduction method
    """
    
    def __init__(
        self,
        bce_weight: float = 1.0,
        dice_weight: float = 1.0,
        reduction: str = 'mean'
    ):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.reduction = reduction
        
        self.bce_loss = nn.BCEWithLogitsLoss(reduction='none')
        self.dice_loss = DiceLoss(reduction='none')
    
    def forward(
        self,
        pred_masks: torch.Tensor,
        target_masks: torch.Tensor,
        mask_weights: torch.Tensor = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Calculate segmentation loss.
        
        Args:
            pred_masks: Predicted mask logits (N, H, W)
            target_masks: Target binary masks (N, H, W)
            mask_weights: Optional per-instance weights (N,)
        
        Returns:
            Tuple of (total_loss, dict with individual losses)
        """
        # BCE Loss
        bce = self.bce_loss(pred_masks, target_masks)
        
        if mask_weights is not None:
            bce = bce * mask_weights.view(-1, 1, 1)
        
        bce = bce.mean(dim=(1, 2))  # Mean over spatial dims
        
        # Dice Loss
        pred_probs = torch.sigmoid(pred_masks)
        dice = self.dice_loss(pred_probs, target_masks)
        
        if mask_weights is not None:
            dice = dice * mask_weights
        
        # Combine losses
        if self.reduction == 'mean':
            bce_loss = bce.mean()
            dice_loss = dice.mean()
        elif self.reduction == 'sum':
            bce_loss = bce.sum()
            dice_loss = dice.sum()
        else:
            bce_loss = bce
            dice_loss = dice
        
        total_loss = self.bce_weight * bce_loss + self.dice_weight * dice_loss
        
        return total_loss, {'bce': bce_loss, 'dice': dice_loss}
    
    def single_mask_loss(
        self,
        pred: torch.Tensor,
        target: torch.Tensor,
        area: torch.Tensor,
        proto: torch.Tensor,
        xyxy: torch.Tensor
    ) -> torch.Tensor:
        """
        Calculate loss for a single instance mask.
        
        Args:
            pred: Mask coefficients (num_protos,)
            target: Target mask (H, W)
            area: Instance area for weighting
            proto: Prototype masks (num_protos, H, W)
            xyxy: Bounding box for cropping
        
        Returns:
            Single instance mask loss
        """
        # Generate mask from coefficients
        pred_mask = (pred @ proto.view(proto.shape[0], -1)).view(proto.shape[1:])
        
        # Crop to bounding box
        x1, y1, x2, y2 = xyxy.int().tolist()
        pred_crop = pred_mask[y1:y2, x1:x2]
        target_crop = target[y1:y2, x1:x2]
        
        # BCE loss within bounding box
        loss = F.binary_cross_entropy_with_logits(pred_crop, target_crop, reduction='mean')
        
        return loss


class MaskIoU(nn.Module):
    """
    Mask IoU calculation for evaluation and quality-aware training.
    """
    
    def __init__(self, threshold: float = 0.5):
        super().__init__()
        self.threshold = threshold
    
    def forward(
        self,
        pred_masks: torch.Tensor,
        target_masks: torch.Tensor
    ) -> torch.Tensor:
        """
        Calculate IoU between predicted and target masks.
        
        Args:
            pred_masks: Predicted mask probabilities (N, H, W)
            target_masks: Target binary masks (N, H, W)
        
        Returns:
            IoU values (N,)
        """
        # Binarize predictions
        pred_binary = (pred_masks > self.threshold).float()
        
        # Flatten spatial dimensions
        pred_flat = pred_binary.flatten(1)
        target_flat = target_masks.flatten(1)
        
        # Calculate intersection and union
        intersection = (pred_flat * target_flat).sum(1)
        union = pred_flat.sum(1) + target_flat.sum(1) - intersection
        
        # IoU
        iou = intersection / (union + 1e-7)
        
        return iou


if __name__ == "__main__":
    # Test segmentation losses
    print("Testing Segmentation Losses...")
    
    pred_masks = torch.randn(4, 160, 160)
    target_masks = (torch.rand(4, 160, 160) > 0.5).float()
    
    # Dice Loss
    dice_loss = DiceLoss()
    loss = dice_loss(torch.sigmoid(pred_masks), target_masks)
    print(f"Dice Loss: {loss.item():.4f}")
    
    # Combined Segmentation Loss
    seg_loss = SegmentationLoss()
    total_loss, loss_dict = seg_loss(pred_masks, target_masks)
    print(f"Segmentation Loss: {total_loss.item():.4f}")
    print(f"  BCE: {loss_dict['bce'].item():.4f}")
    print(f"  Dice: {loss_dict['dice'].item():.4f}")
    
    # Mask IoU
    mask_iou = MaskIoU()
    iou = mask_iou(torch.sigmoid(pred_masks), target_masks)
    print(f"Mask IoU: {iou.mean().item():.4f}")


In [ ]:
%%writefile yolov11/losses/pose_loss.py
"""
Pose Estimation Loss for YOLOv11
Keypoint loss using OKS (Object Keypoint Similarity)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple, Optional


# COCO keypoint sigmas for OKS calculation
# Smaller sigma = stricter matching required
COCO_KEYPOINT_SIGMAS = torch.tensor([
    0.026,  # nose
    0.025,  # left_eye
    0.025,  # right_eye
    0.035,  # left_ear
    0.035,  # right_ear
    0.079,  # left_shoulder
    0.079,  # right_shoulder
    0.072,  # left_elbow
    0.072,  # right_elbow
    0.062,  # left_wrist
    0.062,  # right_wrist
    0.107,  # left_hip
    0.107,  # right_hip
    0.087,  # left_knee
    0.087,  # right_knee
    0.089,  # left_ankle
    0.089,  # right_ankle
])


class OKSLoss(nn.Module):
    """
    Object Keypoint Similarity (OKS) Loss for pose estimation.
    
    OKS measures similarity between predicted and ground truth keypoints,
    accounting for object scale and keypoint-specific uncertainty.
    
    OKS = exp(-d^2 / (2 * s^2 * k^2))
    
    where:
    - d: Euclidean distance between predicted and GT keypoint
    - s: Object scale (sqrt of bounding box area)
    - k: Keypoint-specific sigma (from COCO)
    
    Args:
        sigmas: Per-keypoint sigma values
        use_visibility: Whether to use visibility flags
    """
    
    def __init__(
        self,
        sigmas: torch.Tensor = None,
        use_visibility: bool = True,
        reduction: str = 'mean'
    ):
        super().__init__()
        
        if sigmas is None:
            sigmas = COCO_KEYPOINT_SIGMAS
        
        self.register_buffer('sigmas', sigmas)
        self.use_visibility = use_visibility
        self.reduction = reduction
    
    def forward(
        self,
        pred_kpts: torch.Tensor,
        target_kpts: torch.Tensor,
        target_vis: torch.Tensor,
        area: torch.Tensor
    ) -> torch.Tensor:
        """
        Calculate OKS loss.
        
        Args:
            pred_kpts: Predicted keypoints (N, 17, 2) - x, y coordinates
            target_kpts: Target keypoints (N, 17, 2)
            target_vis: Target visibility flags (N, 17)
            area: Object areas for scale (N,)
        
        Returns:
            OKS loss value
        """
        # Squared distance between predicted and target
        d2 = ((pred_kpts - target_kpts) ** 2).sum(dim=-1)  # (N, 17)
        
        # Scale factor
        s2 = area.unsqueeze(-1)  # (N, 1)
        
        # Keypoint-specific variance
        k2 = (self.sigmas ** 2).unsqueeze(0)  # (1, 17)
        
        # OKS for each keypoint
        oks = torch.exp(-d2 / (2 * s2 * k2 + 1e-9))  # (N, 17)
        
        # Apply visibility mask
        if self.use_visibility:
            valid_mask = target_vis > 0
            oks = oks * valid_mask.float()
            num_valid = valid_mask.sum(dim=-1).clamp(min=1)
            oks = oks.sum(dim=-1) / num_valid  # (N,)
        else:
            oks = oks.mean(dim=-1)  # (N,)
        
        # Loss = 1 - OKS
        loss = 1.0 - oks
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


class KeypointL1Loss(nn.Module):
    """
    L1 Loss for keypoint regression.
    
    Simple but effective loss for keypoint coordinate regression.
    Typically combined with OKS for better results.
    """
    
    def __init__(self, reduction: str = 'mean'):
        super().__init__()
        self.reduction = reduction
    
    def forward(
        self,
        pred_kpts: torch.Tensor,
        target_kpts: torch.Tensor,
        target_vis: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Calculate L1 loss for keypoints.
        
        Args:
            pred_kpts: Predicted keypoints (N, 17, 2)
            target_kpts: Target keypoints (N, 17, 2)
            target_vis: Optional visibility flags (N, 17)
        
        Returns:
            L1 loss value
        """
        loss = F.l1_loss(pred_kpts, target_kpts, reduction='none')
        loss = loss.sum(dim=-1)  # Sum x, y components (N, 17)
        
        if target_vis is not None:
            valid_mask = target_vis > 0
            loss = loss * valid_mask.float()
            loss = loss.sum() / valid_mask.sum().clamp(min=1)
        else:
            loss = loss.mean()
        
        return loss


class VisibilityLoss(nn.Module):
    """
    Binary Cross Entropy Loss for keypoint visibility prediction.
    """
    
    def __init__(self, reduction: str = 'mean'):
        super().__init__()
        self.reduction = reduction
        self.bce = nn.BCEWithLogitsLoss(reduction=reduction)
    
    def forward(
        self,
        pred_vis: torch.Tensor,
        target_vis: torch.Tensor
    ) -> torch.Tensor:
        """
        Calculate visibility loss.
        
        Args:
            pred_vis: Predicted visibility logits (N, 17)
            target_vis: Target visibility flags (N, 17) - 0, 1, or 2
                       0: not labeled, 1: labeled but not visible, 2: visible
        
        Returns:
            Visibility loss value
        """
        # Convert to binary: visible (2) vs not visible (0, 1)
        target_binary = (target_vis == 2).float()
        
        return self.bce(pred_vis, target_binary)


class PoseLoss(nn.Module):
    """
    Combined Pose Estimation Loss for YOLOv11.
    
    Combines:
    - OKS Loss for keypoint similarity
    - L1 Loss for coordinate regression
    - BCE Loss for visibility prediction
    
    Args:
        oks_weight: Weight for OKS loss
        l1_weight: Weight for L1 loss
        vis_weight: Weight for visibility loss
        sigmas: Per-keypoint sigma values for OKS
    """
    
    def __init__(
        self,
        oks_weight: float = 1.0,
        l1_weight: float = 0.5,
        vis_weight: float = 1.0,
        sigmas: torch.Tensor = None
    ):
        super().__init__()
        
        self.oks_weight = oks_weight
        self.l1_weight = l1_weight
        self.vis_weight = vis_weight
        
        self.oks_loss = OKSLoss(sigmas=sigmas)
        self.l1_loss = KeypointL1Loss()
        self.vis_loss = VisibilityLoss()
    
    def forward(
        self,
        pred_kpts: torch.Tensor,
        pred_vis: torch.Tensor,
        target_kpts: torch.Tensor,
        target_vis: torch.Tensor,
        area: torch.Tensor
    ) -> Tuple[torch.Tensor, dict]:
        """
        Calculate combined pose loss.
        
        Args:
            pred_kpts: Predicted keypoints (N, 17, 2)
            pred_vis: Predicted visibility logits (N, 17)
            target_kpts: Target keypoints (N, 17, 2)
            target_vis: Target visibility flags (N, 17)
            area: Object areas (N,)
        
        Returns:
            Tuple of (total_loss, dict with individual losses)
        """
        # OKS loss
        oks = self.oks_loss(pred_kpts, target_kpts, target_vis, area)
        
        # L1 loss
        l1 = self.l1_loss(pred_kpts, target_kpts, target_vis)
        
        # Visibility loss
        vis = self.vis_loss(pred_vis, target_vis)
        
        # Total loss
        total = (self.oks_weight * oks + 
                 self.l1_weight * l1 + 
                 self.vis_weight * vis)
        
        return total, {
            'oks': oks,
            'l1': l1,
            'vis': vis
        }


class WingLoss(nn.Module):
    """
    Wing Loss for keypoint regression.
    
    Better than L1/L2 for small errors, commonly used in facial landmark detection.
    
    w(x) = ln(1 + |x|/ε) * w  if |x| < w
         = |x| - C            otherwise
    
    where C = w - w * ln(1 + w/ε)
    
    Args:
        width: Width parameter (w)
        epsilon: Curvature parameter (ε)
    """
    
    def __init__(
        self,
        width: float = 10.0,
        epsilon: float = 2.0,
        reduction: str = 'mean'
    ):
        super().__init__()
        self.width = width
        self.epsilon = epsilon
        self.reduction = reduction
        
        # Precompute constant
        self.C = width - width * torch.log(torch.tensor(1.0 + width / epsilon))
    
    def forward(
        self,
        pred: torch.Tensor,
        target: torch.Tensor,
        mask: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Calculate Wing loss.
        
        Args:
            pred: Predicted coordinates (N, K, 2)
            target: Target coordinates (N, K, 2)
            mask: Optional mask for valid keypoints (N, K)
        
        Returns:
            Wing loss value
        """
        diff = pred - target
        abs_diff = diff.abs()
        
        # Wing loss formula
        small_error = abs_diff < self.width
        loss = torch.where(
            small_error,
            self.width * torch.log(1.0 + abs_diff / self.epsilon),
            abs_diff - self.C
        )
        
        loss = loss.sum(dim=-1)  # Sum x, y components
        
        if mask is not None:
            loss = loss * mask
            loss = loss.sum() / mask.sum().clamp(min=1)
        elif self.reduction == 'mean':
            loss = loss.mean()
        elif self.reduction == 'sum':
            loss = loss.sum()
        
        return loss


if __name__ == "__main__":
    # Test pose losses
    print("Testing Pose Losses...")
    
    N = 8
    pred_kpts = torch.randn(N, 17, 2) * 100  # Random keypoints
    target_kpts = pred_kpts + torch.randn(N, 17, 2) * 5  # Add noise
    target_vis = torch.randint(0, 3, (N, 17))  # Random visibility
    area = torch.rand(N) * 10000 + 1000  # Random areas
    
    # OKS Loss
    oks_loss = OKSLoss()
    loss = oks_loss(pred_kpts, target_kpts, target_vis, area)
    print(f"OKS Loss: {loss.item():.4f}")
    
    # Keypoint L1 Loss
    l1_loss = KeypointL1Loss()
    loss = l1_loss(pred_kpts, target_kpts, target_vis)
    print(f"Keypoint L1 Loss: {loss.item():.4f}")
    
    # Combined Pose Loss
    pred_vis = torch.randn(N, 17)
    pose_loss = PoseLoss()
    total, loss_dict = pose_loss(pred_kpts, pred_vis, target_kpts, target_vis, area)
    print(f"Pose Loss: {total.item():.4f}")
    print(f"  OKS: {loss_dict['oks'].item():.4f}")
    print(f"  L1: {loss_dict['l1'].item():.4f}")
    print(f"  Vis: {loss_dict['vis'].item():.4f}")
    
    # Wing Loss
    wing_loss = WingLoss()
    loss = wing_loss(pred_kpts, target_kpts, (target_vis > 0).float())
    print(f"Wing Loss: {loss.item():.4f}")


In [ ]:
%%writefile yolov11/losses/enhanced_loss.py
"""
Enhanced Loss Functions for YOLOv11
Includes: Wise-IoU, Quality Focal Loss, Label Smoothing
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional


class WiseIoULoss(nn.Module):
    """
    Wise-IoU Loss with dynamic non-monotonic focusing mechanism.
    
    Handles outliers better than standard IoU losses by using
    a dynamic gradient weighting strategy.
    
    Reference: https://arxiv.org/abs/2301.10051
    """
    
    def __init__(self, eps: float = 1e-7, ratio: float = 0.8):
        super().__init__()
        self.eps = eps
        self.ratio = ratio
    
    def forward(
        self,
        pred: torch.Tensor,
        target: torch.Tensor,
        reduction: str = 'mean'
    ) -> torch.Tensor:
        """
        Args:
            pred: Predicted boxes (N, 4) in xyxy format
            target: Target boxes (N, 4) in xyxy format
        """
        # Calculate intersection
        inter_x1 = torch.max(pred[:, 0], target[:, 0])
        inter_y1 = torch.max(pred[:, 1], target[:, 1])
        inter_x2 = torch.min(pred[:, 2], target[:, 2])
        inter_y2 = torch.min(pred[:, 3], target[:, 3])
        
        inter_w = (inter_x2 - inter_x1).clamp(min=0)
        inter_h = (inter_y2 - inter_y1).clamp(min=0)
        inter_area = inter_w * inter_h
        
        # Calculate union
        pred_area = (pred[:, 2] - pred[:, 0]) * (pred[:, 3] - pred[:, 1])
        target_area = (target[:, 2] - target[:, 0]) * (target[:, 3] - target[:, 1])
        union = pred_area + target_area - inter_area + self.eps
        
        # IoU
        iou = inter_area / union
        
        # Calculate center distance
        pred_cx = (pred[:, 0] + pred[:, 2]) / 2
        pred_cy = (pred[:, 1] + pred[:, 3]) / 2
        target_cx = (target[:, 0] + target[:, 2]) / 2
        target_cy = (target[:, 1] + target[:, 3]) / 2
        
        center_dist = (pred_cx - target_cx) ** 2 + (pred_cy - target_cy) ** 2
        
        # Enclosing box diagonal
        enclose_x1 = torch.min(pred[:, 0], target[:, 0])
        enclose_y1 = torch.min(pred[:, 1], target[:, 1])
        enclose_x2 = torch.max(pred[:, 2], target[:, 2])
        enclose_y2 = torch.max(pred[:, 3], target[:, 3])
        
        c2 = (enclose_x2 - enclose_x1) ** 2 + (enclose_y2 - enclose_y1) ** 2 + self.eps
        
        # Wise-IoU focusing coefficient
        # Uses ratio of current IoU to expected IoU
        wise_scale = torch.exp((center_dist / c2) * self.ratio)
        
        # Wise-IoU loss
        loss = (1 - iou) * wise_scale
        
        if reduction == 'mean':
            return loss.mean()
        elif reduction == 'sum':
            return loss.sum()
        return loss


class QualityFocalLoss(nn.Module):
    """
    Quality Focal Loss for dense object detection.
    
    Combines classification and localization quality into a single loss,
    enabling joint optimization.
    
    Reference: https://arxiv.org/abs/2006.04388
    """
    
    def __init__(self, beta: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.beta = beta
        self.reduction = reduction
    
    def forward(
        self,
        pred: torch.Tensor,
        target: torch.Tensor,
        quality: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Args:
            pred: Predicted class logits (N, C)
            target: Target class indices (N,)
            quality: IoU quality scores (N,), if None uses 1.0
        """
        # Convert to one-hot
        num_classes = pred.shape[-1]
        target_one_hot = F.one_hot(target.long(), num_classes).float()
        
        # Apply quality score
        if quality is not None:
            quality = quality.unsqueeze(-1)
            target_one_hot = target_one_hot * quality
        
        # Sigmoid activation
        pred_sigmoid = torch.sigmoid(pred)
        
        # Calculate focal weight
        pt = pred_sigmoid * target_one_hot + (1 - pred_sigmoid) * (1 - target_one_hot)
        focal_weight = (target_one_hot - pred_sigmoid).abs().pow(self.beta)
        
        # BCE loss with focal weight
        bce = F.binary_cross_entropy_with_logits(pred, target_one_hot, reduction='none')
        loss = focal_weight * bce
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


class VarifocalLoss(nn.Module):
    """
    Varifocal Loss - improved version of Quality Focal Loss.
    
    Uses IoU-aware classification score (IACS) as target.
    """
    
    def __init__(self, alpha: float = 0.75, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(
        self,
        pred: torch.Tensor,
        target: torch.Tensor,
        target_score: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Args:
            pred: Predicted class logits (N, C)
            target: Target class indices (N,)
            target_score: IoU quality scores (N,), if None uses 1.0
        """
        pred_sigmoid = torch.sigmoid(pred)
        
        # Create soft labels with IoU score
        num_classes = pred.shape[-1]
        target_one_hot = F.one_hot(target.long().clamp(0, num_classes-1), num_classes).float()
        
        if target_score is not None:
            target_one_hot = target_one_hot * target_score.unsqueeze(-1)
        
        # Varifocal weighting
        focal_weight = target_one_hot * (target_one_hot > 0).float() + \
                       self.alpha * pred_sigmoid.pow(self.gamma) * (target_one_hot == 0).float()
        
        # BCE loss
        bce = F.binary_cross_entropy_with_logits(pred, target_one_hot, reduction='none')
        loss = focal_weight * bce
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


class LabelSmoothingCE(nn.Module):
    """
    Cross-Entropy Loss with Label Smoothing.
    
    Prevents overconfident predictions and improves generalization.
    """
    
    def __init__(self, num_classes: int, smoothing: float = 0.1, reduction: str = 'mean'):
        super().__init__()
        self.num_classes = num_classes
        self.smoothing = smoothing
        self.reduction = reduction
        self.confidence = 1.0 - smoothing
    
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """
        Args:
            pred: Predicted logits (N, C)  
            target: Target class indices (N,)
        """
        log_probs = F.log_softmax(pred, dim=-1)
        
        # Smooth labels
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (self.num_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1).long(), self.confidence)
        
        loss = -true_dist * log_probs
        loss = loss.sum(dim=-1)
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


if __name__ == "__main__":
    # Test losses
    print("Testing enhanced losses...")
    
    # Wise-IoU
    wise_iou = WiseIoULoss()
    pred_boxes = torch.tensor([[10, 10, 50, 50], [20, 20, 60, 60]], dtype=torch.float32)
    target_boxes = torch.tensor([[12, 12, 48, 48], [22, 22, 58, 58]], dtype=torch.float32)
    loss = wise_iou(pred_boxes, target_boxes)
    print(f"Wise-IoU Loss: {loss.item():.4f}")
    
    # Quality Focal Loss
    qfl = QualityFocalLoss()
    pred_cls = torch.randn(4, 80)
    target_cls = torch.randint(0, 80, (4,))
    quality = torch.rand(4)
    loss = qfl(pred_cls, target_cls, quality)
    print(f"Quality Focal Loss: {loss.item():.4f}")
    
    # Label Smoothing
    lsce = LabelSmoothingCE(num_classes=80)
    loss = lsce(pred_cls, target_cls)
    print(f"Label Smoothing CE: {loss.item():.4f}")
    
    print("All losses OK!")


In [ ]:
%%writefile yolov11/losses/combined_loss.py
"""
Combined YOLOv11 Loss Function
Unifies all task-specific losses with Task-Aligned Assigner
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, List, Tuple, Optional

from .box_loss import CIoULoss, DFLoss, bbox_iou
from .cls_loss import ClassificationLoss, VarifocalLoss
from .seg_loss import SegmentationLoss
from .pose_loss import PoseLoss


class TaskAlignedAssigner(nn.Module):
    """
    Task-Aligned Assigner for YOLOv11.
    
    Assigns ground truth targets to anchor points based on both
    classification and localization quality.
    
    alignment_metric = cls_score^alpha * iou^beta
    
    Args:
        topk: Maximum number of anchors to assign per GT
        alpha: Classification score weight
        beta: IoU weight
    """
    
    def __init__(
        self,
        topk: int = 10,
        num_classes: int = 80,
        alpha: float = 0.5,
        beta: float = 6.0,
        eps: float = 1e-9
    ):
        super().__init__()
        self.topk = topk
        self.num_classes = num_classes
        self.alpha = alpha
        self.beta = beta
        self.eps = eps
    
    @torch.no_grad()
    def forward(
        self,
        pred_scores: torch.Tensor,
        pred_bboxes: torch.Tensor,
        anchor_points: torch.Tensor,
        gt_labels: torch.Tensor,
        gt_bboxes: torch.Tensor,
        mask_gt: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Assign targets to predictions.
        
        Args:
            pred_scores: Predicted class scores (B, N, num_classes)
            pred_bboxes: Predicted boxes (B, N, 4)
            anchor_points: Anchor points (N, 2)
            gt_labels: Ground truth labels (B, M)
            gt_bboxes: Ground truth boxes (B, M, 4)
            mask_gt: Valid GT mask (B, M)
        
        Returns:
            Tuple of (target_labels, target_bboxes, target_scores, fg_mask)
        """
        batch_size = pred_scores.shape[0]
        num_anchors = anchor_points.shape[0]
        num_max_boxes = gt_bboxes.shape[1]
        
        if num_max_boxes == 0:
            device = gt_bboxes.device
            return (
                torch.zeros(batch_size, num_anchors, dtype=torch.long, device=device),
                torch.zeros(batch_size, num_anchors, 4, device=device),
                torch.zeros(batch_size, num_anchors, self.num_classes, device=device),
                torch.zeros(batch_size, num_anchors, dtype=torch.bool, device=device)
            )
        
        # Get positive mask based on anchor points inside GT boxes
        mask_pos, align_metric, overlaps = self._get_pos_mask(
            pred_scores, pred_bboxes, gt_labels, gt_bboxes, anchor_points, mask_gt
        )
        
        # Select top-k anchors for each GT
        target_gt_idx, fg_mask, mask_pos = self._select_topk(
            mask_pos, align_metric, overlaps, num_max_boxes
        )
        
        # Get targets
        target_labels, target_bboxes, target_scores = self._get_targets(
            gt_labels, gt_bboxes, target_gt_idx, fg_mask
        )
        
        # Normalize target scores by max alignment metric
        align_metric *= mask_pos
        pos_align_metrics = align_metric.amax(dim=-1, keepdim=True)
        pos_overlaps = (overlaps * mask_pos).amax(dim=-1, keepdim=True)
        norm_align_metric = (align_metric * pos_overlaps / (pos_align_metrics + self.eps)).amax(-2)
        target_scores = target_scores * norm_align_metric.unsqueeze(-1)
        
        return target_labels, target_bboxes, target_scores, fg_mask
    
    def _get_pos_mask(
        self,
        pred_scores: torch.Tensor,
        pred_bboxes: torch.Tensor,
        gt_labels: torch.Tensor,
        gt_bboxes: torch.Tensor,
        anchor_points: torch.Tensor,
        mask_gt: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Get positive mask and alignment metrics."""
        batch_size = pred_scores.shape[0]
        num_anchors = anchor_points.shape[0]
        num_gt = gt_bboxes.shape[1]
        
        # Check if anchor center is inside GT box
        mask_in_gts = self._is_anchor_in_gt(anchor_points, gt_bboxes)
        
        # Calculate alignment metric
        # Get predicted class scores for GT classes
        batch_idx = torch.arange(batch_size, device=pred_scores.device)[:, None, None]
        gt_idx = torch.arange(num_gt, device=pred_scores.device)[None, :, None]
        
        # pred_scores: (B, N, C), gt_labels: (B, M) -> (B, M, N)
        cls_scores = pred_scores.permute(0, 2, 1)  # (B, C, N)
        gt_labels_expanded = gt_labels.unsqueeze(-1).expand(-1, -1, num_anchors)
        pred_cls_scores = cls_scores.gather(1, gt_labels_expanded.clamp(0, self.num_classes - 1))  # (B, M, N)
        # pred_cls_scores = pred_cls_scores.sigmoid() # REMOVED: redundant as input already has sigmoid
        
        # Calculate IoU between predictions and GTs
        # pred_bboxes: (B, N, 4), gt_bboxes: (B, M, 4)
        overlaps = self._batch_bbox_iou(pred_bboxes, gt_bboxes)  # (B, M, N)
        
        # Alignment metric
        align_metric = pred_cls_scores.pow(self.alpha) * overlaps.pow(self.beta)
        
        # Combine masks
        mask_pos = mask_gt.unsqueeze(-1) * mask_in_gts  # (B, M, N)
        
        return mask_pos, align_metric, overlaps
    
    def _is_anchor_in_gt(
        self,
        anchor_points: torch.Tensor,
        gt_bboxes: torch.Tensor
    ) -> torch.Tensor:
        """Check if anchor centers are inside GT boxes."""
        # anchor_points: (N, 2), gt_bboxes: (B, M, 4)
        x, y = anchor_points.T  # (N,), (N,)
        x1, y1, x2, y2 = gt_bboxes.permute(2, 0, 1)  # (B, M) each
        
        # Compare: anchor (N,) vs GT (B, M)
        x1 = x1.unsqueeze(-1)  # (B, M, 1)
        y1 = y1.unsqueeze(-1)
        x2 = x2.unsqueeze(-1)
        y2 = y2.unsqueeze(-1)
        
        lt = (x > x1) & (y > y1)  # (B, M, N)
        rb = (x < x2) & (y < y2)
        
        return lt & rb
    
    def _batch_bbox_iou(
        self,
        pred_bboxes: torch.Tensor,
        gt_bboxes: torch.Tensor
    ) -> torch.Tensor:
        """Calculate IoU between all pred and GT boxes."""
        # pred_bboxes: (B, N, 4), gt_bboxes: (B, M, 4)
        pred = pred_bboxes.unsqueeze(1)  # (B, 1, N, 4)
        gt = gt_bboxes.unsqueeze(2)  # (B, M, 1, 4)
        
        # Intersection
        inter_x1 = torch.max(pred[..., 0], gt[..., 0])
        inter_y1 = torch.max(pred[..., 1], gt[..., 1])
        inter_x2 = torch.min(pred[..., 2], gt[..., 2])
        inter_y2 = torch.min(pred[..., 3], gt[..., 3])
        
        inter_area = (inter_x2 - inter_x1).clamp(0) * (inter_y2 - inter_y1).clamp(0)
        
        # Union
        pred_area = (pred[..., 2] - pred[..., 0]) * (pred[..., 3] - pred[..., 1])
        gt_area = (gt[..., 2] - gt[..., 0]) * (gt[..., 3] - gt[..., 1])
        union_area = pred_area + gt_area - inter_area
        
        return inter_area / (union_area + self.eps)  # (B, M, N)
    
    def _select_topk(
        self,
        mask_pos: torch.Tensor,
        align_metric: torch.Tensor,
        overlaps: torch.Tensor,
        num_gt: int
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Select top-k anchors for each GT (vectorized implementation)."""
        batch_size, _, num_anchors = mask_pos.shape
        
        # Top-k alignment metric per GT
        topk_metric, topk_idx = torch.topk(
            align_metric * mask_pos.float(),
            k=min(self.topk, num_anchors),
            dim=-1,
            largest=True
        )
        
        # Create top-k mask using vectorized scatter_ operation
        topk_mask = torch.zeros_like(mask_pos, dtype=torch.bool)
        # Expand valid GT mask to match topk_idx shape for scattering
        valid_gt_mask = mask_pos.any(dim=-1, keepdim=True).expand_as(topk_idx)
        # Use scatter to set True at topk indices where GT is valid
        topk_mask.scatter_(-1, topk_idx, valid_gt_mask)
        
        mask_pos = mask_pos & topk_mask
        
        # Resolve conflicts: assign each anchor to only one GT
        fg_mask = mask_pos.any(dim=1)  # (B, N)
        
        # Get target GT index for each anchor
        overlaps_masked = overlaps * mask_pos.float()
        target_gt_idx = overlaps_masked.argmax(dim=1)  # (B, N)
        
        return target_gt_idx, fg_mask, mask_pos
    
    def _get_targets(
        self,
        gt_labels: torch.Tensor,
        gt_bboxes: torch.Tensor,
        target_gt_idx: torch.Tensor,
        fg_mask: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Get target labels, boxes, and scores."""
        batch_size, num_anchors = target_gt_idx.shape
        
        # Get target labels
        batch_idx = torch.arange(batch_size, device=gt_labels.device)[:, None]
        target_labels = gt_labels[batch_idx, target_gt_idx]  # (B, N)
        target_labels = torch.where(fg_mask, target_labels, torch.zeros_like(target_labels))
        
        # Get target boxes
        target_bboxes = gt_bboxes[batch_idx, target_gt_idx]  # (B, N, 4)
        
        # Create one-hot target scores
        target_scores = F.one_hot(
            target_labels.long(),
            num_classes=self.num_classes
        ).float()  # (B, N, num_classes)
        target_scores = target_scores * fg_mask.unsqueeze(-1).float()
        
        return target_labels, target_bboxes, target_scores


class YOLOv11Loss(nn.Module):
    """
    Combined YOLOv11 Loss for all tasks.
    
    Supports:
    - Detection: box + classification loss
    - Segmentation: box + classification + mask loss
    - Pose: box + classification + keypoint loss
    
    Args:
        task: Task type ('detect', 'segment', 'pose')
        num_classes: Number of detection classes
        reg_max: Maximum regression value for DFL
        box_weight: Weight for box loss
        cls_weight: Weight for classification loss
        dfl_weight: Weight for DFL loss
    """
    
    def __init__(
        self,
        task: str = 'detect',
        num_classes: int = 80,
        reg_max: int = 16,
        box_weight: float = 7.5,
        cls_weight: float = 0.5,
        dfl_weight: float = 1.5,
        seg_weight: float = 3.0,
        pose_weight: float = 12.0,
        label_smoothing: float = 0.0
    ):
        super().__init__()
        
        self.task = task
        self.num_classes = num_classes
        self.reg_max = reg_max
        
        # Loss weights
        self.box_weight = box_weight
        self.cls_weight = cls_weight
        self.dfl_weight = dfl_weight
        self.seg_weight = seg_weight
        self.pose_weight = pose_weight
        self.label_smoothing = label_smoothing
        
        # Task-aligned assigner
        self.assigner = TaskAlignedAssigner(
            topk=10,
            num_classes=num_classes
        )
        
        # Loss functions
        self.bce_loss = nn.BCEWithLogitsLoss(reduction='none')
        self.ciou_loss = CIoULoss(reduction='none')
        self.dfl_loss = DFLoss(reg_max)
        
        if task == 'segment':
            self.seg_loss = SegmentationLoss()
        elif task == 'pose':
            self.pose_loss = PoseLoss()
    
    def forward(
        self,
        predictions: Dict[str, torch.Tensor],
        targets: Dict[str, torch.Tensor]
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        Calculate loss.
        
        Args:
            predictions: Model predictions
            targets: Ground truth targets
        
        Returns:
            Tuple of (total_loss, loss_dict)
        """
        # Extract predictions
        cls_preds = predictions['cls']  # List of (B, C, H, W)
        reg_preds = predictions['reg']  # List of (B, 4*reg_max, H, W)
        strides = predictions['strides']
        
        device = cls_preds[0].device
        batch_size = cls_preds[0].shape[0]
        
        # Flatten predictions across scales and get feature map sizes
        cls_flat, reg_flat, anchor_points, feat_sizes = self._flatten_predictions(
            cls_preds, reg_preds, strides, device
        )
        
        # Decode boxes for assignment using actual feature map sizes
        pred_bboxes = self._decode_boxes(reg_flat, anchor_points, strides, feat_sizes)
        
        # Get targets
        gt_labels = targets['labels']  # (B, M)
        gt_bboxes = targets['bboxes']  # (B, M, 4)
        mask_gt = targets.get('mask_gt', gt_labels >= 0)  # (B, M)
        
        # Assign targets
        target_labels, target_bboxes, target_scores, fg_mask = self.assigner(
            cls_flat.sigmoid(),
            pred_bboxes,
            anchor_points,
            gt_labels,
            gt_bboxes,
            mask_gt
        )
        
        target_scores_sum = target_scores.sum().clamp(min=1)
        
        # Apply label smoothing to target scores
        if self.label_smoothing > 0:
            target_scores = target_scores * (1 - self.label_smoothing) + self.label_smoothing / self.num_classes
        
        # Classification loss
        loss_cls = self.bce_loss(cls_flat, target_scores).sum() / target_scores_sum
        
        # Box losses (only for foreground)
        if fg_mask.any():
            loss_box = self.ciou_loss(
                pred_bboxes[fg_mask],
                target_bboxes[fg_mask]
            )
            loss_box = (loss_box * target_scores[fg_mask].sum(-1)).sum() / target_scores_sum
            
            # DFL loss
            # Get stride for each anchor
            num_per_level = [h * w for h, w in feat_sizes]
            stride_tensor = torch.cat([
                torch.full((n,), s, device=device, dtype=torch.float)
                for n, s in zip(num_per_level, strides)
            ]).view(1, -1, 1)
            
            target_ltrb = self._box2dist(anchor_points, target_bboxes, stride_tensor)
            loss_dfl = self._compute_dfl_loss(reg_flat[fg_mask], target_ltrb[fg_mask])
        else:
            loss_box = torch.tensor(0.0, device=device)
            loss_dfl = torch.tensor(0.0, device=device)
        
        # Total loss
        loss = (
            self.box_weight * loss_box +
            self.cls_weight * loss_cls +
            self.dfl_weight * loss_dfl
        )
        
        loss_dict = {
            'loss_box': loss_box.detach(),
            'loss_cls': loss_cls.detach(),
            'loss_dfl': loss_dfl.detach()
        }
        
        # Task-specific losses
        if self.task == 'segment' and 'masks' in targets:
            loss_seg = self._compute_seg_loss(predictions, targets, fg_mask)
            loss = loss + self.seg_weight * loss_seg
            loss_dict['loss_seg'] = loss_seg.detach()
        
        elif self.task == 'pose' and 'keypoints' in targets:
            loss_pose = self._compute_pose_loss(predictions, targets, fg_mask)
            loss = loss + self.pose_weight * loss_pose
            loss_dict['loss_pose'] = loss_pose.detach()
        
        loss_dict['loss'] = loss.detach()
        
        return loss, loss_dict
    
    def _flatten_predictions(
        self,
        cls_preds: List[torch.Tensor],
        reg_preds: List[torch.Tensor],
        strides: List[int],
        device: torch.device
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, List[Tuple[int, int]]]:
        """Flatten multi-scale predictions and return feature map sizes."""
        cls_flat = []
        reg_flat = []
        anchor_points = []
        feat_sizes = []  # Store actual feature map sizes
        
        for cls_pred, reg_pred, stride in zip(cls_preds, reg_preds, strides):
            b, c, h, w = cls_pred.shape
            feat_sizes.append((h, w))  # Store actual size
            
            # Flatten spatial dimensions
            cls_flat.append(cls_pred.view(b, c, -1).permute(0, 2, 1))  # (B, H*W, C)
            reg_flat.append(reg_pred.view(b, -1, h * w).permute(0, 2, 1))  # (B, H*W, 4*reg_max)
            
            # Generate anchor points
            ys, xs = torch.meshgrid(
                torch.arange(h, device=device),
                torch.arange(w, device=device),
                indexing='ij'
            )
            grid = torch.stack([xs, ys], dim=-1).float() + 0.5
            anchor_points.append(grid.view(-1, 2) * stride)
        
        cls_flat = torch.cat(cls_flat, dim=1)  # (B, N, C)
        reg_flat = torch.cat(reg_flat, dim=1)  # (B, N, 4*reg_max)
        anchor_points = torch.cat(anchor_points, dim=0)  # (N, 2)
        
        return cls_flat, reg_flat, anchor_points, feat_sizes
    
    def _decode_boxes(
        self,
        reg_preds: torch.Tensor,
        anchor_points: torch.Tensor,
        strides: List[int],
        feat_sizes: List[Tuple[int, int]]
    ) -> torch.Tensor:
        """Decode box predictions using actual feature map sizes."""
        b = reg_preds.shape[0]
        
        # Apply softmax and integrate
        reg_preds = reg_preds.view(b, -1, 4, self.reg_max)
        reg_preds = F.softmax(reg_preds, dim=-1)
        
        # Weighted sum
        weights = torch.arange(self.reg_max, device=reg_preds.device, dtype=torch.float)
        reg_preds = (reg_preds * weights).sum(dim=-1)  # (B, N, 4)
        
        # ltrb to xyxy
        lt = reg_preds[..., :2]
        rb = reg_preds[..., 2:]
        
        # Get stride for each anchor using actual feature map sizes
        num_per_level = [h * w for h, w in feat_sizes]
        stride_tensor = torch.cat([
            torch.full((n,), s, device=reg_preds.device, dtype=torch.float)
            for n, s in zip(num_per_level, strides)
        ])
        stride_tensor = stride_tensor.view(1, -1, 1)
        
        x1y1 = anchor_points - lt * stride_tensor
        x2y2 = anchor_points + rb * stride_tensor
        
        return torch.cat([x1y1, x2y2], dim=-1)
    
    def _box2dist(
        self,
        anchor_points: torch.Tensor,
        bboxes: torch.Tensor,
        stride: torch.Tensor
    ) -> torch.Tensor:
        """Convert boxes to distance format normalized by stride."""
        x1y1, x2y2 = bboxes[..., :2], bboxes[..., 2:]
        lt = (anchor_points - x1y1) / stride
        rb = (x2y2 - anchor_points) / stride
        return torch.cat([lt, rb], dim=-1).clamp(0, self.reg_max - 0.01)
    
    def _compute_dfl_loss(
        self,
        pred_dist: torch.Tensor,
        target_dist: torch.Tensor
    ) -> torch.Tensor:
        """Compute DFL loss."""
        pred_dist = pred_dist.view(-1, self.reg_max)
        target_dist = target_dist.view(-1)
        
        target_dist = target_dist.clamp(0, self.reg_max - 0.01)
        tl = target_dist.long()
        tr = tl + 1
        wl = tr.float() - target_dist
        wr = 1 - wl
        
        loss = (
            F.cross_entropy(pred_dist, tl, reduction='none') * wl +
            F.cross_entropy(pred_dist, tr.clamp(max=self.reg_max - 1), reduction='none') * wr
        )
        
        return loss.mean()
    
    def _compute_seg_loss(
        self,
        predictions: Dict,
        targets: Dict,
        fg_mask: torch.Tensor
    ) -> torch.Tensor:
        """
        Compute instance segmentation loss.
        
        Args:
            predictions: Model outputs containing 'masks' (coefficients) and 'protos'
            targets: Ground truth containing 'masks' (N, H, W) binary masks
            fg_mask: Foreground mask (B, num_anchors)
            
        Returns:
            Segmentation loss tensor
        """
        if not fg_mask.any():
            return torch.tensor(0.0, device=fg_mask.device)
        
        # Get mask predictions
        mask_coeffs = predictions.get('masks')  # List of (B, num_protos, H, W) per scale
        protos = predictions.get('protos')  # (B, num_protos, proto_H, proto_W)
        
        if mask_coeffs is None or protos is None:
            return torch.tensor(0.0, device=fg_mask.device)
        
        # Flatten mask coefficients across scales
        mask_flat = []
        for mask in mask_coeffs:
            b, c, h, w = mask.shape
            mask_flat.append(mask.view(b, c, -1).permute(0, 2, 1))  # (B, H*W, num_protos)
        mask_flat = torch.cat(mask_flat, dim=1)  # (B, N, num_protos)
        
        # Get target masks
        target_masks = targets['masks']  # (B, M, mask_H, mask_W)
        target_bboxes = targets['bboxes']  # (B, M, 4)
        
        batch_size = fg_mask.shape[0]
        device = fg_mask.device
        total_loss = torch.tensor(0.0, device=device)
        num_fg = 0
        
        for b in range(batch_size):
            fg_idx = fg_mask[b].nonzero(as_tuple=True)[0]
            if len(fg_idx) == 0:
                continue
            
            # Get coefficients for foreground predictions
            coeffs = mask_flat[b, fg_idx]  # (num_fg, num_protos)
            
            # Get prototypes for this batch
            proto = protos[b]  # (num_protos, pH, pW)
            proto_h, proto_w = proto.shape[1:]
            
            # Assemble predicted masks: coeffs @ proto -> (num_fg, pH, pW)
            proto_flat = proto.view(proto.shape[0], -1)  # (num_protos, pH*pW)
            pred_masks = torch.mm(coeffs, proto_flat)  # (num_fg, pH*pW)
            pred_masks = pred_masks.view(-1, proto_h, proto_w)  # (num_fg, pH, pW)
            
            # Get target masks for this batch
            n_targets = min(len(fg_idx), target_masks.shape[1])
            if n_targets == 0:
                continue
                
            gt_masks = target_masks[b, :n_targets]  # (n_targets, mask_H, mask_W)
            
            # Resize target masks to proto size
            gt_masks = F.interpolate(
                gt_masks.unsqueeze(1).float(),
                size=(proto_h, proto_w),
                mode='bilinear',
                align_corners=False
            ).squeeze(1)  # (n_targets, pH, pW)
            
            # Match predictions to targets (simplified - use first n_targets)
            pred_masks_matched = pred_masks[:n_targets]
            
            # Compute BCE loss
            bce_loss = F.binary_cross_entropy_with_logits(
                pred_masks_matched,
                gt_masks,
                reduction='none'
            )
            
            # Compute Dice loss
            pred_sigmoid = pred_masks_matched.sigmoid()
            pred_flat = pred_sigmoid.flatten(1)
            gt_flat = gt_masks.flatten(1)
            
            intersection = (pred_flat * gt_flat).sum(1)
            union = pred_flat.sum(1) + gt_flat.sum(1)
            dice_loss = 1.0 - (2.0 * intersection + 1.0) / (union + 1.0)
            
            # Combine losses
            batch_loss = bce_loss.mean() + dice_loss.mean()
            total_loss = total_loss + batch_loss
            num_fg += n_targets
        
        if num_fg > 0:
            return total_loss / batch_size
        return torch.tensor(0.0, device=device)
    
    def _compute_pose_loss(
        self,
        predictions: Dict,
        targets: Dict,
        fg_mask: torch.Tensor
    ) -> torch.Tensor:
        """
        Compute pose estimation loss using OKS.
        
        Args:
            predictions: Model outputs containing 'kpts' (keypoint predictions)
            targets: Ground truth containing 'keypoints' (N, 17, 3) and 'areas'
            fg_mask: Foreground mask (B, num_anchors)
            
        Returns:
            Pose loss tensor
        """
        if not fg_mask.any():
            return torch.tensor(0.0, device=fg_mask.device)
        
        # Get keypoint predictions for foreground anchors
        kpt_preds = predictions.get('kpts')
        if kpt_preds is None:
            return torch.tensor(0.0, device=fg_mask.device)
        
        # Flatten keypoint predictions across scales
        kpt_flat = []
        for kpt in kpt_preds:
            b, c, h, w = kpt.shape
            # c = num_keypoints * 3 (x, y, visibility) = 51
            kpt_flat.append(kpt.view(b, c, -1).permute(0, 2, 1))  # (B, H*W, 51)
        kpt_flat = torch.cat(kpt_flat, dim=1)  # (B, N, 51)
        
        # Reshape to (B, N, 17, 3)
        num_keypoints = 17
        kpt_flat = kpt_flat.view(kpt_flat.shape[0], kpt_flat.shape[1], num_keypoints, 3)
        
        # Get target keypoints
        target_kpts = targets['keypoints']  # (B, M, 17, 3) - x, y, visibility
        target_areas = targets.get('areas', None)
        
        # For each foreground prediction, get the corresponding target keypoints
        # This is a simplified version - full implementation would use target assignment
        batch_size = fg_mask.shape[0]
        
        pred_kpts_fg = kpt_flat[fg_mask]  # (num_fg, 17, 3)
        
        if pred_kpts_fg.shape[0] == 0:
            return torch.tensor(0.0, device=fg_mask.device)
        
        # Get coordinates and visibility
        pred_coords = pred_kpts_fg[..., :2]  # (num_fg, 17, 2)
        pred_vis = pred_kpts_fg[..., 2]      # (num_fg, 17)
        
        # Compute areas if not provided (use bbox area approximation)
        if target_areas is None:
            # Approximate area from keypoint spread
            target_bboxes = targets['bboxes']  # (B, M, 4)
            areas_flat = (target_bboxes[..., 2] - target_bboxes[..., 0]) * \
                        (target_bboxes[..., 3] - target_bboxes[..., 1])
            # Get areas for foreground (simplified - use mean)
            areas = areas_flat[fg_mask[:, :areas_flat.shape[1]] if fg_mask.shape[1] > areas_flat.shape[1] 
                              else fg_mask].clamp(min=1.0)
        else:
            areas = target_areas[fg_mask[:, :target_areas.shape[1]]].clamp(min=1.0)
        
        # Match predictions to targets (simplified - use first M targets repeated)
        # Full implementation would use Hungarian matching or assignment from detector
        num_fg = pred_kpts_fg.shape[0]
        target_kpts_flat = target_kpts.view(-1, 17, 3)  # (B*M, 17, 3)
        
        # Repeat targets to match number of foreground predictions
        if target_kpts_flat.shape[0] < num_fg:
            repeat_factor = (num_fg // target_kpts_flat.shape[0]) + 1
            target_kpts_flat = target_kpts_flat.repeat(repeat_factor, 1, 1)[:num_fg]
        else:
            target_kpts_flat = target_kpts_flat[:num_fg]
        
        target_coords = target_kpts_flat[..., :2]  # (num_fg, 17, 2)
        target_vis = target_kpts_flat[..., 2]      # (num_fg, 17)
        
        # Ensure areas match
        if areas.shape[0] != num_fg:
            areas = areas.mean().expand(num_fg)
        
        # Compute OKS Loss
        loss, _ = self.pose_loss(
            pred_coords, pred_vis,
            target_coords, target_vis,
            areas
        )
        
        return loss


if __name__ == "__main__":
    print("Testing YOLOv11 Loss...")
    
    # Create loss
    loss_fn = YOLOv11Loss(task='detect', num_classes=80)
    
    # Mock predictions
    predictions = {
        'cls': [torch.randn(2, 80, 80, 80), torch.randn(2, 80, 40, 40), torch.randn(2, 80, 20, 20)],
        'reg': [torch.randn(2, 64, 80, 80), torch.randn(2, 64, 40, 40), torch.randn(2, 64, 20, 20)],
        'strides': [8, 16, 32]
    }
    
    # Mock targets
    targets = {
        'labels': torch.randint(0, 80, (2, 10)),
        'bboxes': torch.rand(2, 10, 4) * 640,
    }
    # Convert to xyxy format
    targets['bboxes'][..., 2:] = targets['bboxes'][..., :2] + targets['bboxes'][..., 2:]
    
    loss, loss_dict = loss_fn(predictions, targets)
    print(f"Total Loss: {loss.item():.4f}")
    for k, v in loss_dict.items():
        print(f"  {k}: {v.item():.4f}")


In [ ]:
%%writefile yolov11/utils/__init__.py
"""
YOLOv11 Utilities Package
"""

from .nms import non_max_suppression, batched_nms
from .metrics import compute_iou, compute_ap, compute_oks
from .visualization import draw_boxes, draw_masks, draw_keypoints, COCO_COLORS
from .training import ModelEMA, WarmupScheduler, ProgressiveResizing, EarlyStopping
from .pruning import unstructured_pruning, structured_pruning, global_pruning, get_sparsity
from .export import export_onnx, quantize_dynamic, get_model_size

__all__ = [
    # NMS
    "non_max_suppression",
    "batched_nms",
    # Metrics
    "compute_iou",
    "compute_ap",
    "compute_oks",
    # Visualization
    "draw_boxes",
    "draw_masks", 
    "draw_keypoints",
    "COCO_COLORS",
    # Training
    "ModelEMA",
    "WarmupScheduler",
    "ProgressiveResizing",
    "EarlyStopping",
    # Pruning
    "unstructured_pruning",
    "structured_pruning",
    "global_pruning",
    "get_sparsity",
    # Export
    "export_onnx",
    "quantize_dynamic",
    "get_model_size",
]


In [ ]:
%%writefile yolov11/utils/nms.py
"""
Non-Maximum Suppression for YOLOv11
"""

import torch
import torchvision
from typing import List, Tuple, Optional


def box_iou(box1: torch.Tensor, box2: torch.Tensor) -> torch.Tensor:
    """Calculate IoU between two sets of boxes (xyxy format)."""
    area1 = (box1[:, 2] - box1[:, 0]) * (box1[:, 3] - box1[:, 1])
    area2 = (box2[:, 2] - box2[:, 0]) * (box2[:, 3] - box2[:, 1])
    
    inter_x1 = torch.max(box1[:, None, 0], box2[:, 0])
    inter_y1 = torch.max(box1[:, None, 1], box2[:, 1])
    inter_x2 = torch.min(box1[:, None, 2], box2[:, 2])
    inter_y2 = torch.min(box1[:, None, 3], box2[:, 3])
    
    inter = (inter_x2 - inter_x1).clamp(0) * (inter_y2 - inter_y1).clamp(0)
    union = area1[:, None] + area2 - inter
    
    return inter / (union + 1e-7)


def non_max_suppression(
    predictions: torch.Tensor,
    conf_thres: float = 0.25,
    iou_thres: float = 0.45,
    classes: Optional[List[int]] = None,
    max_det: int = 300,
    multi_label: bool = True
) -> List[torch.Tensor]:
    """
    Non-Maximum Suppression for object detection.
    
    Args:
        predictions: (batch, num_anchors, 4 + num_classes) or with masks/keypoints
        conf_thres: Confidence threshold
        iou_thres: IoU threshold for NMS
        classes: Filter by class indices
        max_det: Maximum detections per image
        multi_label: Allow multiple labels per box
    
    Returns:
        List of detections per image: (n, 6) [x1, y1, x2, y2, conf, cls]
    """
    batch_size = predictions.shape[0]
    num_classes = predictions.shape[2] - 4
    
    # Settings
    max_wh = 7680  # Max box width/height
    max_nms = 30000  # Max boxes for torchvision.ops.nms
    
    output = [torch.zeros((0, 6), device=predictions.device)] * batch_size
    
    for xi, x in enumerate(predictions):
        # Filter by confidence
        xc = x[:, 4:].amax(1) > conf_thres
        x = x[xc]
        
        if not x.shape[0]:
            continue
        
        # Compute confidence
        box = x[:, :4]
        cls_scores = x[:, 4:]
        
        if multi_label:
            i, j = (cls_scores > conf_thres).nonzero(as_tuple=False).T
            x = torch.cat([box[i], cls_scores[i, j, None], j[:, None].float()], 1)
        else:
            conf, j = cls_scores.max(1, keepdim=True)
            x = torch.cat([box, conf, j.float()], 1)
            x = x[conf.view(-1) > conf_thres]
        
        # Filter by class
        if classes is not None:
            x = x[(x[:, 5:6] == torch.tensor(classes, device=x.device)).any(1)]
        
        n = x.shape[0]
        if not n:
            continue
        
        # Sort by confidence
        x = x[x[:, 4].argsort(descending=True)[:max_nms]]
        
        # Batched NMS
        c = x[:, 5:6] * max_wh
        boxes, scores = x[:, :4] + c, x[:, 4]
        
        i = torchvision.ops.nms(boxes, scores, iou_thres)
        i = i[:max_det]
        
        output[xi] = x[i]
    
    return output


def batched_nms(
    boxes: torch.Tensor,
    scores: torch.Tensor,
    class_ids: torch.Tensor,
    iou_thres: float = 0.5
) -> torch.Tensor:
    """Batched NMS - applies NMS per class."""
    if boxes.numel() == 0:
        return torch.empty((0,), dtype=torch.int64, device=boxes.device)
    
    max_offset = boxes.max()
    offsets = class_ids.to(boxes) * (max_offset + 1)
    boxes_offset = boxes + offsets[:, None]
    
    return torchvision.ops.nms(boxes_offset, scores, iou_thres)


def nms_rotated(boxes: torch.Tensor, scores: torch.Tensor, iou_thres: float) -> torch.Tensor:
    """NMS for rotated boxes (OBB)."""
    # Simplified: convert to axis-aligned for now
    return torchvision.ops.nms(boxes[:, :4], scores, iou_thres)


In [ ]:
%%writefile yolov11/utils/metrics.py
"""
Evaluation Metrics for YOLOv11
"""

import torch
import numpy as np
from typing import List, Tuple, Optional


def compute_iou(box1: torch.Tensor, box2: torch.Tensor) -> torch.Tensor:
    """Compute IoU between two sets of boxes (xyxy format)."""
    area1 = (box1[:, 2] - box1[:, 0]) * (box1[:, 3] - box1[:, 1])
    area2 = (box2[:, 2] - box2[:, 0]) * (box2[:, 3] - box2[:, 1])
    
    inter_x1 = torch.max(box1[:, None, 0], box2[:, 0])
    inter_y1 = torch.max(box1[:, None, 1], box2[:, 1])
    inter_x2 = torch.min(box1[:, None, 2], box2[:, 2])
    inter_y2 = torch.min(box1[:, None, 3], box2[:, 3])
    
    inter = (inter_x2 - inter_x1).clamp(0) * (inter_y2 - inter_y1).clamp(0)
    union = area1[:, None] + area2 - inter
    
    return inter / (union + 1e-7)


def compute_ap(recall: np.ndarray, precision: np.ndarray) -> float:
    """Compute Average Precision using 101-point interpolation."""
    mrec = np.concatenate([[0.0], recall, [1.0]])
    mpre = np.concatenate([[1.0], precision, [0.0]])
    
    # Make precision monotonically decreasing
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    
    # 101-point interpolation
    x = np.linspace(0, 1, 101)
    ap = np.trapz(np.interp(x, mrec, mpre), x)
    
    return float(ap)


def compute_ap_per_class(
    tp: np.ndarray,
    conf: np.ndarray,
    pred_cls: np.ndarray,
    target_cls: np.ndarray,
    eps: float = 1e-16
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute AP for each class.
    
    Returns: (tp, fp, precision, recall, ap) per class
    """
    # Sort by confidence
    i = np.argsort(-conf)
    tp, conf, pred_cls = tp[i], conf[i], pred_cls[i]
    
    unique_classes = np.unique(target_cls)
    nc = len(unique_classes)
    
    ap = np.zeros(nc)
    precision = np.zeros(nc)
    recall = np.zeros(nc)
    
    for ci, c in enumerate(unique_classes):
        i = pred_cls == c
        n_gt = (target_cls == c).sum()
        n_pred = i.sum()
        
        if n_pred == 0 or n_gt == 0:
            continue
        
        # Cumulative sums
        fpc = (1 - tp[i]).cumsum(0)
        tpc = tp[i].cumsum(0)
        
        # Recall
        rec = tpc / (n_gt + eps)
        
        # Precision
        prec = tpc / (tpc + fpc + eps)
        
        # AP
        ap[ci] = compute_ap(rec, prec)
        precision[ci] = prec[-1]
        recall[ci] = rec[-1]
    
    return precision, recall, ap, unique_classes


def compute_oks(
    pred_kpts: np.ndarray,
    gt_kpts: np.ndarray,
    gt_area: float,
    sigmas: Optional[np.ndarray] = None
) -> float:
    """
    Compute Object Keypoint Similarity (OKS).
    
    Args:
        pred_kpts: Predicted keypoints (17, 3)
        gt_kpts: Ground truth keypoints (17, 3)
        gt_area: Object area
        sigmas: Per-keypoint sigmas
    """
    if sigmas is None:
        sigmas = np.array([
            0.026, 0.025, 0.025, 0.035, 0.035, 0.079, 0.079, 0.072, 0.072,
            0.062, 0.062, 0.107, 0.107, 0.087, 0.087, 0.089, 0.089
        ])
    
    # Get visibility
    vis = gt_kpts[:, 2] > 0
    
    if not vis.any():
        return 0.0
    
    # Distance
    d = (pred_kpts[:, 0] - gt_kpts[:, 0]) ** 2 + (pred_kpts[:, 1] - gt_kpts[:, 1]) ** 2
    
    # OKS
    k2 = 2 * sigmas ** 2
    oks = np.exp(-d / (2 * gt_area * k2 + 1e-9))
    
    return float(oks[vis].mean())


class ConfusionMatrix:
    """Confusion matrix for object detection."""
    
    def __init__(self, nc: int, conf: float = 0.25, iou_thres: float = 0.5):
        self.nc = nc
        self.conf = conf
        self.iou_thres = iou_thres
        self.matrix = np.zeros((nc + 1, nc + 1))
    
    def process_batch(self, detections: torch.Tensor, labels: torch.Tensor):
        """Process a batch of detections."""
        if detections is None or len(detections) == 0:
            for label in labels:
                self.matrix[self.nc, int(label[0])] += 1
            return
        
        detections = detections[detections[:, 4] > self.conf]
        gt_classes = labels[:, 0].int()
        detection_classes = detections[:, 5].int()
        
        iou = compute_iou(labels[:, 1:5], detections[:, :4])
        
        x = torch.where(iou > self.iou_thres)
        if x[0].shape[0]:
            matches = torch.cat((torch.stack(x, 1), iou[x[0], x[1]][:, None]), 1).cpu().numpy()
            matches = matches[matches[:, 2].argsort()[::-1]]
            matches = matches[np.unique(matches[:, 1], return_index=True)[1]]
            matches = matches[np.unique(matches[:, 0], return_index=True)[1]]
        else:
            matches = np.zeros((0, 3))
        
        n = matches.shape[0] > 0
        m0, m1, _ = matches.transpose().astype(int)
        
        for i, gc in enumerate(gt_classes):
            j = m0 == i
            if n and j.sum() == 1:
                self.matrix[detection_classes[m1[j]], gc] += 1
            else:
                self.matrix[self.nc, gc] += 1
        
        for i, dc in enumerate(detection_classes):
            if not any(m1 == i):
                self.matrix[dc, self.nc] += 1
    
    @property
    def tp(self) -> np.ndarray:
        return self.matrix.diagonal()[:self.nc]
    
    @property
    def fp(self) -> np.ndarray:
        return self.matrix[:self.nc, self.nc]
    
    @property
    def fn(self) -> np.ndarray:
        return self.matrix[self.nc, :self.nc]


def match_predictions(
    pred_boxes: torch.Tensor,
    pred_cls: torch.Tensor,
    pred_conf: torch.Tensor,
    gt_boxes: torch.Tensor,
    gt_cls: torch.Tensor,
    iou_thresholds: torch.Tensor
) -> torch.Tensor:
    """
    Match predictions to ground truth across multiple IoU thresholds.
    
    Args:
        pred_boxes: (N, 4)
        pred_cls: (N,)
        pred_conf: (N,)
        gt_boxes: (M, 4)
        gt_cls: (M,)
        iou_thresholds: (T,) e.g., [0.5, 0.55, ..., 0.95]
        
    Returns:
        tp: (N, T) boolean matrix of true positives for each threshold
    """
    num_preds = pred_boxes.shape[0]
    num_thresholds = iou_thresholds.shape[0]
    tp = torch.zeros(num_preds, num_thresholds, dtype=torch.bool, device=pred_boxes.device)
    
    if gt_boxes.shape[0] == 0:
        return tp
        
    iou = compute_iou(pred_boxes, gt_boxes)
    
    # Correct classes mask
    correct_cls = (pred_cls[:, None] == gt_cls[None, :])
    
    for i, threshold in enumerate(iou_thresholds):
        # IoU > threshold AND correct class
        matches = (iou > threshold) & correct_cls
        
        if matches.any():
            # For each prediction, find best matching GT
            # To handle multiple predictions for same GT, we'd need more complex matching
            # but for validation metrics, "best first" is common.
            # Simplified matching:
            best_iou, best_gt_idx = iou.max(1)
            for p_idx in range(num_preds):
                g_idx = best_gt_idx[p_idx]
                if matches[p_idx, g_idx]:
                    tp[p_idx, i] = True
                    # Mask out this GT so it can't be matched again in this threshold
                    matches[:, g_idx] = False
                    
    return tp


class Metrics:
    """Detection metrics calculator for mAP50 and mAP50-95."""
    
    def __init__(self, iou_thresholds: Optional[torch.Tensor] = None):
        self.iou_thresholds = iou_thresholds if iou_thresholds is not None else \
                             torch.linspace(0.5, 0.95, 10)
        self.stats = [] # List of (tp, conf, pred_cls, target_cls)
    
    def process_batch(
        self,
        detections: torch.Tensor,
        gt_bboxes: torch.Tensor,
        gt_labels: torch.Tensor
    ):
        """
        Process a batch of detections.
        
        Args:
            detections: (N, 6) [x1, y1, x2, y2, conf, cls]
            gt_bboxes: (M, 4) [x1, y1, x2, y2]
            gt_labels: (M,) class indices
        """
        if detections.shape[0] == 0:
            if gt_labels.shape[0] > 0:
                self.stats.append((
                    torch.zeros(0, len(self.iou_thresholds), dtype=torch.bool),
                    torch.zeros(0),
                    torch.zeros(0),
                    gt_labels.cpu().numpy()
                ))
            return

        tp = match_predictions(
            detections[:, :4],
            detections[:, 5],
            detections[:, 4],
            gt_bboxes,
            gt_labels,
            self.iou_thresholds.to(detections.device)
        )
        
        self.stats.append((
            tp.cpu(),
            detections[:, 4].cpu(),
            detections[:, 5].cpu(),
            gt_labels.cpu().numpy()
        ))
    
    def compute(self) -> dict:
        """Compute final metrics."""
        if not self.stats:
            return {'mAP50': 0, 'mAP50-95': 0, 'precision': 0, 'recall': 0}
        
        tp = torch.cat([s[0] for s in self.stats], 0).numpy()
        conf = torch.cat([s[1] for s in self.stats], 0).numpy()
        pred_cls = torch.cat([s[2] for s in self.stats], 0).numpy()
        target_cls = np.concatenate([s[3] for s in self.stats])
        
        # Compute AP for each class and each threshold
        nc = len(np.unique(target_cls))
        ap = np.zeros((nc, tp.shape[1]))
        
        for i in range(tp.shape[1]):
            precision, recall, ap_thresh, _ = compute_ap_per_class(
                tp[:, i], conf, pred_cls, target_cls
            )
            ap[:, i] = ap_thresh
            
        return {
            'mAP50': ap[:, 0].mean() if ap.size > 0 else 0,
            'mAP50-95': ap.mean() if ap.size > 0 else 0,
            'precision': 0, # Could be added if needed
            'recall': 0
        }


In [ ]:
%%writefile yolov11/utils/training.py
"""
Training Utilities for YOLOv11
Includes: EMA, Progressive Resizing, Learning Rate Schedulers
"""

import torch
import torch.nn as nn
from typing import Optional, Dict, Any
import copy
import math


class ModelEMA:
    """
    Exponential Moving Average (EMA) of model weights.
    
    Maintains a shadow copy of model weights that is updated using
    exponential moving average, which typically leads to better
    generalization and more stable training.
    
    Usage:
        ema = ModelEMA(model)
        for epoch in range(epochs):
            train(model)
            ema.update(model)
        # Use ema.ema for evaluation
    """
    
    def __init__(
        self,
        model: nn.Module,
        decay: float = 0.9999,
        tau: float = 2000,
        updates: int = 0
    ):
        """
        Args:
            model: Model to track
            decay: EMA decay factor (higher = slower updates)
            tau: Time constant for decay warmup
            updates: Initial update count
        """
        self.ema: nn.Module = copy.deepcopy(model).eval()
        self.updates = updates
        self.decay = decay
        self.tau = tau
        
        # Disable gradients for EMA model
        for p in self.ema.parameters():
            p.requires_grad_(False)
    
    def update(self, model: nn.Module):
        """Update EMA weights."""
        with torch.no_grad():
            self.updates += 1
            
            # Decay with warmup
            d = self.decay * (1 - math.exp(-self.updates / self.tau))
            
            # Use unwrapped model state dict to avoid DDP prefixes
            model_to_update = model.module if hasattr(model, 'module') else model
            msd = model_to_update.state_dict()
            
            for k, v in self.ema.state_dict().items():
                if v.dtype.is_floating_point:
                    v *= d
                    v += (1 - d) * msd[k].detach()
    
    def update_attr(self, model: nn.Module, include=(), exclude=('process_group', 'reducer')):
        """Update EMA attributes."""
        for k, v in model.__dict__.items():
            if (len(include) > 0 and k not in include) or k.startswith('_') or k in exclude:
                continue
            setattr(self.ema, k, v)


class WarmupScheduler:
    """
    Learning rate warmup scheduler.
    
    Gradually increases learning rate from 0 to base_lr during warmup epochs.
    """
    
    def __init__(
        self,
        optimizer: torch.optim.Optimizer,
        warmup_epochs: int = 3,
        warmup_bias_lr: float = 0.1,
        warmup_momentum: float = 0.8
    ):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.warmup_bias_lr = warmup_bias_lr
        self.warmup_momentum = warmup_momentum
        
        # Store initial values
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
        self.base_momentums = [pg.get('momentum', 0.9) for pg in optimizer.param_groups]
    
    def step(self, epoch: float, batch_idx: int, num_batches: int):
        """Update learning rate based on warmup progress."""
        if epoch < self.warmup_epochs:
            # Linear warmup
            progress = (epoch * num_batches + batch_idx) / (self.warmup_epochs * num_batches)
            
            for i, pg in enumerate(self.optimizer.param_groups):
                pg['lr'] = self.base_lrs[i] * progress
                if 'momentum' in pg:
                    pg['momentum'] = self.warmup_momentum + (self.base_momentums[i] - self.warmup_momentum) * progress


class CosineAnnealingWarmRestarts:
    """
    Cosine annealing with warm restarts.
    
    Learning rate follows cosine curve with periodic restarts.
    """
    
    def __init__(
        self,
        optimizer: torch.optim.Optimizer,
        T_0: int = 10,
        T_mult: int = 2,
        eta_min: float = 1e-6
    ):
        self.optimizer = optimizer
        self.T_0 = T_0
        self.T_mult = T_mult
        self.eta_min = eta_min
        
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
        self.T_cur = 0
        self.T_i = T_0
    
    def step(self, epoch: int):
        """Update learning rate."""
        if epoch >= self.T_i:
            self.T_cur = 0
            self.T_i = self.T_i * self.T_mult
        
        # Cosine annealing
        for i, pg in enumerate(self.optimizer.param_groups):
            pg['lr'] = self.eta_min + (self.base_lrs[i] - self.eta_min) * \
                       (1 + math.cos(math.pi * self.T_cur / self.T_i)) / 2
        
        self.T_cur += 1


class ProgressiveResizing:
    """
    Progressive resizing strategy for training.
    
    Starts with smaller images and gradually increases size,
    which can speed up training and improve generalization.
    """
    
    def __init__(
        self,
        start_size: int = 320,
        end_size: int = 640,
        epochs: int = 100,
        milestones: Optional[list] = None
    ):
        self.start_size = start_size
        self.end_size = end_size
        self.epochs = epochs
        self.milestones = milestones or [0.3, 0.6, 0.8]  # Fraction of epochs
    
    def get_size(self, epoch: int) -> int:
        """Get image size for current epoch."""
        progress = epoch / self.epochs
        
        # Find current milestone
        sizes = [self.start_size]
        step = (self.end_size - self.start_size) // len(self.milestones)
        
        for i, m in enumerate(self.milestones):
            if progress >= m:
                sizes.append(self.start_size + step * (i + 1))
        
        return min(sizes[-1], self.end_size)


class EarlyStopping:
    """
    Early stopping to prevent overfitting.
    """
    
    def __init__(self, patience: int = 10, min_delta: float = 0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
    
    def __call__(self, val_score: float) -> bool:
        """
        Check if training should stop.
        
        Args:
            val_score: Validation metric (higher is better)
            
        Returns:
            True if training should stop
        """
        if self.best_score is None:
            self.best_score = val_score
        elif val_score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_score
            self.counter = 0
        
        return self.early_stop


if __name__ == "__main__":
    # Test utilities
    from yolov11.model import YOLOv11
    
    print("Testing training utilities...")
    
    # Test EMA
    model = YOLOv11(num_classes=80, model_size='n')
    ema = ModelEMA(model)
    
    # Simulate training update
    ema.update(model)
    print(f"EMA updates: {ema.updates}")
    
    # Test warmup scheduler
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    warmup = WarmupScheduler(optimizer, warmup_epochs=3)
    
    # Simulate warmup
    warmup.step(epoch=0, batch_idx=50, num_batches=100)
    print(f"LR after warmup step: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Test progressive resizing
    resizer = ProgressiveResizing(start_size=320, end_size=640, epochs=100)
    for epoch in [0, 30, 60, 90]:
        size = resizer.get_size(epoch)
        print(f"Epoch {epoch}: image size = {size}")
    
    print("All utilities OK!")


In [ ]:
%%writefile yolov11/utils/visualization.py
"""
Visualization Utilities for YOLOv11
"""

import cv2
import numpy as np
from typing import List, Tuple, Optional


# COCO class colors (80 classes)
np.random.seed(42)
COCO_COLORS = [(np.random.randint(0, 255), np.random.randint(0, 255), np.random.randint(0, 255)) for _ in range(80)]

# COCO skeleton connections
SKELETON = [
    (0, 1), (0, 2), (1, 3), (2, 4),  # Face
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),  # Arms
    (5, 11), (6, 12), (11, 12),  # Torso
    (11, 13), (13, 15), (12, 14), (14, 16)  # Legs
]

# Keypoint colors
KEYPOINT_COLORS = [
    (255, 0, 0), (255, 85, 0), (255, 170, 0), (255, 255, 0), (170, 255, 0),
    (85, 255, 0), (0, 255, 0), (0, 255, 85), (0, 255, 170), (0, 255, 255),
    (0, 170, 255), (0, 85, 255), (0, 0, 255), (85, 0, 255), (170, 0, 255),
    (255, 0, 255), (255, 0, 170)
]


def draw_boxes(
    image: np.ndarray,
    boxes: np.ndarray,
    labels: np.ndarray = None,
    scores: np.ndarray = None,
    class_names: List[str] = None,
    colors: List[Tuple] = None,
    thickness: int = 2
) -> np.ndarray:
    """
    Draw bounding boxes on image.
    
    Args:
        image: BGR image
        boxes: (N, 4) xyxy format
        labels: (N,) class indices
        scores: (N,) confidence scores
        class_names: List of class names
        colors: List of colors per class
        thickness: Line thickness
    """
    image = image.copy()
    
    if colors is None:
        colors = COCO_COLORS
    
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box[:4])
        
        # Get color
        color = colors[int(labels[i]) % len(colors)] if labels is not None else (0, 255, 0)
        
        # Draw box
        cv2.rectangle(image, (x1, y1), (x2, y2), color, thickness)
        
        # Draw label
        if labels is not None:
            label = class_names[int(labels[i])] if class_names else str(int(labels[i]))
            if scores is not None:
                label = f"{label} {scores[i]:.2f}"
            
            (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(image, (x1, y1 - h - 5), (x1 + w, y1), color, -1)
            cv2.putText(image, label, (x1, y1 - 3), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    return image


def draw_masks(
    image: np.ndarray,
    masks: np.ndarray,
    labels: np.ndarray = None,
    colors: List[Tuple] = None,
    alpha: float = 0.5
) -> np.ndarray:
    """
    Draw instance segmentation masks.
    
    Args:
        image: BGR image
        masks: (N, H, W) binary masks
        labels: (N,) class indices
        colors: List of colors per class
        alpha: Transparency
    """
    image = image.copy()
    
    if colors is None:
        colors = COCO_COLORS
    
    overlay = image.copy()
    
    for i, mask in enumerate(masks):
        color = colors[int(labels[i]) % len(colors)] if labels is not None else (0, 255, 0)
        overlay[mask > 0.5] = color
    
    return cv2.addWeighted(overlay, alpha, image, 1 - alpha, 0)


def draw_keypoints(
    image: np.ndarray,
    keypoints: np.ndarray,
    scores: np.ndarray = None,
    threshold: float = 0.5,
    radius: int = 5,
    thickness: int = 2
) -> np.ndarray:
    """
    Draw human pose keypoints and skeleton.
    
    Args:
        image: BGR image
        keypoints: (N, 17, 3) keypoints [x, y, visibility]
        scores: (N,) detection scores
        threshold: Visibility threshold
        radius: Keypoint radius
        thickness: Skeleton line thickness
    """
    image = image.copy()
    
    for person_idx, kpts in enumerate(keypoints):
        # Draw skeleton
        for i, (p1, p2) in enumerate(SKELETON):
            if kpts[p1, 2] > threshold and kpts[p2, 2] > threshold:
                pt1 = (int(kpts[p1, 0]), int(kpts[p1, 1]))
                pt2 = (int(kpts[p2, 0]), int(kpts[p2, 1]))
                color = KEYPOINT_COLORS[i % len(KEYPOINT_COLORS)]
                cv2.line(image, pt1, pt2, color, thickness)
        
        # Draw keypoints
        for j, (x, y, v) in enumerate(kpts):
            if v > threshold:
                color = KEYPOINT_COLORS[j % len(KEYPOINT_COLORS)]
                cv2.circle(image, (int(x), int(y)), radius, color, -1)
                cv2.circle(image, (int(x), int(y)), radius + 1, (0, 0, 0), 1)
    
    return image


def visualize_predictions(
    image: np.ndarray,
    predictions: dict,
    task: str = 'detect',
    class_names: List[str] = None,
    conf_thres: float = 0.25
) -> np.ndarray:
    """
    Visualize model predictions.
    
    Args:
        image: BGR image
        predictions: Model output dict
        task: 'detect', 'segment', or 'pose'
        class_names: List of class names
        conf_thres: Confidence threshold
    """
    # Filter by confidence
    if 'scores' in predictions:
        mask = predictions['scores'] > conf_thres
        for k in predictions:
            if isinstance(predictions[k], np.ndarray):
                predictions[k] = predictions[k][mask]
    
    # Draw based on task
    if task == 'detect':
        image = draw_boxes(
            image,
            predictions.get('boxes', np.zeros((0, 4))),
            predictions.get('labels'),
            predictions.get('scores'),
            class_names
        )
    
    elif task == 'segment':
        if 'masks' in predictions:
            image = draw_masks(
                image,
                predictions['masks'],
                predictions.get('labels')
            )
        image = draw_boxes(
            image,
            predictions.get('boxes', np.zeros((0, 4))),
            predictions.get('labels'),
            predictions.get('scores'),
            class_names
        )
    
    elif task == 'pose':
        image = draw_boxes(
            image,
            predictions.get('boxes', np.zeros((0, 4))),
            predictions.get('labels'),
            predictions.get('scores'),
            class_names
        )
        if 'keypoints' in predictions:
            image = draw_keypoints(
                image,
                predictions['keypoints'],
                predictions.get('scores')
            )
    
    return image


In [ ]:
%%writefile yolov11/utils/pruning.py
"""
Model Pruning Utilities for YOLOv11
Supports structured and unstructured pruning for model compression.
"""

import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
from typing import List, Tuple, Optional, Dict
import copy


def get_prunable_layers(model: nn.Module) -> List[Tuple[nn.Module, str]]:
    """
    Get all prunable layers (Conv2d and Linear) from model.
    
    Returns:
        List of (module, param_name) tuples
    """
    prunable = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            prunable.append((module, 'weight'))
        elif isinstance(module, nn.Linear):
            prunable.append((module, 'weight'))
    return prunable


def unstructured_pruning(
    model: nn.Module,
    amount: float = 0.3,
    method: str = 'l1'
) -> nn.Module:
    """
    Apply unstructured (weight-level) pruning.
    
    Prunes individual weights based on their magnitude.
    
    Args:
        model: Model to prune
        amount: Fraction of weights to prune (0.0-1.0)
        method: Pruning method ('l1', 'random')
    
    Returns:
        Pruned model
    """
    model = copy.deepcopy(model)
    layers = get_prunable_layers(model)
    
    # Select pruning method
    if method == 'l1':
        prune_fn = prune.L1Unstructured
    elif method == 'random':
        prune_fn = prune.RandomUnstructured
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Apply pruning to all layers
    for module, name in layers:
        prune_fn.apply(module, name, amount=amount)
    
    return model


def structured_pruning(
    model: nn.Module,
    amount: float = 0.3,
    dim: int = 0
) -> nn.Module:
    """
    Apply structured (channel-level) pruning.
    
    Prunes entire channels/filters based on their L2 norm.
    More hardware-friendly than unstructured pruning.
    
    Args:
        model: Model to prune
        amount: Fraction of channels to prune (0.0-1.0)
        dim: Dimension to prune (0=output channels, 1=input channels)
    
    Returns:
        Pruned model
    """
    model = copy.deepcopy(model)
    layers = get_prunable_layers(model)
    
    for module, name in layers:
        if isinstance(module, nn.Conv2d):
            prune.ln_structured(module, name, amount=amount, n=2, dim=dim)
    
    return model


def global_pruning(
    model: nn.Module,
    amount: float = 0.3
) -> nn.Module:
    """
    Apply global unstructured pruning.
    
    Prunes weights globally across all layers based on magnitude,
    which typically leads to better accuracy than layer-wise pruning.
    
    Args:
        model: Model to prune
        amount: Fraction of total weights to prune
    
    Returns:
        Pruned model
    """
    model = copy.deepcopy(model)
    layers = get_prunable_layers(model)
    
    prune.global_unstructured(
        layers,
        pruning_method=prune.L1Unstructured,
        amount=amount
    )
    
    return model


def remove_pruning_reparametrization(model: nn.Module) -> nn.Module:
    """
    Remove pruning reparametrization to make pruning permanent.
    
    After this, the model can be saved without pruning masks.
    """
    model = copy.deepcopy(model)
    
    for module, name in get_prunable_layers(model):
        try:
            prune.remove(module, name)
        except ValueError:
            pass  # Not pruned
    
    return model


def get_sparsity(model: nn.Module) -> Dict[str, float]:
    """
    Calculate sparsity statistics for a model.
    
    Returns:
        Dictionary with sparsity metrics
    """
    total_zeros = 0
    total_params = 0
    layer_sparsities = {}
    
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            weight = module.weight
            zeros = (weight == 0).sum().item()
            total = weight.numel()
            
            total_zeros += zeros
            total_params += total
            
            layer_sparsities[name] = zeros / total
    
    return {
        'global_sparsity': total_zeros / total_params if total_params > 0 else 0,
        'total_zeros': total_zeros,
        'total_params': total_params,
        'layer_sparsities': layer_sparsities
    }


class GradualPruning:
    """
    Gradual pruning during training.
    
    Increases sparsity gradually over training epochs,
    which typically leads to better accuracy retention.
    """
    
    def __init__(
        self,
        model: nn.Module,
        initial_sparsity: float = 0.0,
        final_sparsity: float = 0.5,
        begin_epoch: int = 0,
        end_epoch: int = 100,
        frequency: int = 5
    ):
        self.model = model
        self.initial_sparsity = initial_sparsity
        self.final_sparsity = final_sparsity
        self.begin_epoch = begin_epoch
        self.end_epoch = end_epoch
        self.frequency = frequency
        self.current_sparsity = initial_sparsity
    
    def step(self, epoch: int):
        """Apply pruning for current epoch."""
        if epoch < self.begin_epoch or epoch > self.end_epoch:
            return
        
        if epoch % self.frequency != 0:
            return
        
        # Calculate target sparsity using cubic schedule
        progress = (epoch - self.begin_epoch) / (self.end_epoch - self.begin_epoch)
        target_sparsity = self.final_sparsity + \
            (self.initial_sparsity - self.final_sparsity) * (1 - progress) ** 3
        
        # Calculate amount to prune this step
        if target_sparsity > self.current_sparsity:
            # How much more to prune from remaining weights
            additional = (target_sparsity - self.current_sparsity) / (1 - self.current_sparsity)
            
            # Apply pruning
            layers = get_prunable_layers(self.model)
            for module, name in layers:
                prune.l1_unstructured(module, name, amount=additional)
            
            self.current_sparsity = target_sparsity


if __name__ == "__main__":
    from yolov11.model import YOLOv11
    
    print("Testing pruning utilities...")
    
    # Create model
    model = YOLOv11(num_classes=80, model_size='n')
    original_params = sum(p.numel() for p in model.parameters())
    
    # Test unstructured pruning
    pruned = unstructured_pruning(model, amount=0.3, method='l1')
    sparsity = get_sparsity(pruned)
    print(f"Unstructured pruning (30%): {sparsity['global_sparsity']:.2%} sparse")
    
    # Test global pruning
    pruned = global_pruning(model, amount=0.5)
    sparsity = get_sparsity(pruned)
    print(f"Global pruning (50%): {sparsity['global_sparsity']:.2%} sparse")
    
    # Remove reparametrization
    final = remove_pruning_reparametrization(pruned)
    sparsity = get_sparsity(final)
    print(f"After removing reparametrization: {sparsity['global_sparsity']:.2%} sparse")
    
    print("Pruning utilities OK!")


In [ ]:
%%writefile yolov11/utils/export.py
"""
Model Export Utilities for YOLOv11
Supports: ONNX, TensorRT (optional), INT8 Quantization (optional)

Note: TensorRT requires tensorrt package installed separately.
      Quantization uses PyTorch native quantization.
"""

import torch
import torch.nn as nn
import numpy as np
from typing import Tuple, Optional, Dict, Any, List
from pathlib import Path
import warnings


def export_onnx(
    model: nn.Module,
    save_path: str,
    input_size: Tuple[int, int] = (640, 640),
    batch_size: int = 1,
    opset_version: int = 17,
    dynamic_axes: bool = False,
    simplify: bool = True
) -> str:
    """
    Export model to ONNX format.
    
    Args:
        model: PyTorch model
        save_path: Output path for ONNX file
        input_size: Input image size (H, W)
        batch_size: Batch size for export
        opset_version: ONNX opset version
        dynamic_axes: Enable dynamic batch size
        simplify: Simplify ONNX model (requires onnx-simplifier)
    
    Returns:
        Path to saved ONNX model
    """
    model = model.eval()
    
    # Create dummy input
    dummy_input = torch.randn(batch_size, 3, *input_size)
    
    # Set up dynamic axes
    dynamic = None
    if dynamic_axes:
        dynamic = {
            'images': {0: 'batch'},
            'output': {0: 'batch'}
        }
    
    # Export
    save_path = Path(save_path)
    save_path = save_path.with_suffix('.onnx')
    
    torch.onnx.export(
        model,
        dummy_input,
        str(save_path),
        opset_version=opset_version,
        input_names=['images'],
        output_names=['output'],
        dynamic_axes=dynamic
    )
    
    print(f"ONNX model exported to: {save_path}")
    
    # Simplify if requested
    if simplify:
        try:
            import onnx
            from onnxsim import simplify as onnx_simplify
            
            onnx_model = onnx.load(str(save_path))
            onnx_model, check = onnx_simplify(onnx_model)
            
            if check:
                onnx.save(onnx_model, str(save_path))
                print("ONNX model simplified successfully")
        except ImportError:
            print("onnx-simplifier not installed, skipping simplification")
        except Exception as e:
            print(f"Simplification failed: {e}")
    
    return str(save_path)


def export_tensorrt(
    onnx_path: str,
    save_path: str,
    fp16: bool = True,
    int8: bool = False,
    max_batch_size: int = 1,
    workspace_size: int = 4,
    calibrator: Optional[Any] = None
) -> str:
    """
    Export ONNX model to TensorRT engine.
    
    Requires: tensorrt package
    
    Args:
        onnx_path: Path to ONNX model
        save_path: Output path for TensorRT engine
        fp16: Enable FP16 precision
        int8: Enable INT8 precision (requires calibrator)
        max_batch_size: Maximum batch size
        workspace_size: GPU workspace size in GB
        calibrator: INT8 calibrator for quantization
    
    Returns:
        Path to saved TensorRT engine
    """
    try:
        import tensorrt as trt
    except ImportError:
        raise ImportError(
            "TensorRT not installed. Install with:\n"
            "pip install tensorrt\n"
            "Or download from NVIDIA website."
        )
    
    TRT_LOGGER = trt.Logger(trt.Logger.INFO)
    
    # Create builder
    builder = trt.Builder(TRT_LOGGER)
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    parser = trt.OnnxParser(network, TRT_LOGGER)
    
    # Parse ONNX
    with open(onnx_path, 'rb') as f:
        if not parser.parse(f.read()):
            for error in range(parser.num_errors):
                print(parser.get_error(error))
            raise RuntimeError("Failed to parse ONNX model")
    
    # Configure builder
    config = builder.create_builder_config()
    # TRT 10.0+ uses set_memory_pool_limit instead of max_workspace_size
    if hasattr(config, 'set_memory_pool_limit'):
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, workspace_size * (1 << 30))
    else:
        config.max_workspace_size = workspace_size * (1 << 30)
    
    if fp16:
        config.set_flag(trt.BuilderFlag.FP16)
    
    if int8:
        config.set_flag(trt.BuilderFlag.INT8)
        if calibrator:
            config.int8_calibrator = calibrator
    
    # Build engine
    # TRT 10.0+ uses build_serialized_network instead of build_engine
    if hasattr(builder, 'build_serialized_network'):
        engine = builder.build_serialized_network(network, config)
        if engine is None:
            raise RuntimeError("Failed to build TensorRT engine")
        
        # Save engine (it's already serialized)
        save_path = Path(save_path).with_suffix('.engine')
        with open(save_path, 'wb') as f:
            f.write(engine)
    else:
        engine = builder.build_engine(network, config)
        if engine is None:
            raise RuntimeError("Failed to build TensorRT engine")
            
        # Save engine
        save_path = Path(save_path).with_suffix('.engine')
        with open(save_path, 'wb') as f:
            f.write(engine.serialize())
    
    print(f"TensorRT engine exported to: {save_path}")
    return str(save_path)


class TRTInference:
    """Helper class for TensorRT engine inference (Compatible with TRT 10.x)."""
    
    def __init__(self, engine_path: str):
        import tensorrt as trt
        import pycuda.driver as cuda
        import pycuda.autoinit  # Required for CUDA context
        
        self.logger = trt.Logger(trt.Logger.INFO)
        with open(engine_path, 'rb') as f:
            self.runtime = trt.Runtime(self.logger)
            self.engine = self.runtime.deserialize_cuda_engine(f.read())
            self.context = self.engine.create_execution_context()
            
        self.inputs, self.outputs = [], []
        self.input_names, self.output_names = [], []
        self.stream = cuda.Stream()
        
        # TRT 10.x uses tensor names instead of indices
        for i in range(self.engine.num_io_tensors):
            name = self.engine.get_tensor_name(i)
            is_input = self.engine.get_tensor_mode(name) == trt.TensorIOMode.INPUT
            
            shape = self.engine.get_tensor_shape(name)
            dtype = trt.nptype(self.engine.get_tensor_dtype(name))
            
            # Host and Device buffers
            size = trt.volume(shape)
            host_mem = cuda.pagelocked_empty(size, dtype)
            device_mem = cuda.mem_alloc(host_mem.nbytes)
            
            # Bind tensor address
            self.context.set_tensor_address(name, int(device_mem))
            
            if is_input:
                self.input_names.append(name)
                self.inputs.append({'host': host_mem, 'device': device_mem, 'name': name, 'shape': shape})
            else:
                self.output_names.append(name)
                self.outputs.append({'host': host_mem, 'device': device_mem, 'name': name, 'shape': shape})
                
    def __call__(self, img: torch.Tensor) -> List[torch.Tensor]:
        import pycuda.driver as cuda
        # Copy input to host
        input_data = self.inputs[0]
        np_img = img.cpu().numpy().astype(input_data['host'].dtype)
        np.copyto(input_data['host'], np_img.ravel())
        
        # Transfer to device, execute, transfer back
        cuda.memcpy_htod_async(input_data['device'], input_data['host'], self.stream)
        
        # TRT 10.x uses execute_async_v3
        if hasattr(self.context, 'execute_async_v3'):
            self.context.execute_async_v3(stream_handle=self.stream.handle)
        else:
            # Fallback for older versions (unlikely if we reached here with 10.x logic)
            bindings = [int(i['device']) for i in self.inputs] + [int(o['device']) for o in self.outputs]
            self.context.execute_async_v2(bindings=bindings, stream_handle=self.stream.handle)
            
        for out in self.outputs:
            cuda.memcpy_dtoh_async(out['host'], out['device'], self.stream)
        self.stream.synchronize()
        
        # Convert back to torch tensors
        results = []
        for out in self.outputs:
            results.append(torch.from_numpy(out['host'].reshape(out['shape'])))
        return results


class QuantizationConfig:
    """Configuration for model quantization."""
    
    def __init__(
        self,
        backend: str = 'fbgemm',  # 'fbgemm' for x86, 'qnnpack' for ARM
        dtype: str = 'qint8'
    ):
        self.backend = backend
        self.dtype = getattr(torch, dtype)


def quantize_dynamic(
    model: nn.Module,
    config: Optional[QuantizationConfig] = None
) -> nn.Module:
    """
    Apply dynamic quantization (weights only).
    
    Fast and simple, good for inference on CPU.
    Quantizes Linear and LSTM layers.
    
    Args:
        model: Model to quantize
        config: Quantization configuration
    
    Returns:
        Quantized model
    """
    if config is None:
        config = QuantizationConfig()
    
    torch.backends.quantized.engine = config.backend
    
    quantized = torch.quantization.quantize_dynamic(
        model,
        {nn.Linear},
        dtype=config.dtype
    )
    
    return quantized


def quantize_static(
    model: nn.Module,
    calibration_loader,
    config: Optional[QuantizationConfig] = None,
    num_batches: int = 100
) -> nn.Module:
    """
    Apply static quantization (weights and activations).
    
    Requires calibration data for activation statistics.
    Better accuracy than dynamic quantization.
    
    Args:
        model: Model to quantize (must have qconfig set)
        calibration_loader: DataLoader for calibration
        config: Quantization configuration
        num_batches: Number of batches for calibration
    
    Returns:
        Quantized model
    """
    if config is None:
        config = QuantizationConfig()
    
    torch.backends.quantized.engine = config.backend
    
    # Prepare model for quantization
    model.eval()
    model.qconfig = torch.quantization.get_default_qconfig(config.backend)
    
    # Fuse common patterns
    model_fused = _fuse_model(model)
    
    # Insert observers
    model_prepared = torch.quantization.prepare(model_fused)
    
    # Calibration
    print(f"Calibrating with {num_batches} batches...")
    with torch.no_grad():
        for i, (images, _) in enumerate(calibration_loader):
            if i >= num_batches:
                break
            model_prepared(images)
    
    # Convert to quantized
    model_quantized = torch.quantization.convert(model_prepared)
    
    return model_quantized


def _fuse_model(model: nn.Module) -> nn.Module:
    """Fuse Conv-BN-ReLU patterns for quantization."""
    import copy
    model = copy.deepcopy(model)
    
    # Common fusion patterns
    for name, module in model.named_modules():
        if hasattr(module, 'conv') and hasattr(module, 'bn'):
            # Fuse conv + bn
            torch.quantization.fuse_modules(
                module, 
                ['conv', 'bn'], 
                inplace=True
            )
    
    return model


def get_model_size(model: nn.Module) -> Dict[str, float]:
    """
    Get model size statistics.
    
    Returns:
        Dictionary with size information
    """
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    
    return {
        'params_mb': param_size / (1024 ** 2),
        'buffers_mb': buffer_size / (1024 ** 2),
        'total_mb': (param_size + buffer_size) / (1024 ** 2),
        'num_params': sum(p.numel() for p in model.parameters())
    }


if __name__ == "__main__":
    from yolov11.model import YOLOv11
    
    print("Testing export utilities...")
    
    # Create model
    model = YOLOv11(num_classes=80, model_size='n')
    model.eval()
    
    # Get original size
    orig_size = get_model_size(model)
    print(f"Original model: {orig_size['total_mb']:.2f} MB, {orig_size['num_params']:,} params")
    
    # Test dynamic quantization
    try:
        quantized = quantize_dynamic(model)
        quant_size = get_model_size(quantized)
        print(f"Quantized model: {quant_size['total_mb']:.2f} MB")
    except Exception as e:
        print(f"Dynamic quantization: {e}")
    
    # Test ONNX export (if torch.onnx available)
    try:
        import tempfile
        import os
        
        with tempfile.TemporaryDirectory() as tmpdir:
            onnx_path = export_onnx(
                model, 
                os.path.join(tmpdir, "model.onnx"),
                input_size=(320, 320),
                simplify=False
            )
            print(f"ONNX export successful: {os.path.getsize(onnx_path) / 1024 / 1024:.2f} MB")
    except Exception as e:
        print(f"ONNX export: {e}")
    
    print("Export utilities OK!")


In [ ]:
%%writefile yolov11/data/__init__.py
"""
YOLOv11 Data Package
Dataset loaders and augmentations
"""

from .dataset import COCODataset, YOLODataset, create_dataloader
from .augmentations import (
    Mosaic,
    MixUp,
    RandomHSV,
    RandomFlip,
    Compose,
    LetterBox
)

__all__ = [
    "COCODataset",
    "YOLODataset", 
    "create_dataloader",
    "Mosaic",
    "MixUp",
    "RandomHSV",
    "RandomFlip",
    "Compose",
    "LetterBox"
]


In [ ]:
%%writefile yolov11/data/augmentations.py
"""
Data Augmentations for YOLOv11
"""

import cv2
import numpy as np
import torch
import random
from typing import Tuple, List, Dict, Callable


class Compose:
    """Compose multiple augmentations."""
    def __init__(self, transforms: List[Callable]):
        self.transforms = transforms
    
    def __call__(self, image: np.ndarray, labels: Dict) -> Tuple[np.ndarray, Dict]:
        for t in self.transforms:
            image, labels = t(image, labels)
        return image, labels


class LetterBox:
    """Resize and pad image while maintaining aspect ratio."""
    
    def __init__(self, new_shape=(640, 640), color=(114, 114, 114)):
        self.new_shape = new_shape if isinstance(new_shape, tuple) else (new_shape, new_shape)
        self.color = color
    
    def __call__(self, image: np.ndarray, labels: Dict | None = None) -> Tuple[np.ndarray, Dict]:
        shape = image.shape[:2]
        r = min(self.new_shape[0] / shape[0], self.new_shape[1] / shape[1])
        new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
        dw, dh = (self.new_shape[1] - new_unpad[0]) / 2, (self.new_shape[0] - new_unpad[1]) / 2
        
        if shape[::-1] != new_unpad:
            image = cv2.resize(image, new_unpad, interpolation=cv2.INTER_LINEAR)
        
        top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
        left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
        image = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT, value=self.color)
        
        if labels and 'bboxes' in labels and len(labels['bboxes']) > 0:
            labels['bboxes'] = labels['bboxes'].copy()
            labels['bboxes'][:, [0, 2]] = labels['bboxes'][:, [0, 2]] * r + dw
            labels['bboxes'][:, [1, 3]] = labels['bboxes'][:, [1, 3]] * r + dh
        
        return image, labels  # type: ignore


class RandomHSV:
    """Random HSV color augmentation."""
    
    def __init__(self, h_gain=0.015, s_gain=0.7, v_gain=0.4):
        self.h_gain, self.s_gain, self.v_gain = h_gain, s_gain, v_gain
    
    def __call__(self, image: np.ndarray, labels: Dict) -> Tuple[np.ndarray, Dict]:
        if random.random() < 0.5:
            return image, labels
        r = np.random.uniform(-1, 1, 3) * [self.h_gain, self.s_gain, self.v_gain] + 1
        hue, sat, val = cv2.split(cv2.cvtColor(image, cv2.COLOR_BGR2HSV))
        x = np.arange(0, 256, dtype=r.dtype)
        hue = cv2.LUT(hue, ((x * r[0]) % 180).astype(image.dtype))
        sat = cv2.LUT(sat, np.clip(x * r[1], 0, 255).astype(image.dtype))
        val = cv2.LUT(val, np.clip(x * r[2], 0, 255).astype(image.dtype))
        return cv2.cvtColor(cv2.merge([hue, sat, val]), cv2.COLOR_HSV2BGR), labels


class RandomFlip:
    """Random horizontal flip with keypoint swap support."""
    
    def __init__(self, p=0.5):
        self.p = p
        self.flip_idx = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]
    
    def __call__(self, image: np.ndarray, labels: Dict) -> Tuple[np.ndarray, Dict]:
        if random.random() < self.p:
            h, w = image.shape[:2]
            image = np.fliplr(image)
            if 'bboxes' in labels and len(labels['bboxes']) > 0:
                bboxes = labels['bboxes'].copy()
                bboxes[:, [0, 2]] = w - bboxes[:, [2, 0]]
                labels['bboxes'] = bboxes
            if 'keypoints' in labels and len(labels['keypoints']) > 0:
                kpts = labels['keypoints'].copy()
                kpts[:, :, 0] = w - kpts[:, :, 0]
                labels['keypoints'] = kpts[:, self.flip_idx, :]
        return np.ascontiguousarray(image), labels


class Mosaic:
    """Mosaic augmentation combining 4 images."""
    
    def __init__(self, img_size=640, p=1.0):
        self.img_size = img_size
        self.p = p
    
    def __call__(self, images: List[np.ndarray], labels_list: List[Dict]) -> Tuple[np.ndarray, Dict]:
        if random.random() > self.p or len(images) < 4:
            return images[0], labels_list[0]
        
        s = self.img_size
        xc, yc = [int(random.uniform(s * 0.5, s * 1.5)) for _ in range(2)]
        img4 = np.full((s * 2, s * 2, 3), 114, dtype=np.uint8)
        bboxes_list, labels_cls = [], []
        
        for i, (img, labels) in enumerate(zip(images[:4], labels_list[:4])):
            h, w = img.shape[:2]
            if i == 0:
                x1a, y1a, x2a, y2a = max(xc-w, 0), max(yc-h, 0), xc, yc
            elif i == 1:
                x1a, y1a, x2a, y2a = xc, max(yc-h, 0), min(xc+w, s*2), yc
            elif i == 2:
                x1a, y1a, x2a, y2a = max(xc-w, 0), yc, xc, min(s*2, yc+h)
            else:
                x1a, y1a, x2a, y2a = xc, yc, min(xc+w, s*2), min(s*2, yc+h)
            
            if i == 0:
                x1b = w - (x2a - x1a)
                y1b = h - (y2a - y1a)
                x2b, y2b = w, h
            elif i == 1:
                x1b = 0
                y1b = h - (y2a - y1a)
                x2b = min(x2a - x1a, w)
                y2b = h
            elif i == 2:
                x1b = w - (x2a - x1a)
                y1b = 0
                x2b, y2b = w, min(y2a - y1a, h)
            else:
                x1b, y1b = 0, 0
                x2b = min(x2a - x1a, w)
                y2b = min(y2a - y1a, h)
            
            img4[y1a:y2a, x1a:x2a] = img[y1b:y2b, x1b:x2b]
            
            if 'bboxes' in labels and len(labels['bboxes']) > 0:
                bboxes = labels['bboxes'].copy()
                bboxes[:, [0, 2]] += (x1a - x1b)
                bboxes[:, [1, 3]] += (y1a - y1b)
                bboxes_list.append(bboxes)
                labels_cls.append(labels['labels'])
        
        result = {}
        if bboxes_list:
            result['bboxes'] = np.clip(np.concatenate(bboxes_list), 0, 2*s)
            result['labels'] = np.concatenate(labels_cls)
        
        img4 = img4[yc-s//2:yc+s//2, xc-s//2:xc+s//2]
        if 'bboxes' in result:
            result['bboxes'][:, [0, 2]] -= (xc - s//2)
            result['bboxes'][:, [1, 3]] -= (yc - s//2)
        return img4, result


class MixUp:
    """MixUp augmentation for blending two images."""
    
    def __init__(self, alpha=32.0, p=0.5):
        self.alpha, self.p = alpha, p
    
    def __call__(self, img1, labels1, img2, labels2):
        if random.random() > self.p:
            return img1, labels1
        r = np.random.beta(self.alpha, self.alpha)
        img = (img1 * r + img2 * (1 - r)).astype(np.uint8)
        labels = {}
        if 'bboxes' in labels1 and 'bboxes' in labels2:
            labels['bboxes'] = np.concatenate([labels1['bboxes'], labels2['bboxes']])
            labels['labels'] = np.concatenate([labels1['labels'], labels2['labels']])
        return img, labels


class ToTensor:
    """Convert numpy array to PyTorch tensor."""
    
    def __call__(self, image: np.ndarray, labels: Dict):
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = torch.from_numpy(image).float().permute(2, 0, 1) / 255.0  # type: ignore
        for k, v in labels.items():
            if isinstance(v, np.ndarray):
                labels[k] = torch.from_numpy(v)
        return image, labels


class CutMix:
    """
    CutMix augmentation for object detection.
    Cuts a patch from one image and pastes onto another.
    """
    
    def __init__(self, p: float = 0.5, beta: float = 1.0):
        self.p = p
        self.beta = beta
    
    def __call__(self, img1, labels1, img2, labels2):
        if random.random() > self.p:
            return img1, labels1
        
        h, w = img1.shape[:2]
        
        lam = np.random.beta(self.beta, self.beta)
        
        cut_rat = np.sqrt(1.0 - lam)
        cut_w = int(w * cut_rat)
        cut_h = int(h * cut_rat)
        
        cx = np.random.randint(w)
        cy = np.random.randint(h)
        
        bbx1 = np.clip(cx - cut_w // 2, 0, w)
        bby1 = np.clip(cy - cut_h // 2, 0, h)
        bbx2 = np.clip(cx + cut_w // 2, 0, w)
        bby2 = np.clip(cy + cut_h // 2, 0, h)
        
        img = img1.copy()
        img[bby1:bby2, bbx1:bbx2] = img2[bby1:bby2, bbx1:bbx2]
        
        labels = {}
        if 'bboxes' in labels1 and 'bboxes' in labels2:
            boxes1 = labels1['bboxes'].copy()
            keep1 = []
            for i, box in enumerate(boxes1):
                box_area = (box[2] - box[0]) * (box[3] - box[1])
                inter_x1 = max(box[0], bbx1)
                inter_y1 = max(box[1], bby1)
                inter_x2 = min(box[2], bbx2)
                inter_y2 = min(box[3], bby2)
                inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
                if inter_area < 0.5 * box_area:
                    keep1.append(i)
            
            boxes2 = labels2['bboxes'].copy()
            keep2 = []
            for i, box in enumerate(boxes2):
                box_area = (box[2] - box[0]) * (box[3] - box[1])
                inter_x1 = max(box[0], bbx1)
                inter_y1 = max(box[1], bby1)
                inter_x2 = min(box[2], bbx2)
                inter_y2 = min(box[3], bby2)
                inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
                if inter_area > 0.5 * box_area:
                    keep2.append(i)
            
            kept_boxes1 = boxes1[keep1] if len(keep1) > 0 else np.zeros((0, 4))
            kept_boxes2 = boxes2[keep2] if len(keep2) > 0 else np.zeros((0, 4))
            kept_labels1 = labels1['labels'][keep1] if len(keep1) > 0 else np.array([])
            kept_labels2 = labels2['labels'][keep2] if len(keep2) > 0 else np.array([])
            
            labels['bboxes'] = np.concatenate([kept_boxes1, kept_boxes2]) if len(kept_boxes1) + len(kept_boxes2) > 0 else np.zeros((0, 4))
            labels['labels'] = np.concatenate([kept_labels1, kept_labels2]) if len(kept_labels1) + len(kept_labels2) > 0 else np.array([])
        
        return img, labels


class CopyPaste:
    """
    Copy-Paste augmentation for object detection.
    
    Copies object regions from one image and pastes them onto another,
    merging the bounding boxes and labels accordingly.
    
    Works with both masks (for segmentation) and bounding boxes (for detection).
    """
    
    def __init__(self, p: float = 0.5, max_objects: int = 3):
        """
        Args:
            p: Probability of applying augmentation
            max_objects: Maximum number of objects to copy
        """
        self.p = p
        self.max_objects = max_objects
    
    def __call__(self, img1, labels1, img2, labels2):
        """
        Apply CopyPaste augmentation.
        
        Args:
            img1: Target image (where objects will be pasted)
            labels1: Target labels dict with 'bboxes' and 'labels'
            img2: Source image (where objects will be copied from)
            labels2: Source labels dict
            
        Returns:
            Augmented image and labels
        """
        if random.random() > self.p:
            return img1, labels1
        
        # Check if source has objects to copy
        if 'bboxes' not in labels2 or len(labels2.get('bboxes', [])) == 0:
            return img1, labels1
        
        h, w = img1.shape[:2]
        source_bboxes = labels2['bboxes']
        source_labels = labels2['labels']
        n_objects = len(source_bboxes)
        
        if n_objects == 0:
            return img1, labels1
        
        # Select random objects to copy
        n_copy = min(self.max_objects, n_objects)
        indices = random.sample(range(n_objects), n_copy)
        
        img_out = img1.copy()
        new_bboxes = []
        new_labels = []
        
        for idx in indices:
            bbox = source_bboxes[idx].astype(int)
            x1, y1, x2, y2 = bbox
            
            # Ensure valid bbox
            x1, x2 = max(0, x1), min(w, x2)
            y1, y2 = max(0, y1), min(h, y2)
            
            if x2 <= x1 or y2 <= y1:
                continue
            
            # Extract object region from source
            obj_region = img2[y1:y2, x1:x2].copy()
            
            # Random placement in target image
            obj_h, obj_w = obj_region.shape[:2]
            
            # Calculate valid placement range
            max_x = w - obj_w
            max_y = h - obj_h
            
            if max_x <= 0 or max_y <= 0:
                continue
            
            # Random position
            paste_x = random.randint(0, max_x)
            paste_y = random.randint(0, max_y)
            
            # Paste object
            img_out[paste_y:paste_y+obj_h, paste_x:paste_x+obj_w] = obj_region
            
            # Create new bbox
            new_bbox = np.array([paste_x, paste_y, paste_x + obj_w, paste_y + obj_h])
            new_bboxes.append(new_bbox)
            new_labels.append(source_labels[idx])
        
        # Merge labels
        result_labels = {}
        if 'bboxes' in labels1 and len(labels1['bboxes']) > 0:
            if new_bboxes:
                result_labels['bboxes'] = np.concatenate([labels1['bboxes'], np.array(new_bboxes)])
                result_labels['labels'] = np.concatenate([labels1['labels'], np.array(new_labels)])
            else:
                result_labels['bboxes'] = labels1['bboxes']
                result_labels['labels'] = labels1['labels']
        elif new_bboxes:
            result_labels['bboxes'] = np.array(new_bboxes)
            result_labels['labels'] = np.array(new_labels)
        else:
            result_labels['bboxes'] = np.zeros((0, 4))
            result_labels['labels'] = np.array([])
        
        return img_out, result_labels



In [ ]:
%%writefile yolov11/data/dataset.py
"""
Dataset Loaders for YOLOv11
Supports COCO and YOLO format datasets
"""

import os
import json
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from typing import Dict, List, Tuple, Optional
from pathlib import Path

from .augmentations import Compose, LetterBox, RandomHSV, RandomFlip, Mosaic, ToTensor


class COCODataset(Dataset):
    """
    COCO format dataset for detection, segmentation, and pose estimation.
    
    Args:
        root: Path to images directory
        ann_file: Path to COCO annotation JSON file
        task: 'detect', 'segment', or 'pose'
        img_size: Target image size
        augment: Apply augmentations
        mosaic: Use mosaic augmentation
    """
    
    def __init__(
        self,
        root: str,
        ann_file: str,
        task: str = 'detect',
        img_size: int = 640,
        augment: bool = True,
        mosaic: bool = True
    ):
        super().__init__()
        self.root = Path(root)
        self.task = task
        self.img_size = img_size
        self.augment = augment
        self.use_mosaic = mosaic and augment
        
        with open(ann_file, 'r') as f:
            coco = json.load(f)
        
        self.images = {img['id']: img for img in coco['images']}
        self.categories = {cat['id']: cat for cat in coco['categories']}
        
        self.cat_id_to_idx = {cat_id: i for i, cat_id in enumerate(sorted(self.categories.keys()))}
        
        self.img_to_anns = {}
        for ann in coco['annotations']:
            img_id = ann['image_id']
            if img_id not in self.img_to_anns:
                self.img_to_anns[img_id] = []
            self.img_to_anns[img_id].append(ann)
        
        self.img_ids = [img_id for img_id in self.images.keys() if img_id in self.img_to_anns]
        
        self.letterbox = LetterBox((img_size, img_size))
        self.mosaic = Mosaic(img_size) if self.use_mosaic else None
        
        if augment:
            self.transforms = Compose([RandomHSV(), RandomFlip()])
        else:
            self.transforms = None
        
        self.to_tensor = ToTensor()
    
    def __len__(self) -> int:
        return len(self.img_ids)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Dict]:
        if self.use_mosaic and np.random.random() < 0.5:
            return self._load_mosaic(idx)
        return self._load_single(idx)
    
    def _load_single(self, idx: int) -> Tuple[torch.Tensor, Dict]:
        img_id = self.img_ids[idx]
        img_info = self.images[img_id]
        
        filename = img_info['file_name']
        img_path = self.root / filename
        
        # If image not found in root, check subdirectories (e.g., train2017, val2017)
        if not img_path.exists():
            for split in ['train2017', 'val2017', 'test2017']:
                alt_path = self.root / split / filename
                if alt_path.exists():
                    img_path = alt_path
                    break
        
        image = cv2.imread(str(img_path))
        if image is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        
        labels = self._parse_annotations(img_id, img_info)
        
        image, labels = self.letterbox(image, labels)
        
        if self.transforms:
            image, labels = self.transforms(image, labels)
        
        image, labels = self.to_tensor(image, labels)
        
        return image, labels
    
    def _load_mosaic(self, idx: int) -> Tuple[torch.Tensor, Dict]:
        indices = [idx] + [np.random.randint(len(self)) for _ in range(3)]
        images, labels_list = [], []
        
        for i in indices:
            img_id = self.img_ids[i]
            img_info = self.images[img_id]
            filename = img_info['file_name']
            img_path = self.root / filename
            
            if not img_path.exists():
                for split in ['train2017', 'val2017', 'test2017']:
                    alt_path = self.root / split / filename
                    if alt_path.exists():
                        img_path = alt_path
                        break
            
            img = cv2.imread(str(img_path))
            if img is not None:
                images.append(cv2.resize(img, (self.img_size, self.img_size)))
                labels_list.append(self._parse_annotations(img_id, img_info))
        
        if len(images) < 4:
            return self._load_single(idx)
        
        image, labels = self.mosaic(images, labels_list)
        
        if self.transforms:
            image, labels = self.transforms(image, labels)
        
        image, labels = self.to_tensor(image, labels)
        return image, labels
    
    def _parse_annotations(self, img_id: int, img_info: Dict) -> Dict:
        anns = self.img_to_anns.get(img_id, [])
        h, w = img_info['height'], img_info['width']
        
        bboxes, labels, keypoints, masks = [], [], [], []
        
        for ann in anns:
            x, y, bw, bh = ann['bbox']
            bboxes.append([x, y, x + bw, y + bh])
            labels.append(self.cat_id_to_idx[ann['category_id']])
            
            if self.task == 'pose' and 'keypoints' in ann:
                kpts = np.array(ann['keypoints']).reshape(-1, 3)
                keypoints.append(kpts)
            
            if self.task == 'segment' and 'segmentation' in ann:
                mask = self._decode_segmentation(ann['segmentation'], h, w)
                masks.append(mask)
        
        result = {
            'bboxes': np.array(bboxes, dtype=np.float32) if bboxes else np.zeros((0, 4), dtype=np.float32),
            'labels': np.array(labels, dtype=np.int64) if labels else np.zeros(0, dtype=np.int64),
            'image_id': img_id
        }
        
        if keypoints:
            result['keypoints'] = np.array(keypoints, dtype=np.float32)
        if masks:
            result['masks'] = np.array(masks, dtype=np.float32)
        
        return result
    
    def _decode_segmentation(self, segm, height: int, width: int) -> np.ndarray:
        """Decode COCO segmentation to binary mask."""
        from pycocotools import mask as maskUtils
        
        if isinstance(segm, list):
            rles = maskUtils.frPyObjects(segm, height, width)
            rle = maskUtils.merge(rles)
        else:
            rle = segm
        
        return maskUtils.decode(rle)


class YOLODataset(Dataset):
    """
    YOLO format dataset (txt files with labels).
    Label format: class x_center y_center width height (normalized)
    """
    
    def __init__(
        self,
        root: str,
        img_size: int = 640,
        augment: bool = True
    ):
        super().__init__()
        self.root = Path(root)
        self.img_size = img_size
        self.augment = augment
        
        self.img_files = list(self.root.glob('images/**/*.jpg')) + \
                         list(self.root.glob('images/**/*.png'))
        
        self.letterbox = LetterBox((img_size, img_size))
        self.transforms = Compose([RandomHSV(), RandomFlip()]) if augment else None
        self.to_tensor = ToTensor()
    
    def __len__(self) -> int:
        return len(self.img_files)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Dict]:
        img_path = self.img_files[idx]
        
        image = cv2.imread(str(img_path))
        h, w = image.shape[:2]
        
        label_path = str(img_path).replace('images', 'labels').replace('.jpg', '.txt').replace('.png', '.txt')
        labels = self._load_labels(label_path, w, h)
        
        image, labels = self.letterbox(image, labels)
        if self.transforms:
            image, labels = self.transforms(image, labels)
        image, labels = self.to_tensor(image, labels)
        
        return image, labels
    
    def _load_labels(self, path: str, img_w: int, img_h: int) -> Dict:
        bboxes, labels = [], []
        
        if os.path.exists(path):
            with open(path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls, xc, yc, w, h = map(float, parts[:5])
                        x1 = (xc - w/2) * img_w
                        y1 = (yc - h/2) * img_h
                        x2 = (xc + w/2) * img_w
                        y2 = (yc + h/2) * img_h
                        bboxes.append([x1, y1, x2, y2])
                        labels.append(int(cls))
        
        return {
            'bboxes': np.array(bboxes, dtype=np.float32) if bboxes else np.zeros((0, 4)),
            'labels': np.array(labels, dtype=np.int64) if labels else np.zeros(0, dtype=np.int64)
        }


def collate_fn(batch: List[Tuple]) -> Tuple[torch.Tensor, Dict]:
    """Custom collate function for variable-size targets."""
    images, labels_list = zip(*batch)
    images = torch.stack(images, 0)
    
    max_objs = max(len(lab['labels']) for lab in labels_list)
    batch_size = len(labels_list)
    
    batch_labels = torch.zeros(batch_size, max_objs, dtype=torch.int64)
    batch_bboxes = torch.zeros(batch_size, max_objs, 4)
    mask_gt = torch.zeros(batch_size, max_objs, dtype=torch.bool)
    
    for i, lab in enumerate(labels_list):
        n = len(lab['labels'])
        if n > 0:
            batch_labels[i, :n] = lab['labels'] if isinstance(lab['labels'], torch.Tensor) else torch.tensor(lab['labels'])
            batch_bboxes[i, :n] = lab['bboxes'] if isinstance(lab['bboxes'], torch.Tensor) else torch.tensor(lab['bboxes'])
            mask_gt[i, :n] = True
    
    targets = {'labels': batch_labels, 'bboxes': batch_bboxes, 'mask_gt': mask_gt}
    
    if 'keypoints' in labels_list[0]:
        batch_kpts = torch.zeros(batch_size, max_objs, 17, 3)
        for i, lab in enumerate(labels_list):
            if 'keypoints' in lab and len(lab['keypoints']) > 0:
                n = len(lab['keypoints'])
                kpts = lab['keypoints'] if isinstance(lab['keypoints'], torch.Tensor) else torch.tensor(lab['keypoints'])
                batch_kpts[i, :n] = kpts
        targets['keypoints'] = batch_kpts
    
    return images, targets


def create_dataloader(
    root: str,
    ann_file: str = None,
    task: str = 'detect',
    img_size: int = 640,
    batch_size: int = 16,
    augment: bool = True,
    shuffle: bool = True,
    num_workers: int = 4,
    distributed: bool = False
) -> DataLoader:
    """Create a DataLoader for training or validation."""
    
    if ann_file:
        dataset = COCODataset(root, ann_file, task, img_size, augment)
    else:
        dataset = YOLODataset(root, img_size, augment)
    
    sampler = DistributedSampler(dataset, shuffle=shuffle) if distributed else None
    
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=(shuffle and sampler is None),
        num_workers=num_workers,
        sampler=sampler,
        collate_fn=collate_fn,
        pin_memory=True,
        drop_last=True
    )


In [ ]:
%%writefile train.py
"""
YOLOv11 Training Script
Supports training for detection, segmentation, and pose estimation
"""

import os
import sys
import argparse
import time
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional, Union

import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm

# Add parent directory to path
sys.path.insert(0, str(Path(__file__).parent))

from yolov11.model import YOLOv11, create_model
from yolov11.losses import YOLOv11Loss
from yolov11.data import create_dataloader
from yolov11.utils.metrics import Metrics
from yolov11.utils.nms import non_max_suppression
from yolov11.utils.training import ModelEMA, WarmupScheduler


def train_one_epoch(
    model: nn.Module,
    dataloader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    scaler: 'Any',
    device: torch.device,
    epoch: int,
    use_amp: bool = True,
    warmup: 'WarmupScheduler | None' = None,
    ema: 'ModelEMA | None' = None
) -> dict:
    """Train for one epoch with warmup and EMA support."""
    model.train()
    
    total_loss = 0
    loss_items = {'loss_box': 0, 'loss_cls': 0, 'loss_dfl': 0}
    num_batches = len(dataloader)
    rank = int(os.environ.get('RANK', -1))
    
    pbar = None
    if rank in [-1, 0]:
        pbar = tqdm(dataloader, desc=f'Ep{epoch}', bar_format='{desc}: {percentage:3.0f}%|{bar:10}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]{postfix}')
    
    for batch_idx, (images, targets) in enumerate(pbar or dataloader):
        # Apply warmup scheduler
        if warmup is not None:
            warmup.step(epoch, batch_idx, num_batches)
        
        images = images.to(device)
        for k in targets:
            if isinstance(targets[k], torch.Tensor):
                targets[k] = targets[k].to(device)
        
        optimizer.zero_grad()
        
        if use_amp:
            with torch.amp.autocast('cuda' if device.type == 'cuda' else 'cpu', enabled=use_amp):
                outputs = model(images)
                loss, loss_dict = criterion(outputs, targets)
        else:
            outputs = model(images)
            loss, loss_dict = criterion(outputs, targets)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        # Update EMA after optimizer step
        if ema is not None:
            ema.update(model)
        
        total_loss += loss.item()
        for k in loss_items:
            if k in loss_dict:
                loss_items[k] += loss_dict[k].item()
        
        if pbar and (batch_idx + 1) % 5 == 0:  # Update every 5 batches for smoothness and less stutter
            avg_loss = total_loss / (batch_idx + 1)
            avg_box = loss_items['loss_box'] / (batch_idx + 1)
            avg_cls = loss_items['loss_cls'] / (batch_idx + 1)
            pbar.set_postfix({
                'loss': f'{avg_loss:.3f}',
                'box': f'{avg_box:.3f}',
                'cls': f'{avg_cls:.3f}'
            })
    
    n = len(dataloader)
    if n == 0:
        return {
            'loss': 0,
            **{k: 0 for k in loss_items}
        }
    
    return {
        'loss': total_loss / n,
        **{k: v / n for k, v in loss_items.items()}
    }


@torch.no_grad()
def validate(
    model: nn.Module,
    dataloader,
    criterion: nn.Module,
    device: torch.device
) -> dict:
    """Validate model."""
    model.eval()
    
    total_loss = 0
    loss_items = {'loss_box': 0, 'loss_cls': 0, 'loss_dfl': 0}
    
    # Create metrics calculator
    metrics = Metrics()
    
    rank = int(os.environ.get('RANK', -1))
    
    loader_iter = tqdm(dataloader, desc='Validating') if rank in [-1, 0] else dataloader
    for images, targets in loader_iter:
        images = images.to(device)
        for k in targets:
            if isinstance(targets[k], torch.Tensor):
                targets[k] = targets[k].to(device)
        
        outputs = model(images)
        loss, loss_dict = criterion(outputs, targets)
        
        total_loss += loss.item()
        for k in loss_items:
            if k in loss_dict:
                loss_items[k] += loss_dict[k].item()
        
        # Performance evaluation (mAP)
        # Reconstruct predictions for NMS
        # Use 'getattr' to keep Pyright happy while accessing YOLOv11 specific methods
        m = model.module if hasattr(model, 'module') else model
        get_anchors = getattr(m, 'get_anchors', None)
        head = getattr(m, 'head', None)
        
        if get_anchors and head and hasattr(head, 'decode_boxes'):
            # Handle model outputs
            cls_outputs, reg_outputs = outputs['cls'], outputs['reg']
            strides = outputs['strides']
            anchors = get_anchors(device)
            
            # Decode boxes
            decoded_boxes = head.decode_boxes(reg_outputs, anchors, strides)
            
            # Format for NMS
            cls_preds = []
            for cls in cls_outputs:
                b, c, fh, fw = cls.shape
                cls_preds.append(cls.view(b, c, -1))
            cls_preds = torch.cat(cls_preds, dim=-1).permute(0, 2, 1).sigmoid()
            
            predictions = torch.cat([decoded_boxes, cls_preds], dim=-1)
            
            # NMS and Metrics processing
            results = non_max_suppression(predictions, conf_thres=0.001, iou_thres=0.6)
            
            for i, det in enumerate(results):
                # Filter ground truth for this image
                mask = targets['mask_gt'][i]
                gt_boxes = targets['bboxes'][i][mask]
                gt_labels = targets['labels'][i][mask]
                
                metrics.process_batch(det, gt_boxes, gt_labels)
        
    n = len(dataloader)
    m = metrics.compute()
    
    # In DDP, we should ideally aggregate metrics across GPUs
    if dist.is_initialized():
        # Aggregate loss
        loss_tensor = torch.tensor([total_loss, n], device=device)
        dist.all_reduce(loss_tensor, op=dist.ReduceOp.SUM)
        avg_val_loss = loss_tensor[0] / max(1, loss_tensor[1].item())
        
        # Aggregate mAP (simplified: average mAP across ranks)
        # A more precise way would be to gather detections and re-compute AP
        map_tensor = torch.tensor([m['mAP50'], m['mAP50-95']], device=device)
        dist.all_reduce(map_tensor, op=dist.ReduceOp.SUM)
        map_tensor /= dist.get_world_size()
        
        return {
            'val_loss': avg_val_loss.item(),
            'mAP50': map_tensor[0].item(),
            'mAP50-95': map_tensor[1].item()
        }
    
    return {
        'val_loss': total_loss / n,
        **{f'val_{k}': v / max(1, n) for k, v in loss_items.items()},
        'mAP50': m['mAP50'],
        'mAP50-95': m['mAP50-95']
    }


def save_checkpoint(
    model: nn.Module,
    optimizer: optim.Optimizer,
    epoch: int,
    loss: float,
    path: str
):
    """Save training checkpoint."""
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss
    }, path)
    print(f'Checkpoint saved: {path}')


def main():
    parser = argparse.ArgumentParser(description='Train YOLOv11')
    parser.add_argument('--task', type=str, default='detect', choices=['detect', 'segment', 'pose'])
    parser.add_argument('--model', type=str, default='s', choices=['n', 's', 'm', 'l', 'x'])
    parser.add_argument('--data', type=str, required=True, help='Path to data directory')
    parser.add_argument('--ann', type=str, default=None, help='Path to COCO annotation file')
    parser.add_argument('--epochs', type=int, default=100)
    parser.add_argument('--batch', type=int, default=16)
    parser.add_argument('--img-size', type=int, default=640)
    parser.add_argument('--lr', type=float, default=0.01)
    parser.add_argument('--workers', type=int, default=4)
    parser.add_argument('--device', type=str, default='0')
    parser.add_argument('--resume', type=str, default=None, help='Resume from checkpoint')
    parser.add_argument('--save-dir', type=str, default='runs/train')
    parser.add_argument('--num-classes', type=int, default=80)
    parser.add_argument('--compile', action='store_true', help='Use torch.compile for 20-50% speedup (PyTorch 2.0+)')
    # Training enhancements
    parser.add_argument('--ema-decay', type=float, default=0.9999, help='EMA decay factor')
    parser.add_argument('--warmup-epochs', type=int, default=3, help='Number of warmup epochs')
    parser.add_argument('--label-smoothing', type=float, default=0.0, help='Label smoothing factor (0.0-0.1)')
    parser.add_argument('--copypaste', type=float, default=0.0, help='CopyPaste augmentation probability')
    args = parser.parse_args()
    
    # Multi-GPU Setup at the very beginning
    rank = int(os.environ.get('RANK', -1))
    world_size = int(os.environ.get('WORLD_SIZE', 1))
    local_rank = int(os.environ.get('LOCAL_RANK', -1))
    distributed = rank != -1
    
    if distributed:
        torch.cuda.set_device(local_rank)
        device = torch.device('cuda', local_rank)
        dist.init_process_group(backend='nccl', init_method='env://')
        if rank == 0:
            print(f'Distributed training initialized: Rank {rank}, World Size {world_size}')
        
        # Adjust batch size for distributed training
        # args.batch is the TOTAL batch size across all GPUs
        per_gpu_batch = max(1, args.batch // world_size)
        if rank == 0:
            print(f'Adjusted batch size per GPU: {per_gpu_batch} (Total: {args.batch})')
    else:
        if args.device == 'cpu':
            device = torch.device('cpu')
        else:
            device = torch.device(f'cuda:{args.device}' if torch.cuda.is_available() else 'cpu')
        per_gpu_batch = args.batch

        # Use DataParallel if multiple GPUs available and not distributed
        if not distributed and torch.cuda.device_count() > 1 and args.device != 'cpu':
            if rank in [-1, 0]:
                print(f'Using {torch.cuda.device_count()} GPUs with DataParallel')
    
    if rank in [-1, 0]:
        print(f'Using device: {device}')
    
    # Create save directory (only on rank 0)
    save_dir = Path(args.save_dir) / datetime.now().strftime('%Y%m%d_%H%M%S')
    if rank in [-1, 0]:
        save_dir.mkdir(parents=True, exist_ok=True)
        print(f'Saving to: {save_dir}')
    
    # Create model
    model = create_model(
        num_classes=args.num_classes,
        task=args.task,
        model_size=args.model
    ).to(device)
    
    if distributed:
        model = DDP(model, device_ids=[local_rank], output_device=local_rank)
    elif not distributed and torch.cuda.device_count() > 1 and args.device != 'cpu':
        model = nn.DataParallel(model)
        
    if rank in [-1, 0]:
        # Unwrap for info
        (model.module if hasattr(model, 'module') else model).info()
    
    # Apply torch.compile for PyTorch 2.0+ speedup
    if args.compile:
        if hasattr(torch, 'compile'):
            print('Compiling model with torch.compile (reduce-overhead mode)...')
            model = torch.compile(model, mode='reduce-overhead')
            print('Model compiled successfully!')
        else:
            print('Warning: torch.compile requires PyTorch 2.0+, skipping compilation')
    
    # Create loss function with label smoothing
    criterion = YOLOv11Loss(
        task=args.task,
        num_classes=args.num_classes,
        label_smoothing=args.label_smoothing
    )
    
    # Create optimizer
    optimizer = optim.SGD(
        (model.module if hasattr(model, 'module') else model).parameters(),
        lr=args.lr,
        momentum=0.937,
        weight_decay=0.0005,
        nesterov=True
    )
    
    # Learning rate scheduler (cosine annealing)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=args.epochs,
        eta_min=args.lr * 0.01
    )
    
    # Create dataloader
    train_loader = create_dataloader(
        root=args.data,
        ann_file=args.ann,
        task=args.task,
        img_size=args.img_size,
        batch_size=per_gpu_batch,
        augment=True,
        shuffle=True,
        num_workers=args.workers,
        distributed=distributed
    )
    
    # Resume from checkpoint
    start_epoch = 0
    if args.resume:
        checkpoint = torch.load(args.resume, map_location=device)
        (model.module if hasattr(model, 'module') else model).load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        if rank in [-1, 0]:
            print(f'Resumed from epoch {start_epoch}')
    
    # Mixed precision scaler
    # Use torch.amp.GradScaler for newer PyTorch versions
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler'):
        scaler = torch.amp.GradScaler('cuda')
    else:
        scaler = GradScaler()
    
    # Initialize EMA (all ranks to support distributed validation)
    ema_model = model.module if hasattr(model, 'module') else model
    ema = ModelEMA(ema_model, decay=args.ema_decay)
    if rank in [-1, 0]:
        print(f'EMA initialized with decay={args.ema_decay}')
    
    # Initialize warmup scheduler
    warmup = WarmupScheduler(optimizer, warmup_epochs=args.warmup_epochs)
    if rank in [-1, 0]:
        print(f'Warmup scheduled for {args.warmup_epochs} epochs')
    
    # Training loop
    best_loss = float('inf')
    
    # Create val dataloader if annotations provided or auto-detected
    val_loader = None
    if args.ann:
        val_ann = args.ann
    else:
        # Try to auto-detect coco_mini val set
        val_ann_path = Path(args.data) / "annotations/instances_val2017.json"
        val_ann = str(val_ann_path) if val_ann_path.exists() else None
        
    if val_ann:
        print(f"Using validation annotations: {val_ann}")
        val_loader = create_dataloader(
            root=args.data,
            ann_file=val_ann,
            task=args.task,
            img_size=args.img_size,
            batch_size=per_gpu_batch,
            augment=False,
            shuffle=False,
            num_workers=args.workers,
            distributed=distributed
        )
    elif rank in [-1, 0]:
        print("Warning: No validation annotations found. mAP evaluation will be skipped.")
    
    print(f'\nStarting training for {args.epochs} epochs...\n')
    
    for epoch in range(start_epoch, args.epochs):
        if distributed:
            train_loader.sampler.set_epoch(epoch)
            
        train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device, epoch,
            warmup=warmup if epoch < args.warmup_epochs else None,
            ema=ema
        )
        
        # Update scheduler (only after warmup completes)
        if epoch >= args.warmup_epochs:
            scheduler.step()
        
        # Periodic Evaluation every 5 epochs using EMA model
        eval_metrics = {}
        if val_loader and (epoch + 1) % 5 == 0:
            eval_metrics = validate(ema.ema, val_loader, criterion, device)
            if rank in [-1, 0]:
                print(f'Evaluation Epoch {epoch}: mAP50={eval_metrics["mAP50"]:.4f}, '
                      f'mAP50-95={eval_metrics["mAP50-95"]:.4f}')
        
        # Synchronize ranks if needed, but validate is only on rank 0
        if distributed:
            dist.barrier()
        
        # Save checkpoints (only on rank 0)
        if rank in [-1, 0]:
            save_score = eval_metrics.get('mAP50-95', -train_metrics['loss'])
            if save_score > best_loss: # Higher mAP or lower negative loss
                best_loss = save_score
                save_checkpoint(ema.ema, optimizer, epoch, train_metrics['loss'], str(save_dir / 'best.pt'))
            
            save_checkpoint(model.module if hasattr(model, 'module') else model, optimizer, epoch, train_metrics['loss'], str(save_dir / 'last.pt'))
            torch.save(ema.ema.state_dict(), str(save_dir / 'last_ema.pt'))
    
    if rank in [-1, 0]:
        print(f'\nTraining complete! Best Score: {best_loss:.4f}')
        print(f'Weights saved to: {save_dir}')
    
    if distributed:
        dist.destroy_process_group()


if __name__ == '__main__':
    main()


In [ ]:
%%writefile download_coco.py
"""
COCO Dataset Downloader
Downloads a subset of COCO 2017 images and annotations
"""

import os
import json
import argparse
import urllib.request
import zipfile
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


COCO_BASE_URL = "http://images.cocodataset.org"
ANNOTATIONS_URL = f"{COCO_BASE_URL}/annotations/annotations_trainval2017.zip"

ANNOTATION_FILES = {
    'detect': 'instances_{split}2017.json',
    'segment': 'instances_{split}2017.json',
    'pose': 'person_keypoints_{split}2017.json'
}


def download_file(url: str, dest: str) -> bool:
    """Download a single file."""
    try:
        urllib.request.urlretrieve(url, dest)
        return True
    except Exception as e:
        print(f"Error downloading {url}: {e}")
        return False


def download_image(args):
    """Download a single image (for ThreadPoolExecutor)."""
    img_info, dest_dir, base_url = args
    filename = img_info['file_name']
    url = f"{base_url}/{filename}"
    dest_path = dest_dir / filename
    
    if dest_path.exists():
        return True
    
    return download_file(url, str(dest_path))


def download_coco_subset(
    output_dir: str,
    num_images: int = 100,
    split: str = 'val',
    task: str = 'detect',
    num_workers: int = 8
):
    """
    Download a subset of COCO dataset.
    
    Args:
        output_dir: Target directory
        num_images: Number of images to download
        split: 'train' or 'val'
        task: 'detect', 'segment', or 'pose'
        num_workers: Number of download threads
    """
    output_dir = Path(output_dir)
    images_dir = output_dir / f"{split}2017"
    annotations_dir = output_dir / "annotations"
    
    images_dir.mkdir(parents=True, exist_ok=True)
    annotations_dir.mkdir(parents=True, exist_ok=True)
    
    ann_filename = ANNOTATION_FILES[task].format(split=split)
    ann_path = annotations_dir / ann_filename
    
    print(f"\nDownloading COCO {split}2017 annotations for {task}...")
    
    try:
        from pycocotools.coco import COCO
        
        if not ann_path.exists():
            print(f"Downloading: {ann_filename}")
            ann_url = f"http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
            
            print("Downloading annotation archive (~250MB)...")
            zip_path = output_dir / "annotations.zip"
            
            with tqdm(unit='B', unit_scale=True, desc="annotations") as pbar:
                def reporthook(count, block_size, total_size):
                    if pbar.total is None and total_size > 0:
                        pbar.total = total_size
                    pbar.update(block_size)
                
                urllib.request.urlretrieve(ann_url, str(zip_path), reporthook)
            
            print("Extracting...")
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(output_dir)
            
            zip_path.unlink()
            print("Annotations downloaded!")
    
    except ImportError:
        print("pycocotools not installed. Installing...")
        os.system("pip install pycocotools")
        from pycocotools.coco import COCO
    
    print(f"\nLoading annotations from {ann_path}...")
    coco = COCO(str(ann_path))
    
    if task == 'pose':
        cat_ids = coco.getCatIds(catNms=['person'])
        img_ids = coco.getImgIds(catIds=cat_ids)
    else:
        img_ids = list(coco.imgs.keys())
    
    img_ids = img_ids[:num_images]
    print(f"Selected {len(img_ids)} images to download")
    
    images_info = coco.loadImgs(img_ids)
    
    print(f"\nDownloading {len(images_info)} images...")
    base_url = f"{COCO_BASE_URL}/{split}2017"
    
    download_args = [(img, images_dir, base_url) for img in images_info]
    
    successful = 0
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = {executor.submit(download_image, arg): arg for arg in download_args}
        
        with tqdm(total=len(futures), desc="Downloading") as pbar:
            for future in as_completed(futures):
                if future.result():
                    successful += 1
                pbar.update(1)
    
    print(f"Downloaded {successful}/{len(images_info)} images")
    
    subset_ann_path = annotations_dir / f"{task}_{split}2017_subset_{num_images}.json"
    
    print(f"\nCreating subset annotation file...")
    
    ann_ids = coco.getAnnIds(imgIds=img_ids)
    annotations = coco.loadAnns(ann_ids)
    
    subset_coco = {
        'info': coco.dataset.get('info', {}),
        'licenses': coco.dataset.get('licenses', []),
        'images': images_info,
        'annotations': annotations,
        'categories': coco.dataset.get('categories', [])
    }
    
    with open(subset_ann_path, 'w') as f:
        json.dump(subset_coco, f)
    
    print(f"Annotations saved: {subset_ann_path}")
    
    print(f"\n{'='*50}")
    print(f"SUMMARY")
    print(f"{'='*50}")
    print(f"Images: {images_dir}")
    print(f"Annotations: {subset_ann_path}")
    print(f"Image count: {successful}")
    print(f"Annotation count: {len(annotations)}")
    print(f"\nReady for training!")
    print(f"\npython train.py --task {task} --model s \\")
    print(f"    --data \"{images_dir}\" \\")
    print(f"    --ann \"{subset_ann_path}\" \\")
    print(f"    --epochs 50 --batch 8")
    
    return str(images_dir), str(subset_ann_path)


def main():
    parser = argparse.ArgumentParser(description='Download COCO dataset subset')
    parser.add_argument('--output', type=str, default='coco', help='Output directory')
    parser.add_argument('--num-images', type=int, default=100, help='Number of images to download')
    parser.add_argument('--split', type=str, default='val', choices=['train', 'val'], help='Data split')
    parser.add_argument('--task', type=str, default='detect', choices=['detect', 'segment', 'pose'], help='Task type')
    parser.add_argument('--workers', type=int, default=8, help='Number of threads')
    args = parser.parse_args()
    
    download_coco_subset(
        output_dir=args.output,
        num_images=args.num_images,
        split=args.split,
        task=args.task,
        num_workers=args.workers
    )


if __name__ == '__main__':
    main()


In [ ]:
%%writefile download_coco_pose.py
"""
COCO Keypoints (Pose) Dataset Downloader
Downloads and prepares COCO keypoints dataset for pose estimation training
"""

import os
import sys
import json
import zipfile
import shutil
from pathlib import Path
from urllib.request import urlretrieve
from tqdm import tqdm

# COCO Keypoints URLs
COCO_URLS = {
    'train2017': 'http://images.cocodataset.org/zips/train2017.zip',
    'val2017': 'http://images.cocodataset.org/zips/val2017.zip',
    'annotations': 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'
}

# File sizes for progress display
FILE_SIZES = {
    'train2017': '18GB',
    'val2017': '1GB',
    'annotations': '252MB'
}


class DownloadProgressBar(tqdm):
    """tqdm progress bar for downloads."""
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)


def download_file(url: str, dest: Path, desc: str = None):
    """Download file with progress bar."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    
    if dest.exists():
        print(f'  {dest.name} already exists, skipping...')
        return
    
    print(f'  Downloading {desc or dest.name}...')
    with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=dest.name) as t:
        urlretrieve(url, dest, reporthook=t.update_to)


def extract_zip(zip_path: Path, dest_dir: Path):
    """Extract zip file with progress."""
    print(f'  Extracting {zip_path.name}...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        members = zip_ref.namelist()
        for member in tqdm(members, desc='Extracting'):
            zip_ref.extract(member, dest_dir)


def filter_person_annotations(ann_file: Path, output_file: Path):
    """
    Filter annotations to keep only person instances with keypoints.
    This significantly reduces memory usage during training.
    """
    print(f'  Filtering person annotations from {ann_file.name}...')
    
    with open(ann_file, 'r') as f:
        data = json.load(f)
    
    # Keep only person category (id=1)
    person_cat = [cat for cat in data['categories'] if cat['name'] == 'person']
    
    # Filter annotations: keep only person with keypoints
    person_anns = []
    valid_image_ids = set()
    
    for ann in tqdm(data['annotations'], desc='Filtering'):
        if ann['category_id'] == 1 and 'keypoints' in ann:
            # Check if at least some keypoints are visible
            kpts = ann['keypoints']
            num_visible = sum(1 for i in range(2, len(kpts), 3) if kpts[i] > 0)
            if num_visible >= 5:  # At least 5 visible keypoints
                person_anns.append(ann)
                valid_image_ids.add(ann['image_id'])
    
    # Filter images to keep only those with valid annotations
    filtered_images = [img for img in data['images'] if img['id'] in valid_image_ids]
    
    # Create filtered dataset
    filtered_data = {
        'info': data.get('info', {}),
        'licenses': data.get('licenses', []),
        'categories': person_cat,
        'images': filtered_images,
        'annotations': person_anns
    }
    
    with open(output_file, 'w') as f:
        json.dump(filtered_data, f)
    
    print(f'  Filtered: {len(person_anns)} person annotations in {len(filtered_images)} images')


def download_coco_pose(output_dir: str = 'coco_pose', download_images: bool = True):
    """
    Download and prepare COCO Keypoints dataset.
    
    Args:
        output_dir: Output directory
        download_images: Whether to download images (set False if you already have them)
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    print('=' * 60)
    print('COCO Keypoints (Pose) Dataset Downloader')
    print('=' * 60)
    print(f'\nOutput directory: {output_path.absolute()}')
    print(f'\nDataset sizes:')
    for name, size in FILE_SIZES.items():
        print(f'  - {name}: ~{size}')
    print()
    
    # Download annotations first (always needed)
    print('\n[1/4] Downloading annotations...')
    ann_zip = output_path / 'annotations_trainval2017.zip'
    download_file(COCO_URLS['annotations'], ann_zip, 'annotations (~252MB)')
    
    # Extract annotations
    if not (output_path / 'annotations').exists():
        extract_zip(ann_zip, output_path)
    
    # Filter person keypoint annotations
    print('\n[2/4] Preparing keypoint annotations...')
    
    # Training annotations
    train_ann = output_path / 'annotations' / 'person_keypoints_train2017.json'
    train_filtered = output_path / 'annotations' / 'person_keypoints_train2017_filtered.json'
    if train_ann.exists() and not train_filtered.exists():
        filter_person_annotations(train_ann, train_filtered)
    
    # Validation annotations
    val_ann = output_path / 'annotations' / 'person_keypoints_val2017.json'
    val_filtered = output_path / 'annotations' / 'person_keypoints_val2017_filtered.json'
    if val_ann.exists() and not val_filtered.exists():
        filter_person_annotations(val_ann, val_filtered)
    
    if download_images:
        # Download training images
        print('\n[3/4] Downloading training images...')
        train_zip = output_path / 'train2017.zip'
        download_file(COCO_URLS['train2017'], train_zip, 'train2017 (~18GB)')
        
        if not (output_path / 'train2017').exists():
            extract_zip(train_zip, output_path)
        
        # Download validation images
        print('\n[4/4] Downloading validation images...')
        val_zip = output_path / 'val2017.zip'
        download_file(COCO_URLS['val2017'], val_zip, 'val2017 (~1GB)')
        
        if not (output_path / 'val2017').exists():
            extract_zip(val_zip, output_path)
    else:
        print('\n[3/4] Skipping image download (--no-images flag)')
        print('[4/4] Skipping image download')
    
    # Print summary
    print('\n' + '=' * 60)
    print('Download Complete!')
    print('=' * 60)
    print(f'\nDataset structure:')
    print(f'  {output_path}/')
    print(f'  ├── train2017/           # Training images')
    print(f'  ├── val2017/             # Validation images')
    print(f'  └── annotations/')
    print(f'      ├── person_keypoints_train2017.json')
    print(f'      ├── person_keypoints_train2017_filtered.json  # Person-only')
    print(f'      ├── person_keypoints_val2017.json')
    print(f'      └── person_keypoints_val2017_filtered.json    # Person-only')
    
    print(f'\nTo train YOLOv11-Pose:')
    print(f'  python train.py \\')
    print(f'      --task pose \\')
    print(f'      --model m \\')
    print(f'      --data {output_path}/train2017 \\')
    print(f'      --ann {output_path}/annotations/person_keypoints_train2017_filtered.json \\')
    print(f'      --batch 32 \\')
    print(f'      --epochs 100')


def main():
    import argparse
    parser = argparse.ArgumentParser(description='Download COCO Keypoints Dataset')
    parser.add_argument('--output', type=str, default='coco_pose', help='Output directory')
    parser.add_argument('--no-images', action='store_true', help='Skip image download (annotations only)')
    args = parser.parse_args()
    
    download_coco_pose(args.output, download_images=not args.no_images)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile download_coco_seg.py
"""
COCO Instance Segmentation Dataset Downloader
Downloads and prepares COCO instance segmentation dataset for training
"""

import os
import sys
import json
import zipfile
import shutil
from pathlib import Path
from urllib.request import urlretrieve
from tqdm import tqdm

# COCO URLs
COCO_URLS = {
    'train2017': 'http://images.cocodataset.org/zips/train2017.zip',
    'val2017': 'http://images.cocodataset.org/zips/val2017.zip',
    'annotations': 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'
}

# File sizes
FILE_SIZES = {
    'train2017': '18GB',
    'val2017': '1GB',
    'annotations': '252MB'
}


class DownloadProgressBar(tqdm):
    """tqdm progress bar for downloads."""
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)


def download_file(url: str, dest: Path, desc: str = None):
    """Download file with progress bar."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    
    if dest.exists():
        print(f'  {dest.name} already exists, skipping...')
        return
    
    print(f'  Downloading {desc or dest.name}...')
    with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=dest.name) as t:
        urlretrieve(url, dest, reporthook=t.update_to)


def extract_zip(zip_path: Path, dest_dir: Path):
    """Extract zip file with progress."""
    print(f'  Extracting {zip_path.name}...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        members = zip_ref.namelist()
        for member in tqdm(members, desc='Extracting'):
            zip_ref.extract(member, dest_dir)


def create_mini_dataset(ann_file: Path, output_file: Path, max_images: int = 5000):
    """
    Create a mini version of the dataset for quick testing.
    
    Args:
        ann_file: Full annotation file
        output_file: Output file for mini dataset
        max_images: Maximum number of images to include
    """
    print(f'  Creating mini dataset ({max_images} images)...')
    
    with open(ann_file, 'r') as f:
        data = json.load(f)
    
    # Select first N images
    selected_images = data['images'][:max_images]
    selected_image_ids = {img['id'] for img in selected_images}
    
    # Filter annotations
    selected_anns = [ann for ann in data['annotations'] 
                     if ann['image_id'] in selected_image_ids]
    
    mini_data = {
        'info': data.get('info', {}),
        'licenses': data.get('licenses', []),
        'categories': data['categories'],
        'images': selected_images,
        'annotations': selected_anns
    }
    
    with open(output_file, 'w') as f:
        json.dump(mini_data, f)
    
    print(f'  Created: {len(selected_images)} images, {len(selected_anns)} annotations')


def download_coco_seg(output_dir: str = 'coco_seg', download_images: bool = True, mini: bool = False):
    """
    Download and prepare COCO Instance Segmentation dataset.
    
    Args:
        output_dir: Output directory
        download_images: Whether to download images
        mini: Create mini dataset for testing
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    print('=' * 60)
    print('COCO Instance Segmentation Dataset Downloader')
    print('=' * 60)
    print(f'\nOutput directory: {output_path.absolute()}')
    print(f'\nDataset sizes:')
    for name, size in FILE_SIZES.items():
        print(f'  - {name}: ~{size}')
    print()
    
    # Download annotations
    print('\n[1/3] Downloading annotations...')
    ann_zip = output_path / 'annotations_trainval2017.zip'
    download_file(COCO_URLS['annotations'], ann_zip, 'annotations (~252MB)')
    
    # Extract annotations
    if not (output_path / 'annotations').exists():
        extract_zip(ann_zip, output_path)
    
    # Create mini dataset if requested
    if mini:
        print('\n[2/3] Creating mini datasets...')
        train_ann = output_path / 'annotations' / 'instances_train2017.json'
        train_mini = output_path / 'annotations' / 'instances_train2017_mini.json'
        if train_ann.exists() and not train_mini.exists():
            create_mini_dataset(train_ann, train_mini, max_images=5000)
        
        val_ann = output_path / 'annotations' / 'instances_val2017.json'
        val_mini = output_path / 'annotations' / 'instances_val2017_mini.json'
        if val_ann.exists() and not val_mini.exists():
            create_mini_dataset(val_ann, val_mini, max_images=1000)
    else:
        print('\n[2/3] Skipping mini dataset creation')
    
    if download_images:
        # Download training images
        print('\n[3/3] Downloading images...')
        train_zip = output_path / 'train2017.zip'
        download_file(COCO_URLS['train2017'], train_zip, 'train2017 (~18GB)')
        
        if not (output_path / 'train2017').exists():
            extract_zip(train_zip, output_path)
        
        # Download validation images
        val_zip = output_path / 'val2017.zip'
        download_file(COCO_URLS['val2017'], val_zip, 'val2017 (~1GB)')
        
        if not (output_path / 'val2017').exists():
            extract_zip(val_zip, output_path)
    else:
        print('\n[3/3] Skipping image download')
    
    # Print summary
    print('\n' + '=' * 60)
    print('Download Complete!')
    print('=' * 60)
    print(f'\nDataset structure:')
    print(f'  {output_path}/')
    print(f'  ├── train2017/           # 118k training images')
    print(f'  ├── val2017/             # 5k validation images')
    print(f'  └── annotations/')
    print(f'      ├── instances_train2017.json      # Full training (860k instances)')
    print(f'      ├── instances_val2017.json        # Full validation')
    if mini:
        print(f'      ├── instances_train2017_mini.json # Mini training (5k images)')
        print(f'      └── instances_val2017_mini.json   # Mini validation (1k images)')
    
    print(f'\nTo train YOLOv11-Seg:')
    ann_suffix = '_mini' if mini else ''
    print(f'  python train.py \\')
    print(f'      --task segment \\')
    print(f'      --model m \\')
    print(f'      --data {output_path}/train2017 \\')
    print(f'      --ann {output_path}/annotations/instances_train2017{ann_suffix}.json \\')
    print(f'      --batch 16 \\')
    print(f'      --epochs 100')


def main():
    import argparse
    parser = argparse.ArgumentParser(description='Download COCO Instance Segmentation Dataset')
    parser.add_argument('--output', type=str, default='coco_seg', help='Output directory')
    parser.add_argument('--no-images', action='store_true', help='Skip image download')
    parser.add_argument('--mini', action='store_true', help='Create mini dataset for testing')
    args = parser.parse_args()
    
    download_coco_seg(args.output, download_images=not args.no_images, mini=args.mini)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile inference.py
"""
YOLOv11 Inference Script
Supports image, video, and webcam inference
"""

import argparse
import cv2
import torch
import torch.nn.functional as F
import numpy as np
from pathlib import Path
import time
from typing import Union
import sys
sys.path.insert(0, str(Path(__file__).parent))

from yolov11.model import YOLOv11
from yolov11.data.augmentations import LetterBox
from yolov11.utils.nms import non_max_suppression
from yolov11.utils.visualization import visualize_predictions


# COCO class names
COCO_NAMES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat',
    'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat',
    'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack',
    'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple',
    'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop',
    'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink',
    'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]


class YOLOv11Predictor:
    """YOLOv11 inference wrapper."""
    
    def __init__(
        self,
        weights: str,
        task: str = 'detect',
        device: str = '0',
        conf_thres: float = 0.25,
        iou_thres: float = 0.45,
        img_size: int = 640
    ):
        self.device = torch.device(f'cuda:{device}' if torch.cuda.is_available() and device != 'cpu' else 'cpu')
        self.conf_thres = conf_thres
        self.iou_thres = iou_thres
        self.img_size = img_size
        self.task = task
        self.weights_path = Path(weights)
        self.format = self.weights_path.suffix.lower() # '.pt', '.onnx', '.engine'
        self.half = False
        self.strides = [8, 16, 32]
        self._anchors = None # Cache for anchors
        
        if self.format == '.pt':
            # Load PyTorch model
            checkpoint = torch.load(weights, map_location=self.device, weights_only=False)
            config = checkpoint.get('config', {})
            model_size = config.get('model_size', 's')
            num_classes = config.get('num_classes', 80)
            self.task = config.get('task', task)
            
            self.model = YOLOv11(
                num_classes=num_classes,
                task=self.task,
                model_size=model_size
            ).to(self.device)
            
            if 'ema_state_dict' in checkpoint:
                self.model.load_state_dict(checkpoint['ema_state_dict'])
            else:
                state_dict = checkpoint.get('model_state_dict', checkpoint)
                self.model.load_state_dict(state_dict)
            self.model.eval()
            self.model.fuse() # Fuse Conv + BN for inference
            print(f'Loaded PyTorch model: {weights}')
            
        elif self.format == '.onnx':
            import onnxruntime as ort
            providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if self.device.type == 'cuda' else ['CPUExecutionProvider']
            sess_options = ort.SessionOptions()
            sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
            self.session = ort.InferenceSession(weights, sess_options, providers=providers)
            print(f'Loaded ONNX model: {weights}')
            
        elif self.format == '.engine':
            import tensorrt as trt
            from yolov11.utils.export import TRTInference # We should add a helper for this
            self.trt_model = TRTInference(weights)
            print(f'Loaded TensorRT engine: {weights}')
            
        else:
            raise ValueError(f"Unsupported weight format: {self.format}. Supported: .pt, .onnx, .engine")
            
        # Preprocessing
        self.letterbox = LetterBox((img_size, img_size))

    
    def set_half(self, half: bool):
        """Toggle FP16 (half precision) inference."""
        if half and self.device.type == 'cuda':
            self.half = True
            if self.format == '.pt':
                self.model.half()
            print("FP16 (Half precision) enabled.")
        else:
            self.half = False

    def get_anchors(self):
        """Generate anchor points for decoding."""
        if self._anchors is not None:
            return self._anchors
        
        anchors = []
        for stride in self.strides:
            grid_size = self.img_size // stride
            ys, xs = torch.meshgrid(
                torch.arange(grid_size),
                torch.arange(grid_size),
                indexing='ij'
            )
            grid = torch.stack([xs, ys], dim=-1).float() + 0.5
            grid = grid * stride
            anchors.append(grid.to(self.device))
        
        self._anchors = anchors
        return anchors

    def preprocess(self, image: np.ndarray) -> torch.Tensor:
        """Preprocess image for inference."""
        # Resize with letterbox
        img, _ = self.letterbox(image, {})
        
        # BGR to RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # HWC to CHW, normalize, move to device
        img = torch.from_numpy(img.transpose(2, 0, 1)).to(self.device)
        img = img.half() if self.half else img.float()
        img /= 255.0
        
        # Add batch dimension
        return img.unsqueeze(0)
    
    @torch.no_grad()
    def predict(self, image: np.ndarray) -> dict:
        """Run inference on image."""
        h, w = image.shape[:2]
        img = self.preprocess(image)
        
        # Inference based on format
        if self.format == '.pt':
            outputs = self.model(img)
        elif self.format == '.onnx':
            ort_inputs = {self.session.get_inputs()[0].name: img.cpu().numpy()}
            outputs = self.session.run(None, ort_inputs)
            # Move outputs to device
            outputs = [torch.from_numpy(x).to(self.device) for x in outputs]
        elif self.format == '.engine':
            # TensorRT outputs are on host (CPU), move to device
            outputs = [x.to(self.device) for x in self.trt_model(img)]
        else:
            raise ValueError(f"Cannot run prediction on unsupported format: {self.format}")
        
        return self.postprocess(outputs, (h, w))
    
    def postprocess(self, outputs: Union[dict, list], orig_shape: tuple) -> dict:
        """Post-process model outputs."""
        h, w = orig_shape
        
        if isinstance(outputs, dict):
            cls_outputs, reg_outputs = outputs['cls'], outputs['reg']
            strides = outputs['strides']
            anchors = self.model.get_anchors(self.device)
            
            # Decode actual boxes
            if hasattr(self.model, 'head') and hasattr(self.model.head, 'decode_boxes'):
                boxes = self.model.head.decode_boxes(reg_outputs, anchors, strides)
            else:
                # Fallback decoding if head is not accessible
                boxes = self._decode_boxes(reg_outputs, anchors, strides)
                
            cls_preds = []
            for cls in cls_outputs:
                b, c, fh, fw = cls.shape
                cls_preds.append(cls.view(b, c, -1))
            cls_preds = torch.cat(cls_preds, dim=-1).permute(0, 2, 1).sigmoid()
        else:
            # ONNX/TRT return a flat list of 6 tensors for Detection (3 cls, 3 reg)
            cls_outputs, reg_outputs = outputs[:3], outputs[3:6]
            strides = self.strides
            anchors = self.get_anchors()
            
            # Decode boxes for ONNX/TRT
            boxes = self._decode_boxes(reg_outputs, anchors, strides)
            
            cls_preds = []
            for cls in cls_outputs:
                b, c, fh, fw = cls.shape
                cls_preds.append(cls.view(b, c, -1))
            cls_preds = torch.cat(cls_preds, dim=-1).permute(0, 2, 1).sigmoid()
            
        # Combine
        predictions = torch.cat([boxes, cls_preds], dim=-1)
        
        # NMS
        results = non_max_suppression(
            predictions,
            conf_thres=self.conf_thres,
            iou_thres=self.iou_thres
        )[0]
        
        if len(results) == 0:
            return {
                'boxes': np.zeros((0, 4)),
                'scores': np.zeros(0),
                'labels': np.zeros(0, dtype=int)
            }
        
        # Scale boxes to original image
        scale = min(self.img_size / h, self.img_size / w)
        pad = ((self.img_size - w * scale) / 2, (self.img_size - h * scale) / 2)
        
        boxes = results[:, :4].cpu().numpy()
        boxes[:, [0, 2]] = (boxes[:, [0, 2]] - pad[0]) / scale
        boxes[:, [1, 3]] = (boxes[:, [1, 3]] - pad[1]) / scale
        
        # Clip to image bounds
        boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, w)
        boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, h)
        
        output = {
            'boxes': boxes,
            'scores': results[:, 4].cpu().numpy(),
            'labels': results[:, 5].cpu().numpy().astype(int)
        }
        
        # Handle pose keypoints
        if self.task == 'pose' and isinstance(outputs, dict) and 'kpts' in outputs:
            kpt_outputs = outputs['kpts']
            anchors = self.model.get_anchors(self.device)
            strides = outputs['strides']
            
            # Decode keypoints
            if hasattr(self.model, 'head') and hasattr(self.model.head, 'decode_keypoints'):
                keypoints = self.model.head.decode_keypoints(kpt_outputs, anchors, strides)
            else:
                keypoints = self._decode_keypoints(kpt_outputs, anchors, strides)
            
            # Get keypoints for NMS results (need to track indices)
            # For simplicity, we get keypoints from flattened predictions
            # and match by spatial location
            keypoints = keypoints[0].cpu().numpy()  # (N, 17, 3)
            
            # Scale keypoints to original image
            keypoints[..., 0] = (keypoints[..., 0] - pad[0]) / scale
            keypoints[..., 1] = (keypoints[..., 1] - pad[1]) / scale
            
            # Clip to image bounds
            keypoints[..., 0] = np.clip(keypoints[..., 0], 0, w)
            keypoints[..., 1] = np.clip(keypoints[..., 1], 0, h)
            
            # Match keypoints to detection results (simplified - take first N)
            num_dets = len(boxes)
            if keypoints.shape[0] >= num_dets:
                output['keypoints'] = keypoints[:num_dets]
            else:
                # Pad with zeros if not enough keypoints
                output['keypoints'] = np.zeros((num_dets, 17, 3))
                output['keypoints'][:len(keypoints)] = keypoints
        
        # Handle segmentation masks
        if self.task == 'segment' and isinstance(outputs, dict) and 'masks' in outputs and 'protos' in outputs:
            mask_coeffs = outputs['masks']  # List of (B, num_protos, H, W) per scale
            protos = outputs['protos']  # (B, num_protos, proto_H, proto_W)
            
            # Flatten mask coefficients
            mask_flat = []
            for mask in mask_coeffs:
                b, c, mh, mw = mask.shape
                mask_flat.append(mask.view(b, c, -1).permute(0, 2, 1))
            mask_flat = torch.cat(mask_flat, dim=1)[0]  # (N, num_protos)
            
            # Get prototypes
            proto = protos[0]  # (num_protos, pH, pW)
            proto_h, proto_w = proto.shape[1:]
            proto_flat = proto.view(proto.shape[0], -1)  # (num_protos, pH*pW)
            
            # Assemble all masks
            all_masks = torch.mm(mask_flat, proto_flat)  # (N, pH*pW)
            all_masks = all_masks.sigmoid().view(-1, proto_h, proto_w)  # (N, pH, pW)
            
            # Select masks for detections (simplified - take first num_dets)
            num_dets = len(boxes)
            if all_masks.shape[0] >= num_dets:
                masks = all_masks[:num_dets]
            else:
                masks = all_masks
            
            # Resize masks to original image size
            masks = F.interpolate(
                masks.unsqueeze(1),
                size=(h, w),
                mode='bilinear',
                align_corners=False
            ).squeeze(1)
            
            # Apply letterbox inverse transform
            masks = masks.cpu().numpy()
            
            # Crop padding (masks are already in letterbox space)
            pad_h = int(pad[1] / scale)
            pad_w = int(pad[0] / scale)
            
            output['masks'] = (masks > 0.5).astype(np.uint8)
        
        return output

    def _decode_boxes(self, reg_outputs, anchors, strides) -> torch.Tensor:
        """Internal helper for box decoding (DFL excluded for simplicity in ONNX)."""
        decoded_boxes = []
        for reg_out, anchor, stride in zip(reg_outputs, anchors, strides):
            b, c, h, w = reg_out.shape
            
            # For simplicity, if reg_out has 64 channels (4*16), we take the mean or first 4
            # A full DFL decoding would be better but this is more robust for exported models
            if c == 64:
                # Simplified DFL integration: take the mean of distribution
                reg_dist = reg_out.view(b, 4, 16, -1).softmax(2)
                weights = torch.arange(16, device=reg_out.device).view(1, 1, 16, 1)
                reg_dist = (reg_dist * weights).sum(2) # (B, 4, H*W)
            else:
                reg_dist = reg_out.view(b, 4, -1)
            
            reg_dist = reg_dist.permute(0, 2, 1) # (B, H*W, 4)
            anchor = anchor.view(-1, 2)
            
            lt = reg_dist[..., :2]
            rb = reg_dist[..., 2:]
            
            x1y1 = anchor - lt * stride
            x2y2 = anchor + rb * stride
            decoded_boxes.append(torch.cat([x1y1, x2y2], dim=-1))
            
        return torch.cat(decoded_boxes, dim=1)
    
    def _decode_keypoints(self, kpt_outputs, anchors, strides) -> torch.Tensor:
        """Internal helper for keypoint decoding."""
        decoded_kpts = []
        for kpt_out, anchor, stride in zip(kpt_outputs, anchors, strides):
            b, c, h, w = kpt_out.shape
            num_keypoints = c // 3  # x, y, visibility per keypoint
            
            # Reshape: (B, 51, H, W) -> (B, 17, 3, H*W)
            kpt = kpt_out.view(b, num_keypoints, 3, -1)
            
            # Decode: keypoint offset from anchor
            anchor_flat = anchor.view(-1, 2)  # (H*W, 2)
            
            # x, y offsets (scaled by stride)
            xy = kpt[:, :, :2, :]  # (B, 17, 2, H*W)
            vis = kpt[:, :, 2:3, :]  # (B, 17, 1, H*W)
            
            # Decode coordinates
            xy = xy.permute(0, 3, 1, 2)  # (B, H*W, 17, 2)
            vis = vis.permute(0, 3, 1, 2)  # (B, H*W, 17, 1)
            
            # Add anchor offset and scale
            xy = xy * stride + anchor_flat.view(1, -1, 1, 2)
            
            # Combine
            kpts = torch.cat([xy, vis.sigmoid()], dim=-1)  # (B, H*W, 17, 3)
            decoded_kpts.append(kpts)
        
        return torch.cat(decoded_kpts, dim=1)  # (B, N, 17, 3)

def run_image(predictor: YOLOv11Predictor, source: str, save_path: str = None):
    """Run inference on image."""
    # Use numpy.fromfile to handle Unicode paths
    try:
        image = cv2.imdecode(np.fromfile(source, dtype=np.uint8), cv2.IMREAD_COLOR)
    except Exception:
        image = cv2.imread(source)
    
    if image is None:
        raise FileNotFoundError(f'Image not found: {source}')

    
    start = time.time()
    results = predictor.predict(image)
    inference_time = time.time() - start
    
    print(f'Inference: {inference_time * 1000:.1f}ms, {len(results["boxes"])} detections')
    
    # Visualize
    vis_image = visualize_predictions(
        image, results, predictor.task, COCO_NAMES
    )
    
    if save_path:
        cv2.imwrite(save_path, vis_image)
        print(f'Saved: {save_path}')
    else:
        # Try to display, if fails auto-save
        try:
            cv2.imshow('YOLOv11', vis_image)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
        except cv2.error:
            auto_save = 'output_detection.jpg'
            cv2.imwrite(auto_save, vis_image)
            print(f'GUI not available, saved to: {auto_save}')

    
    return results


def run_video(predictor: YOLOv11Predictor, source, save_path: str = None):
    """Run inference on video or webcam with FPS counter."""
    # Convert string digit to int (for webcam source)
    if isinstance(source, str) and source.isdigit():
        source = int(source)
    
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        print(f"Error: Could not open source {source}")
        return
    
    if save_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v') #type: ignore
        fps_video = cap.get(cv2.CAP_PROP_FPS) or 30
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        writer = cv2.VideoWriter(save_path, fourcc, fps_video, (w, h))
    
    print("Press 'q' to quit")
    
    # Accurate FPS calculation across frames
    prev_time = time.time()
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Inference
        results = predictor.predict(frame)
        
        # FPS calculation
        curr_time = time.time()
        fps = 1 / (curr_time - prev_time + 1e-9)
        prev_time = curr_time
        
        # Visualize
        start_vis = time.time()
        vis_frame = visualize_predictions(
            frame, results, predictor.task, COCO_NAMES
        )
        vis_time = time.time() - start_vis
        
        # Add FPS Overlay
        cv2.putText(
            vis_frame, f'FPS: {fps:.1f} (Vis: {vis_time*1000:.1f}ms)', (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2
        )
        
        if save_path:
            writer.write(vis_frame)
        else:
            cv2.imshow('YOLOv11 Inference', vis_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    
    cap.release()
    if save_path:
        writer.release()
    cv2.destroyAllWindows()



def main():
    parser = argparse.ArgumentParser(description='YOLOv11 Inference')
    parser.add_argument('--weights', type=str, required=True, help='Model weights path')
    parser.add_argument('--source', type=str, required=True, help='Image/video path or webcam index')
    parser.add_argument('--task', type=str, default='detect', choices=['detect', 'segment', 'pose'])
    parser.add_argument('--conf', type=float, default=0.25, help='Confidence threshold')
    parser.add_argument('--iou', type=float, default=0.45, help='NMS IoU threshold')
    parser.add_argument('--img-size', type=int, default=640, help='Image size')
    parser.add_argument('--device', type=str, default='0', help='Device (cuda:0 or cpu)')
    parser.add_argument('--save', type=str, default=None, help='Save path')
    parser.add_argument('--half', action='store_true', help='Use FP16 half precision')
    args = parser.parse_args()
    
    # Create predictor
    predictor = YOLOv11Predictor(
        weights=args.weights,
        task=args.task,
        device=args.device,
        conf_thres=args.conf,
        iou_thres=args.iou,
        img_size=args.img_size
    )
    
    if args.half:
        predictor.set_half(True)
    
    # Determine source type
    source = args.source
    if source.isdigit() or source.endswith(('.mp4', '.avi', '.mov', '.mkv')):
        run_video(predictor, source, args.save)
    else:
        run_image(predictor, source, args.save)


if __name__ == '__main__':
    main()


## 1. Data Preparation

First, we need to download a subset of the COCO dataset. We'll use `download_coco.py` to do this automatically.
This will create a `coco_mini` directory with 100 images.

In [ ]:
# Download COCO subset (100 images)
!python download_coco.py --output coco_mini --num-images 100 --split val

## 2. Distributed Training (Multi-GPU)

Once data is downloaded, launch the training using `torchrun` to use both GPUs.

In [ ]:
# Launch training with 2 GPUs
# Note: We specify the annotation file explicitly for the subset
!torchrun --nproc_per_node=2 train.py --data coco_mini --ann coco_mini/annotations/detect_val2017_subset_100.json --epochs 50 --batch 16

## Inference

In [ ]:
# Run inference on an image
import os, glob
runs = glob.glob('runs/train/20*')
if runs:
    latest_run = max(runs, key=os.path.getmtime)
    weights_path = os.path.join(latest_run, 'last.pt')
    print(f'Latest weights: {weights_path}')
    # Example command to run inference (uncomment to run):
    # !python inference.py --weights {weights_path} --source coco_mini/val2017/000000000139.jpg
else:
    print('No training runs found.')